# AI for quantum systems: workload orchestration and backend selection — open science — full-paper validations

**live-required: reproducible, source-aware, and configured to require live IBM/Qiskit Runtime metadata before synthetic generation.**

This notebook implements a complete pipeline to study how machine learning techniques can support operational decisions in hybrid quantum-classical systems. The central task is to select suitable backends for quantum workloads, considering runtime, expected fidelity, failure probability, cost, availability, capacity, and the possibility of using real quantum hardware.

The main dataset is **synthetic and controlled**, but this version includes an **external plausibility validation layer**. The validation compares simulated ranges with one of the following evidence levels, explicitly recorded in the outputs: live IBM/Qiskit Runtime metadata from the current execution. Cached metadata, Qiskit snapshots/fake backends, and static plausibility references are disabled in this final-paper configuration.

> **Correct scope:** this notebook is a methodological proof of concept, a source-aware externally anchored synthetic benchmark, and an end-to-end evaluation of an operational decision policy. It does not claim production performance on real QPUs or quantum advantage. In this configuration, the notebook must stop if live IBM/Qiskit Runtime metadata cannot be collected in the current execution.



## 0. Reading guide and scientific contribution

This notebook is structured as an open science artifact associated with the paper. It contains not only executable code, but also the methodological explanation needed for another researcher to understand, reproduce, and critique the experiment.

### Research question

How can machine learning models support operational backend selection in hybrid quantum-classical systems while jointly considering performance, fidelity, failure, cost, availability, capacity, and the possibility of execution on real hardware?

### Operational hypothesis

Backend selection can be modeled as a multicriteria decision learned from workload and backend attributes. Instead of using only a fixed rule, the notebook generates a synthetic decision policy and evaluates whether supervised models can reproduce it and generalize it to unseen workloads.

### What this notebook provides

1. An enriched synthetic dataset of `workload-backend` pairs.
2. Regression models for runtime and fidelity prediction.
3. Classification models for execution recommendation.
4. End-to-end evaluation with propagated predictions.
5. Backend ranking and final selection diagnostics.
6. External plausibility checks against IBM/Qiskit metadata, snapshots, or documented references.



## 0.1 Execution checklist

Run the notebook from beginning to end. For quick tests, set `FAST_MODE = True` in Section 1.

### Main outputs

Outputs are saved in:

- `outputs_ai_quantum_v9_6/data`
- `outputs_ai_quantum_v9_6/tables`
- `outputs_ai_quantum_v9_6/figures`
- `outputs_ai_quantum_v9_6/real_validation`

### Recommended tables for the paper

Prefer the final tables with the `final_` prefix and `v9_6` version suffix. Tables with `combined_audit` are retained only for internal traceability and may include empty fields because they combine different tasks.

### Mandatory evidence check before writing the paper

This final-paper configuration requires live IBM/Qiskit Runtime anchoring. Inspect `external_anchor_evidence_scope_v9_6.csv` or `real_validation_summary_v9_6.csv`; the run is valid for the paper only if `anchor_evidence_scope = live_ibm_qiskit_runtime_metadata` and `anchor_priority_values` includes `live_current_run`. If live collection fails, the notebook stops before synthetic generation.



In [ ]:
# In Google Colab, uncomment if needed.
!pip install scikit-learn pandas numpy matplotlib



## 1. Imports, configuration, and output directories



In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    ExtraTreesRegressor,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet, SGDRegressor, SGDClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.inspection import permutation_importance

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Execution mode
# ------------------------------------------------------------
# FAST_MODE=True reduces dataset size and execution cost.
# Use FAST_MODE=False to generate reportable results for the paper.
FAST_MODE = False

N_WORKLOADS = 160 if FAST_MODE else 700
N_GROUP_FOLDS = 3
N_OOF_FOLDS = 3

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

OUTPUT_DIR = Path("outputs_ai_quantum_v9_6")

# Persistent cache for external metadata.
# Unlike OUTPUT_DIR, this folder should not be deleted at each run,
# because real IBM/Qiskit metadata may be reused to generate the synthetic dataset.
EXTERNAL_CACHE_DIR = Path("external_qpu_cache")
EXTERNAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Ensures that tables and figures reflect only the current run.
# This avoids mixing results from previous runs.
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

FIG_DIR = OUTPUT_DIR / "figures"
TAB_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "data"

for directory in [FIG_DIR, TAB_DIR, DATA_DIR, EXTERNAL_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directories created:")
print(FIG_DIR)
print(TAB_DIR)
print(DATA_DIR)
print(EXTERNAL_CACHE_DIR)


RUN_CONFIG = {
    "version": "v9_6_modality_aware_real_data_anchor",
    "seed": SEED,
    "fast_mode": FAST_MODE,
    "n_workloads": N_WORKLOADS,
    "n_group_folds": N_GROUP_FOLDS,
    "n_oof_folds": N_OOF_FOLDS,
}

print("Run configuration:")
display(pd.DataFrame([RUN_CONFIG]))



## 1.1 IBM/Qiskit collection before synthetic generation — live-required full-paper mode

For the full-paper run, external evidence must enter before synthetic dataset generation. This section collects live backend metadata from IBM Quantum/Qiskit Runtime and saves them in the persistent `external_qpu_cache` directory.

In this corrected version, cached metadata, Qiskit snapshots/fake backends, and static plausibility references are **not** allowed to replace live IBM/Qiskit Runtime metadata. If live collection is unavailable, the notebook raises an error and stops before generating the synthetic dataset. This prevents the paper from accidentally reporting a static plausibility check as live Runtime validation.

Provide the IBM Quantum token and Runtime instance through environment variables (`IBM_QUANTUM_TOKEN` and `IBM_QUANTUM_INSTANCE`) or through the interactive prompt in the IBM configuration cell.


In [ ]:
# ============================================================
# Installation of required Qiskit packages
# ============================================================

import sys
import subprocess
import importlib

def install_and_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            pip_name
        ])
        return importlib.import_module(import_name)

qiskit = install_and_import("qiskit", "qiskit")
qiskit_ibm_runtime = install_and_import(
    "qiskit_ibm_runtime",
    "qiskit-ibm-runtime"
)

print("qiskit installed:", qiskit.__version__)
print("qiskit-ibm-runtime installed:", qiskit_ibm_runtime.__version__)
# Qiskit Aer is required for local validation with small circuits.
# The canonical import may vary across versions; therefore, the package is installed here
# and Section 26.2 handles the import in a compatible way.
try:
    qiskit_aer = install_and_import("qiskit_aer", "qiskit-aer")
    print("qiskit-aer installed:", getattr(qiskit_aer, "__version__", "unknown"))
except Exception as e:
    qiskit_aer = None
    print("Warning: qiskit-aer could not be installed/imported in this run:", type(e).__name__, str(e)[:180])



In [ ]:
# ============================================================
# 1.0 IBM Quantum credentials for the live-required run
# ============================================================
# Preferred use in Colab/local execution:
#   1. Run this cell.
#   2. Paste your IBM Cloud API key and IBM Quantum Runtime CRN/instance when prompted.
#   3. Continue the notebook.
#
# You may also set the environment variables before running the notebook:
#   IBM_QUANTUM_TOKEN
#   IBM_QUANTUM_INSTANCE

import os
from getpass import getpass

if not os.environ.get("IBM_QUANTUM_TOKEN"):
    os.environ["IBM_QUANTUM_TOKEN"] = getpass("Paste your IBM Cloud API key for IBM Quantum Runtime: ")
else:
    print("IBM_QUANTUM_TOKEN already set.")

if not os.environ.get("IBM_QUANTUM_INSTANCE"):
    os.environ["IBM_QUANTUM_INSTANCE"] = getpass("Paste your IBM Quantum Runtime instance CRN: ")
else:
    print("IBM_QUANTUM_INSTANCE already set.")


In [ ]:
# ============================================================
# 1.1a Optional IBM Quantum Runtime connection diagnostics
# ============================================================
# This cell is diagnostic only. It does not interrupt the notebook if the connection fails.
# Use it to check token, CRN, and channel before the main collection step.

import os
from getpass import getpass

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    _diagnostic_runtime_available = True
except Exception as e:
    _diagnostic_runtime_available = False
    print("qiskit-ibm-runtime unavailable:", type(e).__name__, str(e))


def _diagnostic_clean_secret(value):
    if value is None:
        return ""
    value = str(value).strip()
    if (
        (value.startswith("'") and value.endswith("'"))
        or (value.startswith('"') and value.endswith('"'))
    ):
        value = value[1:-1].strip()
    return value


def _diagnostic_instance_status(instance):
    instance = _diagnostic_clean_secret(instance)
    return {
        "instance_provided": bool(instance),
        "starts_with_crn_v1": instance.startswith("crn:v1:"),
        "contains_quantum_computing": "quantum-computing" in instance.lower(),
        "looks_like_url": instance.lower().startswith(("http://", "https://")),
        "contains_watson_or_assistant": ("watson" in instance.lower()) or ("assistant" in instance.lower()),
        "prefix": instance[:28] if instance else "",
        "suffix": instance[-16:] if instance else "",
    }


def _diagnostic_build_attempts(token, instance):
    token = _diagnostic_clean_secret(token)
    instance = _diagnostic_clean_secret(instance)

    invalid_instance = (
        instance.lower().startswith(("http://", "https://"))
        or "watson" in instance.lower()
        or "assistant" in instance.lower()
    )
    valid_instance = bool(instance) and not invalid_instance

    attempts = []
    if token and valid_instance:
        attempts.append({"channel": "ibm_quantum_platform", "token": token, "instance": instance})
        attempts.append({"channel": "ibm_cloud", "token": token, "instance": instance})
    if token:
        attempts.append({"channel": "ibm_quantum_platform", "token": token})
        attempts.append({"channel": "ibm_cloud", "token": token})
    attempts.append({"channel": "ibm_quantum_platform"})
    attempts.append({"channel": "ibm_cloud"})
    return attempts, invalid_instance


if _diagnostic_runtime_available:
    token = (
        os.environ.get("IBM_QUANTUM_TOKEN")
        or os.environ.get("QISKIT_IBM_TOKEN")
        or ""
    )
    instance = (
        os.environ.get("IBM_QUANTUM_INSTANCE")
        or os.environ.get("QISKIT_IBM_INSTANCE")
        or ""
    )

    print("CRN/instance diagnostics:")
    for k, v in _diagnostic_instance_status(instance).items():
        print(f"- {k}: {v}")

    attempts, invalid_instance = _diagnostic_build_attempts(token, instance)
    if invalid_instance:
        print("WARNING: the instance appears to be a URL or a non-Quantum service. It will be ignored in this test.")

    connected = False
    last_error = None
    for attempt in attempts:
        safe_attempt = {k: ("***" if k == "token" else v) for k, v in attempt.items()}
        try:
            service = QiskitRuntimeService(**attempt)
            backends = list(service.backends())
            print("Connection OK.")
            print("Parameters used:", safe_attempt)
            try:
                print("Active instance:", service.active_instance())
            except Exception:
                pass
            print("Available backends:", len(backends))
            print([b.name() if callable(getattr(b, "name", None)) else getattr(b, "name", str(b)) for b in backends[:10]])
            connected = True
            break
        except Exception as e:
            last_error = e
            print("Attempt failed:", safe_attempt, "=>", type(e).__name__, str(e)[:180])

    if not connected:
        print("No connection attempt worked.")
        if last_error is not None:
            print("Last error:", type(last_error).__name__, str(last_error))
        print("Check whether the API key belongs to the same account as the IBM Quantum Runtime instance and whether the CRN was copied completely.")


In [ ]:

# ============================================================
# 1.1 Source-aware IBM/Qiskit collection before synthetic generation
# ============================================================
# Methodological objective:
# Collect live metadata from IBM/Qiskit Runtime when credentials are available.
# If live metadata are not available, use cache/static fallback only as a clearly
# labeled plausibility reference, never as evidence of live IBM Runtime validation.
#
# Main correction in this version:
# - the connection is robustly tested;
# - the API key is validated in IBM IAM before QiskitRuntimeService;
# - backends are initially listed without filters using service.backends();
# - attributes such as simulator, operational, pending_jobs, and status are collected
#   as observed variables, not used as initial elimination filters;
# - real metadata are saved in cache for reproducibility.

import os
import sys
import json
import subprocess
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ============================================================
# Execution flags
# ============================================================

FULL_PAPER_COLLECT_IBM_REAL_DATA = True
FULL_PAPER_INSTALL_QISKIT_IF_MISSING = True
FULL_PAPER_SAVE_IBM_ACCOUNT_LOCALLY = False
FULL_PAPER_ASK_TOKEN_IF_MISSING = True
FULL_PAPER_MAX_BACKENDS_TO_COLLECT = None

# For the final paper run, keep True.
FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR = True

# Use True only for debugging without credentials.
# If True, the notebook may use a static reference, but the run is no longer
# "IBM Runtime anchored" and becomes only "plausibility anchored".
FULL_PAPER_ALLOW_STATIC_FALLBACK_FOR_DRY_RUN = False
FULL_PAPER_REQUIRE_LIVE_CURRENT_RUN = True


# ============================================================
# Directories and caches
# ============================================================

try:
    EXTERNAL_CACHE_DIR
except NameError:
    EXTERNAL_CACHE_DIR = Path("external_qpu_cache")

EXTERNAL_CACHE_DIR = Path(EXTERNAL_CACHE_DIR)
EXTERNAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

IBM_LIVE_BACKENDS_CACHE = EXTERNAL_CACHE_DIR / "ibm_backend_metadata_live_v9_6.csv"
IBM_LIVE_QUBITS_CACHE = EXTERNAL_CACHE_DIR / "ibm_qubit_properties_live_v9_6.csv"
IBM_LIVE_GATES_CACHE = EXTERNAL_CACHE_DIR / "ibm_gate_properties_live_v9_6.csv"
IBM_LIVE_STATUS_CACHE = EXTERNAL_CACHE_DIR / "ibm_queue_status_live_v9_6.csv"
IBM_LIVE_LOG_CACHE = EXTERNAL_CACHE_DIR / "ibm_collection_log_live_v9_6.csv"
IBM_CONNECTION_DIAG_CACHE = EXTERNAL_CACHE_DIR / "ibm_runtime_connection_diagnostics_v9_6.csv"
QPU_ANCHOR_CACHE = EXTERNAL_CACHE_DIR / "external_qpu_metadata_for_generation_v9_6.csv"

# Caches antigos, caso existam.
PREVIOUS_IBM_LIVE_BACKENDS_CACHE = EXTERNAL_CACHE_DIR / "ibm_backend_metadata_live_v9_6.csv"
PREVIOUS_QPU_ANCHOR_CACHE = EXTERNAL_CACHE_DIR / "external_qpu_metadata_for_generation_v9_6.csv"


# ============================================================
# General utility functions
# ============================================================

def _fp_safe_read_csv(path):
    """Reads CSV robustly and returns an empty DataFrame when the file is absent or invalid."""
    try:
        path = Path(path)
        if (not path.exists()) or path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def _fp_safe_csv_row_count(path):
    return int(len(_fp_safe_read_csv(path)))


def _fp_install_if_missing(import_name, pip_name):
    """Installs a package if the import is unavailable."""
    try:
        __import__(import_name)
        return "already_installed"
    except Exception:
        if not FULL_PAPER_INSTALL_QISKIT_IF_MISSING:
            return "missing_not_installed"

        print(f"Installing {pip_name}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            pip_name,
        ])
        __import__(import_name)
        return "installed_now"


def _fp_clean_secret(value):
    """Removes accidental spaces and quotes from tokens/CRNs."""
    if value is None:
        return ""

    value = str(value).strip()

    if (
        (value.startswith("'") and value.endswith("'"))
        or (value.startswith('"') and value.endswith('"'))
    ):
        value = value[1:-1].strip()

    return value


def _fp_safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def _fp_safe_bool(x):
    try:
        if pd.isna(x):
            return np.nan
    except Exception:
        pass

    if isinstance(x, bool):
        return x

    if isinstance(x, str):
        low = x.strip().lower()
        if low in ["true", "1", "yes", "sim"]:
            return True
        if low in ["false", "0", "no", "no", "nao"]:
            return False

    try:
        return bool(x)
    except Exception:
        return np.nan


def _fp_safe_median(values):
    vals = []

    for value in values:
        value = _fp_safe_float(value)
        if not pd.isna(value):
            vals.append(value)

    return float(np.median(vals)) if vals else np.nan


def _fp_backend_name(backend):
    """Gets the backend name in a way compatible with different Qiskit versions."""
    name = getattr(backend, "name", None)

    try:
        return name() if callable(name) else str(name)
    except Exception:
        return str(name)


def _fp_safe_getattr(obj, attr, default=None):
    try:
        value = getattr(obj, attr, default)
        return value() if callable(value) else value
    except Exception:
        return default


# ============================================================
# API key and instance diagnostics
# ============================================================

def _fp_instance_is_invalid(instance):
    """Detects clearly invalid instance values for Qiskit Runtime."""
    instance = _fp_clean_secret(instance)

    if not instance:
        return False

    low = instance.lower()

    return (
        low.startswith(("http://", "https://"))
        or "watson" in low
        or "assistant" in low
    )


def _fp_instance_diagnostics(instance):
    """Returns safe CRN/instance diagnostics without exposing the full value."""
    instance = _fp_clean_secret(instance)

    return {
        "provided": bool(instance),
        "starts_with_crn_v1": instance.startswith("crn:v1:"),
        "contains_quantum_computing": "quantum-computing" in instance.lower(),
        "looks_like_url": instance.lower().startswith(("http://", "https://")),
        "contains_watson_or_assistant": (
            ("watson" in instance.lower()) or ("assistant" in instance.lower())
        ),
        "prefix": instance[:32] if instance else "",
        "suffix": instance[-18:] if instance else "",
    }


def _fp_validate_ibm_api_key_with_iam(api_key):
    """
    Validates the API key in IBM IAM before calling QiskitRuntimeService.
    This avoids ambiguous messages when the issue is an invalid key.
    """
    import requests

    api_key = _fp_clean_secret(api_key)

    if not api_key:
        return {
            "ok": False,
            "status_code": None,
            "message": "API key vazia.",
        }

    try:
        resp = requests.post(
            "https://iam.cloud.ibm.com/identity/token",
            data={
                "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
                "apikey": api_key,
            },
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
                "Accept": "application/json",
            },
            timeout=30,
        )

        if resp.status_code == 200:
            return {
                "ok": True,
                "status_code": 200,
                "message": "API key valid in IBM IAM.",
            }

        return {
            "ok": False,
            "status_code": resp.status_code,
            "message": resp.text[:700],
        }

    except Exception as e:
        return {
            "ok": False,
            "status_code": None,
            "message": f"{type(e).__name__}: {str(e)[:500]}",
        }


# ============================================================
# Robust connection to IBM/Qiskit Runtime
# ============================================================

def _fp_build_runtime_service_attempts(token, instance):
    """
    Builds connection attempts compatible with recent qiskit-ibm-runtime versions.
    """
    token = _fp_clean_secret(token)
    instance = _fp_clean_secret(instance)
    invalid_instance = _fp_instance_is_invalid(instance)
    usable_instance = bool(instance) and not invalid_instance

    attempts = []

    # Prioridade: IBM Quantum Platform com token + CRN.
    if token and usable_instance:
        attempts.append({
            "channel": "ibm_quantum_platform",
            "token": token,
            "instance": instance,
        })
        attempts.append({
            "channel": "ibm_cloud",
            "token": token,
            "instance": instance,
        })

    # Without an explicit instance: try to discover the account instance.
    if token:
        attempts.append({
            "channel": "ibm_quantum_platform",
            "token": token,
        })
        attempts.append({
            "channel": "ibm_cloud",
            "token": token,
        })

    # Account previously saved/configured in the environment.
    attempts.append({"channel": "ibm_quantum_platform"})
    attempts.append({"channel": "ibm_cloud"})

    return attempts, invalid_instance


def _fp_connect_ibm_runtime(QiskitRuntimeService, token="", instance=""):
    """
    Attempts to connect to IBM Quantum Runtime and forces a service.backends() call
    to validate the account/instance.
    """
    attempts, invalid_instance = _fp_build_runtime_service_attempts(token, instance)
    diagnostics = []
    last_error = None
    used_kwargs = None
    service = None

    if invalid_instance:
        print("WARNING: IBM_QUANTUM_INSTANCE/QISKIT_IBM_INSTANCE appears invalid.")
        print("Use the full CRN of the IBM Quantum Runtime instance, not a Watson/Assistant URL.")

    for kwargs in attempts:
        clean_kwargs = {
            k: v
            for k, v in kwargs.items()
            if v not in [None, ""]
        }

        safe_kwargs = {
            k: ("***" if k == "token" else v)
            for k, v in clean_kwargs.items()
        }

        try:
            candidate_service = QiskitRuntimeService(**clean_kwargs)

            # Validate the account/instance.
            candidate_backends = list(candidate_service.backends())

            service = candidate_service
            used_kwargs = clean_kwargs

            diagnostics.append({
                "attempt": json.dumps(safe_kwargs),
                "status": "ok",
                "message": f"connected; backends={len(candidate_backends)}",
            })

            return service, used_kwargs, diagnostics

        except Exception as e:
            last_error = e
            diagnostics.append({
                "attempt": json.dumps(safe_kwargs),
                "status": "error",
                "message": f"{type(e).__name__}: {str(e)[:300]}",
            })

    raise RuntimeError(
        "Could not connect to IBM Quantum Runtime. "
        "Confira se a API key pertence à mesma conta da open-instance, "
        "whether the CRN was fully copied using the Copy CRN button "
        "and whether the instance is quantum-computing. "
        f"Last error: {type(last_error).__name__}: {str(last_error)}"
    )


# ============================================================
# Extraction of configuration/properties/status metadata
# ============================================================

def _fp_get_backend_config_info(backend):
    """Extracts structural backend information through configuration/target."""
    info = {
        "backend": _fp_backend_name(backend),
        "n_qubits": np.nan,
        "basis_gates_count": np.nan,
        "coupling_edges": np.nan,
        "coupling_density": np.nan,
        "simulator": np.nan,
        "backend_version": "",
        "processor_type": "",
    }

    # BackendV1/configuration
    try:
        cfg = backend.configuration()

        info["n_qubits"] = getattr(
            cfg,
            "n_qubits",
            getattr(cfg, "num_qubits", np.nan),
        )

        basis = getattr(cfg, "basis_gates", []) or []
        info["basis_gates_count"] = len(basis)

        cm = getattr(cfg, "coupling_map", None)
        if cm is not None:
            info["coupling_edges"] = len(cm)

        info["simulator"] = _fp_safe_bool(getattr(cfg, "simulator", np.nan))
        info["backend_version"] = str(getattr(cfg, "backend_version", ""))
    except Exception:
        pass

    # BackendV2/target
    try:
        target = getattr(backend, "target", None)

        if target is not None:
            if hasattr(target, "num_qubits") and target.num_qubits is not None:
                info["n_qubits"] = target.num_qubits

            if hasattr(target, "operation_names"):
                info["basis_gates_count"] = len(list(target.operation_names))

            if hasattr(target, "build_coupling_map"):
                cm = target.build_coupling_map()
                if cm is not None:
                    try:
                        info["coupling_edges"] = len(cm.get_edges())
                    except Exception:
                        pass
    except Exception:
        pass

    # Atributos diretos
    try:
        direct_n = _fp_safe_getattr(backend, "num_qubits", None)
        if direct_n is not None:
            info["n_qubits"] = direct_n
    except Exception:
        pass

    try:
        direct_sim = _fp_safe_getattr(backend, "simulator", None)
        if direct_sim is not None:
            info["simulator"] = _fp_safe_bool(direct_sim)
    except Exception:
        pass

    try:
        processor_type = _fp_safe_getattr(backend, "processor_type", "")
        info["processor_type"] = str(processor_type)
    except Exception:
        pass

    # Densidade de acoplamento
    try:
        n = int(info["n_qubits"])

        if n > 1 and not pd.isna(info["coupling_edges"]):
            info["coupling_density"] = float(info["coupling_edges"] / (n * (n - 1)))
    except Exception:
        pass

    return info


def _fp_extract_props(backend):
    """
    Extracts physical backend properties:
    - T1/T2 per qubit;
    - readout error;
    - gate errors and durations.
    """
    backend_name = _fp_backend_name(backend)

    qrows = []
    grows = []

    t1_vals = []
    t2_vals = []
    readout_vals = []
    oneq_errors = []
    twoq_errors = []
    gate_lengths = []

    try:
        props = backend.properties()
    except Exception:
        props = None

    if props is not None:
        # Propriedades de qubits
        try:
            qubits = getattr(props, "qubits", []) or []

            for q_idx, qprops in enumerate(qubits):
                qd = {
                    "backend": backend_name,
                    "qubit": q_idx,
                    "t1_us": np.nan,
                    "t2_us": np.nan,
                    "readout_error": np.nan,
                }

                for item in qprops:
                    name = str(getattr(item, "name", ""))
                    value = _fp_safe_float(getattr(item, "value", np.nan))
                    unit = str(getattr(item, "unit", ""))

                    if name == "T1":
                        qd["t1_us"] = value if unit.lower() in ["us", "µs"] else value * 1e6
                        t1_vals.append(qd["t1_us"])

                    elif name == "T2":
                        qd["t2_us"] = value if unit.lower() in ["us", "µs"] else value * 1e6
                        t2_vals.append(qd["t2_us"])

                    elif name == "readout_error":
                        qd["readout_error"] = value
                        readout_vals.append(value)

                qrows.append(qd)

        except Exception:
            pass

        # Propriedades de gates
        try:
            for gate in getattr(props, "gates", []) or []:
                gate_name = str(getattr(gate, "gate", ""))
                qubits = list(getattr(gate, "qubits", []) or [])

                gate_error = np.nan
                gate_length_ns = np.nan

                for param in getattr(gate, "parameters", []) or []:
                    pname = str(getattr(param, "name", ""))
                    pval = _fp_safe_float(getattr(param, "value", np.nan))
                    punit = str(getattr(param, "unit", ""))

                    if pname == "gate_error":
                        gate_error = pval

                    elif pname == "gate_length":
                        gate_length_ns = pval if punit.lower() == "ns" else pval * 1e9

                grows.append({
                    "backend": backend_name,
                    "gate": gate_name,
                    "qubits": json.dumps(qubits),
                    "n_gate_qubits": len(qubits),
                    "gate_error": gate_error,
                    "gate_length_ns": gate_length_ns,
                })

                if not pd.isna(gate_error):
                    if len(qubits) <= 1:
                        oneq_errors.append(gate_error)
                    elif len(qubits) >= 2:
                        twoq_errors.append(gate_error)

                if not pd.isna(gate_length_ns):
                    gate_lengths.append(gate_length_ns)

        except Exception:
            pass

    return {
        "median_t1_us": _fp_safe_median(t1_vals),
        "median_t2_us": _fp_safe_median(t2_vals),
        "median_readout_error": _fp_safe_median(readout_vals),
        "median_1q_gate_error": _fp_safe_median(oneq_errors),
        "median_2q_gate_error": _fp_safe_median(twoq_errors),
        "median_gate_length_ns": _fp_safe_median(gate_lengths),
    }, qrows, grows


def _fp_collect_backend(backend):
    """Collects the aggregate backend row and detailed qubit/gate rows."""
    base = _fp_get_backend_config_info(backend)
    prop_summary, qrows, grows = _fp_extract_props(backend)

    base.update(prop_summary)

    # Public IBM Quantum is treated here as superconducting modality.
    base["qpu_modality"] = "superconducting"
    base["backend_type"] = "qpu" if base.get("simulator") is not True else "simulator"
    base["backend_family"] = (
        "qpu_cloud_superconducting"
        if base["backend_type"] == "qpu"
        else "simulator"
    )

    base["source"] = "ibm_qiskit_runtime_live_metadata"
    base["collection_status"] = "ok"
    base["evidence_level"] = "live_runtime_metadata"
    base["collected_at_utc"] = datetime.now(timezone.utc).isoformat()

    try:
        status = backend.status()

        base["operational"] = _fp_safe_bool(getattr(status, "operational", np.nan))
        base["pending_jobs"] = _fp_safe_float(getattr(status, "pending_jobs", np.nan))
        base["status_msg"] = str(getattr(status, "status_msg", ""))
    except Exception:
        base["operational"] = np.nan
        base["pending_jobs"] = np.nan
        base["status_msg"] = "status_unavailable"

    return base, qrows, grows


# ============================================================
# Optional static reference for debugging
# ============================================================

def _fp_static_anchor_reference():
    """
    Static reference only for debugging without credentials.
    It should not be used as the main evidence in the full paper.
    """
    return pd.DataFrame([
        {
            "backend": "static_superconducting_reference_low_error",
            "qpu_modality": "superconducting",
            "backend_type": "qpu",
            "backend_family": "qpu_cloud_superconducting",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "evidence_level": "plausibility_or_static_reference",
            "n_qubits": 127,
            "median_1q_gate_error": 3.0e-4,
            "median_2q_gate_error": 8.0e-3,
            "median_readout_error": 1.5e-2,
            "coupling_density": 0.020,
            "median_t1_us": 180.0,
            "median_t2_us": 120.0,
            "pending_jobs": np.nan,
            "operational": np.nan,
        },
        {
            "backend": "static_superconducting_reference_moderate_error",
            "qpu_modality": "superconducting",
            "backend_type": "qpu",
            "backend_family": "qpu_cloud_superconducting",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "evidence_level": "plausibility_or_static_reference",
            "n_qubits": 127,
            "median_1q_gate_error": 8.0e-4,
            "median_2q_gate_error": 2.0e-2,
            "median_readout_error": 3.0e-2,
            "coupling_density": 0.030,
            "median_t1_us": 90.0,
            "median_t2_us": 70.0,
            "pending_jobs": np.nan,
            "operational": np.nan,
        },
        {
            "backend": "static_iontrap_reference_high_connectivity",
            "qpu_modality": "iontrap",
            "backend_type": "qpu",
            "backend_family": "qpu_cloud_iontrap",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "evidence_level": "plausibility_or_static_reference",
            "n_qubits": 32,
            "median_1q_gate_error": 5.0e-4,
            "median_2q_gate_error": 1.2e-2,
            "median_readout_error": 2.0e-2,
            "coupling_density": 0.95,
            "median_t1_us": 1.0e6,
            "median_t2_us": 1.0e5,
            "pending_jobs": np.nan,
            "operational": np.nan,
        },
    ])


# ============================================================
# Main IBM/Qiskit Runtime collection
# ============================================================

ibm_backend_metadata_live = pd.DataFrame()
ibm_qubit_properties_live = pd.DataFrame()
ibm_gate_properties_live = pd.DataFrame()
ibm_queue_status_live = pd.DataFrame()
ibm_collection_log_live = []
ibm_runtime_connection_diagnostics = []

if FULL_PAPER_COLLECT_IBM_REAL_DATA:
    try:
        # 1) Instala/importa Qiskit e qiskit-ibm-runtime.
        for import_name, pip_name in [
            ("qiskit", "qiskit"),
            ("qiskit_ibm_runtime", "qiskit-ibm-runtime"),
        ]:
            print(pip_name, _fp_install_if_missing(import_name, pip_name))

        from qiskit_ibm_runtime import QiskitRuntimeService

        # 2) Recupera token de ambiente ou pede interativamente.
        token = _fp_clean_secret(
            os.environ.get("IBM_QUANTUM_TOKEN")
            or os.environ.get("QISKIT_IBM_TOKEN")
            or ""
        )

        if not token and FULL_PAPER_ASK_TOKEN_IF_MISSING:
            token = _fp_clean_secret(getpass(
                "Paste the IBM Cloud API key for IBM Quantum Runtime. "
                "Empty Enter tries an already configured account: "
            ))

        # 3) Recupera CRN/instance.
        ibm_instance = _fp_clean_secret(
            os.environ.get("IBM_QUANTUM_INSTANCE")
            or os.environ.get("QISKIT_IBM_INSTANCE")
            or ""
        )

        if not ibm_instance and FULL_PAPER_ASK_TOKEN_IF_MISSING:
            ibm_instance = _fp_clean_secret(getpass(
                "Enter the full CRN of the IBM Quantum Runtime instance. "
                "Empty Enter tries automatic discovery: "
            ))

        # 4) Safe instance diagnostics.
        print("IBM Quantum Runtime instance diagnostics:")

        for key, value in _fp_instance_diagnostics(ibm_instance).items():
            print(f"- {key}: {value}")

        # 5) Validate the API key in IBM IAM before calling QiskitRuntimeService.
        if token:
            iam_check = _fp_validate_ibm_api_key_with_iam(token)

            print("IAM API key validation:")
            print("- ok:", iam_check["ok"])
            print("- status_code:", iam_check["status_code"])
            print("- message:", iam_check["message"][:300])

            if not iam_check["ok"]:
                raise RuntimeError(
                    "The provided API key was not accepted by IBM IAM. "
                    "Crie uma nova IBM Cloud API key e copie o valor secreto completo, "
                    "not the key ID."
                )

        # 6) Conecta ao Runtime.
        service, service_kwargs, ibm_runtime_connection_diagnostics = _fp_connect_ibm_runtime(
            QiskitRuntimeService,
            token=token,
            instance=ibm_instance,
        )

        safe_service_kwargs = {
            k: ("***" if k == "token" else v)
            for k, v in service_kwargs.items()
        }

        print("QiskitRuntimeService inicializado com sucesso.")
        print("Parâmetros efetivos:", safe_service_kwargs)

        try:
            print("Active instance:", service.active_instance())
        except Exception:
            pass

        # 7) Opcional: salva conta localmente.
        if token and FULL_PAPER_SAVE_IBM_ACCOUNT_LOCALLY:
            save_kwargs = dict(service_kwargs)
            save_kwargs["overwrite"] = True
            save_kwargs["set_as_default"] = True
            QiskitRuntimeService.save_account(**save_kwargs)

        # 8) Lista backends SEM FILTRO RESTRITIVO.
        try:
            all_backends = list(service.backends())
        except Exception as e:
            raise RuntimeError(
                f"Failed to list IBM/Qiskit Runtime backends: "
                f"{type(e).__name__}: {e}"
            )

        print("IBM/Qiskit backends found without filter:", len(all_backends))
        print([_fp_backend_name(b) for b in all_backends])

        # 9) Local filtering: remove simulators only when the attribute is reliable.
        physical_backends = []

        for backend in all_backends:
            try:
                info = _fp_get_backend_config_info(backend)
                is_simulator = _fp_safe_bool(info.get("simulator", False))
            except Exception:
                is_simulator = False

            if is_simulator is not True:
                physical_backends.append(backend)

        if len(physical_backends) > 0:
            backends = physical_backends
        else:
            print(
                "Warning: physical backends could not be identified by the simulator attribute. "
                "Using all backends returned by service.backends()."
            )
            backends = all_backends

        if FULL_PAPER_MAX_BACKENDS_TO_COLLECT is not None:
            backends = backends[:FULL_PAPER_MAX_BACKENDS_TO_COLLECT]

        print("IBM/Qiskit backends selected for collection:", len(backends))
        print([_fp_backend_name(b) for b in backends])

        # 10) Collect metadata backend by backend.
        backend_rows = []
        qubit_rows = []
        gate_rows = []
        status_rows = []

        for backend in backends:
            backend_name = _fp_backend_name(backend)

            try:
                brow, qrows, grows = _fp_collect_backend(backend)

                backend_rows.append(brow)
                qubit_rows.extend(qrows)
                gate_rows.extend(grows)

                status_rows.append({
                    "backend": brow.get("backend"),
                    "operational": brow.get("operational"),
                    "pending_jobs": brow.get("pending_jobs"),
                    "status_msg": brow.get("status_msg"),
                    "collected_at_utc": brow.get("collected_at_utc"),
                })

                ibm_collection_log_live.append({
                    "backend": brow.get("backend"),
                    "status": "ok",
                    "message": "metadata_collected",
                })

            except Exception as e:
                ibm_collection_log_live.append({
                    "backend": backend_name,
                    "status": "error",
                    "message": f"{type(e).__name__}: {str(e)[:220]}",
                })

        ibm_backend_metadata_live = pd.DataFrame(backend_rows)
        ibm_qubit_properties_live = pd.DataFrame(qubit_rows)
        ibm_gate_properties_live = pd.DataFrame(gate_rows)
        ibm_queue_status_live = pd.DataFrame(status_rows)

        # 11) Save logs and diagnostics.
        pd.DataFrame(ibm_collection_log_live).to_csv(
            IBM_LIVE_LOG_CACHE,
            index=False,
        )

        if ibm_runtime_connection_diagnostics:
            pd.DataFrame(ibm_runtime_connection_diagnostics).to_csv(
                IBM_CONNECTION_DIAG_CACHE,
                index=False,
            )

        # 12) Save real metadata if collected.
        if not ibm_backend_metadata_live.empty:
            ibm_backend_metadata_live.to_csv(
                IBM_LIVE_BACKENDS_CACHE,
                index=False,
            )

            ibm_qubit_properties_live.to_csv(
                IBM_LIVE_QUBITS_CACHE,
                index=False,
            )

            ibm_gate_properties_live.to_csv(
                IBM_LIVE_GATES_CACHE,
                index=False,
            )

            ibm_queue_status_live.to_csv(
                IBM_LIVE_STATUS_CACHE,
                index=False,
            )

            print("Real IBM/Qiskit metadata collected:", ibm_backend_metadata_live.shape)

            cols_to_show = [
                "backend",
                "n_qubits",
                "simulator",
                "operational",
                "pending_jobs",
                "median_t1_us",
                "median_t2_us",
                "median_1q_gate_error",
                "median_2q_gate_error",
                "median_readout_error",
                "coupling_density",
                "source",
                "collection_status",
                "evidence_level",
            ]

            cols_to_show = [
                c for c in cols_to_show
                if c in ibm_backend_metadata_live.columns
            ]

            display(ibm_backend_metadata_live[cols_to_show])

        else:
            print("No real IBM/Qiskit backend was collected in this run.")

    except Exception as e:
        ibm_collection_log_live.append({
            "backend": "global",
            "status": "error",
            "message": f"{type(e).__name__}: {str(e)[:300]}",
        })

        pd.DataFrame(ibm_collection_log_live).to_csv(
            IBM_LIVE_LOG_CACHE,
            index=False,
        )

        if ibm_runtime_connection_diagnostics:
            pd.DataFrame(ibm_runtime_connection_diagnostics).to_csv(
                IBM_CONNECTION_DIAG_CACHE,
                index=False,
            )

        print(
            "Real IBM/Qiskit collection not executed:",
            type(e).__name__,
            str(e),
        )




# ============================================================
# Source-aware evidence classification
# ============================================================

def classify_external_anchor_evidence(df_obj):
    """Classifies the evidence level of the external QPU metadata.

    This function is intentionally conservative. Static references and fallback
    references are not allowed to masquerade as live IBM/Qiskit Runtime metadata.
    The returned scope should be used verbatim in tables, figures, and paper text.
    """
    if not isinstance(df_obj, pd.DataFrame) or df_obj.empty:
        return pd.DataFrame([{
            "anchor_evidence_scope": "no_external_reference",
            "live_ibm_runtime_available": False,
            "cached_ibm_runtime_available": False,
            "static_reference_used": False,
            "recommended_claim": "synthetic benchmark without external QPU anchor in this run",
            "paper_wording": "No external QPU metadata were available for this run.",
        }])

    sources = set(df_obj.get("source", pd.Series(dtype=str)).astype(str).str.lower())
    statuses = set(df_obj.get("collection_status", pd.Series(dtype=str)).astype(str).str.lower())
    evidence = set(df_obj.get("evidence_level", pd.Series(dtype=str)).astype(str).str.lower())
    priority = set(df_obj.get("anchor_priority", pd.Series(dtype=str)).astype(str).str.lower())

    has_live = (
        "ibm_qiskit_runtime_live_metadata" in sources
        or "live_runtime_metadata" in evidence
        or "live_current_run" in priority
    )
    has_cache = (
        any("ibm_qiskit_runtime" in s for s in sources)
        and not has_live
    ) or any("cache" in p for p in priority) or "ibm_runtime_cached_metadata" in evidence
    has_static = (
        any("static_plausibility_reference" in s for s in sources)
        or "fallback_static_reference" in statuses
        or "plausibility_or_static_reference" in evidence
        or any("static" in p for p in priority)
    )
    has_snapshot = "qiskit_fake_backend_snapshot" in statuses or any("fake" in s or "snapshot" in s for s in sources)

    if has_live:
        scope = "live_ibm_qiskit_runtime_metadata"
        claim = "partially anchored in live IBM/Qiskit Runtime metadata"
        wording = "Live IBM/Qiskit Runtime metadata were available in this run and were used as the superconducting QPU reference."
    elif has_cache:
        scope = "cached_ibm_qiskit_metadata"
        claim = "anchored in cached IBM/Qiskit metadata, not live metadata from this run"
        wording = "Cached IBM/Qiskit metadata were used; this is not a live Runtime synchronization for the current run."
    elif has_snapshot:
        scope = "qiskit_snapshot_or_fake_backend_reference"
        claim = "externally checked using Qiskit snapshot/fake-backend references"
        wording = "Qiskit snapshot or fake-backend references were used as a plausibility check, not as live Runtime metadata."
    elif has_static:
        scope = "static_plausibility_reference_not_current"
        claim = "checked against a documented static plausibility reference, not live IBM/Qiskit Runtime metadata"
        wording = "A non-current static plausibility reference was used; the run must not be described as live IBM/Qiskit Runtime validated."
    else:
        scope = "external_reference_unknown_level"
        claim = "externally checked with an unknown or mixed evidence level"
        wording = "External metadata were present, but the source level could not be classified confidently."

    return pd.DataFrame([{
        "anchor_evidence_scope": scope,
        "live_ibm_runtime_available": bool(has_live),
        "cached_ibm_runtime_available": bool(has_cache),
        "static_reference_used": bool(has_static),
        "qiskit_snapshot_used": bool(has_snapshot),
        "source_values": ";".join(sorted(sources)),
        "collection_status_values": ";".join(sorted(statuses)),
        "evidence_level_values": ";".join(sorted(evidence)),
        "anchor_priority_values": ";".join(sorted(priority)),
        "recommended_claim": claim,
        "paper_wording": wording,
    }])

# ============================================================
# Consolidation of the external anchor for synthetic generation
# ============================================================

external_qpu_metadata = pd.DataFrame()

# 1) Final-paper requirement: only newly collected live metadata from this execution
#    can anchor the QPU catalog. Cached, snapshot, and static references are disabled
#    when FULL_PAPER_REQUIRE_LIVE_CURRENT_RUN=True.
if not ibm_backend_metadata_live.empty:
    external_qpu_metadata = ibm_backend_metadata_live.copy()
    external_qpu_metadata["anchor_priority"] = "live_current_run"

elif FULL_PAPER_REQUIRE_LIVE_CURRENT_RUN:
    raise RuntimeError(
        "Live IBM/Qiskit Runtime metadata were not collected in this execution. "
        "The notebook is configured for full-paper live validation and will not use "
        "cached, snapshot, or static fallback references. Check IBM_QUANTUM_TOKEN, "
        "IBM_QUANTUM_INSTANCE, account permissions, qiskit-ibm-runtime installation, "
        "and network access, then rerun from Section 1."
    )

# 2) Debug-only paths below are unreachable in the final-paper configuration.
elif _fp_safe_csv_row_count(IBM_LIVE_BACKENDS_CACHE) > 0:
    external_qpu_metadata = _fp_safe_read_csv(IBM_LIVE_BACKENDS_CACHE)
    external_qpu_metadata["anchor_priority"] = "live_cache_v9_6"

elif _fp_safe_csv_row_count(PREVIOUS_IBM_LIVE_BACKENDS_CACHE) > 0:
    external_qpu_metadata = _fp_safe_read_csv(PREVIOUS_IBM_LIVE_BACKENDS_CACHE)
    external_qpu_metadata["anchor_priority"] = "live_cache_previous_version"

elif FULL_PAPER_ALLOW_STATIC_FALLBACK_FOR_DRY_RUN:
    external_qpu_metadata = _fp_static_anchor_reference()
    external_qpu_metadata["anchor_priority"] = "static_dry_run_reference"

else:
    external_qpu_metadata = pd.DataFrame()

# Minimal normalization of expected columns.
if not external_qpu_metadata.empty:
    if "source" not in external_qpu_metadata.columns:
        external_qpu_metadata["source"] = "unknown_external_source"

    if "collection_status" not in external_qpu_metadata.columns:
        external_qpu_metadata["collection_status"] = "unknown"

    if "evidence_level" not in external_qpu_metadata.columns:
        external_qpu_metadata["evidence_level"] = np.where(
            external_qpu_metadata["source"].astype(str).str.contains("live", case=False, na=False),
            "live_runtime_metadata",
            "plausibility_or_static_reference",
        )

    if "backend_type" not in external_qpu_metadata.columns:
        external_qpu_metadata["backend_type"] = "qpu"

    if "backend_family" not in external_qpu_metadata.columns:
        external_qpu_metadata["backend_family"] = np.where(
            external_qpu_metadata.get("qpu_modality", "").astype(str).str.contains("ion", case=False, na=False),
            "qpu_cloud_iontrap",
            "qpu_cloud_superconducting",
        )

    # Canonical feature used in later sections.
    # This assignment prevents Section 3.2 from falling back to the static reference despite live collection.
    external_qpu_metadata_for_generation = external_qpu_metadata.copy()

    if FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR:
        has_live_ibm_anchor = external_qpu_metadata_for_generation["source"].astype(str).str.contains(
            "ibm_qiskit_runtime_live_metadata", case=False, na=False
        ).any()
        has_current_run_anchor = external_qpu_metadata_for_generation.get(
            "anchor_priority", pd.Series(dtype=str)
        ).astype(str).str.contains("live_current_run", case=False, na=False).any()
        if not (has_live_ibm_anchor and has_current_run_anchor):
            raise RuntimeError(
                "FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR=True, but the available external anchor "
                "is not live IBM/Qiskit Runtime metadata collected in this execution. "
                "Cached, snapshot, or static references are not acceptable for the final-paper run."
            )

    external_anchor_evidence_scope = classify_external_anchor_evidence(external_qpu_metadata_for_generation)
    external_qpu_metadata_for_generation.to_csv(
        QPU_ANCHOR_CACHE,
        index=False,
    )
    external_anchor_evidence_scope.to_csv(
        EXTERNAL_CACHE_DIR / "external_anchor_evidence_scope_v9_6.csv",
        index=False,
    )

    print("External metadata rows for anchoring:", len(external_qpu_metadata))
    print("Source-aware anchoring evidence:")
    display(external_anchor_evidence_scope)
    print("Anchor rows:")
    display(
        external_qpu_metadata[
            [c for c in ["backend", "source", "collection_status", "evidence_level", "anchor_priority"] if c in external_qpu_metadata.columns]
        ].head(20)
    )

    if not bool(external_anchor_evidence_scope["live_ibm_runtime_available"].iloc[0]):
        print(
            "IMPORTANT: this run is not live IBM/Qiskit Runtime validated. "
            "Use the recommended_claim/paper_wording above instead of claiming live metadata synchronization."
        )

else:
    print("No external IBM/Qiskit anchor available for synthetic generation.")

    if FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR:
        raise RuntimeError(
            "FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR=True, mas nenhum metadado IBM/Qiskit Runtime "
            "live or live cache was found. For a full-paper run, fix the IBM connection "
            "or provide a valid live cache. For debugging only, set "
            "FULL_PAPER_ALLOW_STATIC_FALLBACK_FOR_DRY_RUN=True."
        )



## 2. Problem formulation

Each quantum workload is combined with multiple candidate backends. For each workload-backend pair, the notebook estimates:

- total runtime, including queueing, transpilation, execution, latency, and post-processing;
- expected fidelity;
- operational failure probability;
- binary execution recommendation;
- multicriteria utility score.

The recommendation is not a fixed rule such as `if fidelity > threshold, then recommend`. It is sampled from a probability derived from a utility function:

\[
U_i = \sum_k w_k s_{ik} + \epsilon_i,
\]

where \(s_{ik}\) denotes normalized criteria such as fidelity, runtime, cost, availability, failure risk, real-hardware validation value, and capacity penalty. Weights \(w_k\) vary by decision profile.



### 2.1 Formal task definition

For each workload \(w\) and candidate backend \(b\), the notebook builds an instance \(x_{w,b}\). The objective is to estimate three output families:

1. **Performance regression:** predict `total_runtime_sec`.
2. **Quality regression:** predict `expected_fidelity`.
3. **Operational classification:** predict `recommended_execution`.

The final orchestration decision is obtained by utility ranking within each `workload_id`, selecting one backend per workload. When no backend is acceptable, the selection is marked as fallback.

### 2.2 Label nature

The variable `recommended_execution` is **not an empirical label collected in production**. It is generated by a probabilistic synthetic policy designed to represent plausible operational criteria. Therefore, the supervised task should be interpreted as:

> learning a synthetic operational decision policy, not empirically discovering a real optimal backend policy.

This distinction is important to avoid circular interpretation in the paper.



## 3. Backend catalog

The catalog distinguishes simulators and cloud QPUs. Each backend has baseline attributes, while the final operational state is later sampled for each workload-backend pair.



### 3.1 Meaning of backend attributes

- `backend_family`: distinguishes simulators and QPUs.
- `base_availability`: expected average availability before dynamic sampling.
- `base_cost_per_shot`: approximate unit cost per shot.
- `speed_factor`: relative backend speed factor.
- `max_effective_qubits`: effective qubit limit used for capacity penalties.
- `supports_real_hardware`: indicates whether the backend represents real quantum hardware.
- `base_single_qubit_error`, `base_two_qubit_error`, `base_readout_error`: initial noise parameters.
- `base_coupling_density`: proxy for qubit connectivity.

These values are not intended to reproduce a specific backend exactly. They define plausible operating ranges that are later perturbed by dynamic state, queueing, calibration age, and stress.



In [ ]:
backend_catalog = pd.DataFrame([
    {
        "backend": "statevector_sim",
        "backend_family": "simulator",
        "qpu_modality": "not_applicable",
        "native_strength": "small_medium_exact",
        "base_availability": 0.96,
        "base_cost_per_shot": 0.00001,
        "speed_factor": 1.15,
        "max_effective_qubits": 34,
        "supports_real_hardware": 0,
        "base_single_qubit_error": 0.00005,
        "base_two_qubit_error": 0.00010,
        "base_readout_error": 0.00010,
        "base_coupling_density": 1.00,
    },
    {
        "backend": "tensor_network_sim",
        "backend_family": "simulator",
        "qpu_modality": "not_applicable",
        "native_strength": "structured_low_entanglement",
        "base_availability": 0.92,
        "base_cost_per_shot": 0.00003,
        "speed_factor": 0.78,
        "max_effective_qubits": 90,
        "supports_real_hardware": 0,
        "base_single_qubit_error": 0.00020,
        "base_two_qubit_error": 0.00060,
        "base_readout_error": 0.00020,
        "base_coupling_density": 0.80,
    },
    {
        "backend": "noisy_sim",
        "backend_family": "simulator",
        "qpu_modality": "not_applicable",
        "native_strength": "noise_experiments",
        "base_availability": 0.88,
        "base_cost_per_shot": 0.00005,
        "speed_factor": 1.00,
        "max_effective_qubits": 56,
        "supports_real_hardware": 0,
        "base_single_qubit_error": 0.00150,
        "base_two_qubit_error": 0.00900,
        "base_readout_error": 0.01500,
        "base_coupling_density": 0.70,
    },
    {
        "backend": "qpu_cloud_superconducting",
        "backend_family": "qpu",
        "qpu_modality": "superconducting",
        "native_strength": "fast_hardware_sampling",
        "base_availability": 0.74,
        "base_cost_per_shot": 0.00085,
        "speed_factor": 1.55,
        "max_effective_qubits": 64,
        "supports_real_hardware": 1,
        "base_single_qubit_error": 0.00120,
        "base_two_qubit_error": 0.01500,
        "base_readout_error": 0.02500,
        "base_coupling_density": 0.32,
    },
    {
        "backend": "qpu_cloud_iontrap",
        "backend_family": "qpu",
        "qpu_modality": "iontrap",
        "native_strength": "high_connectivity_hardware",
        "base_availability": 0.68,
        "base_cost_per_shot": 0.00130,
        "speed_factor": 1.90,
        "max_effective_qubits": 36,
        "supports_real_hardware": 1,
        "base_single_qubit_error": 0.00080,
        "base_two_qubit_error": 0.01000,
        "base_readout_error": 0.01800,
        "base_coupling_density": 0.95,
    },
])

backend_catalog



## 3.2 Live external calibration of the QPU catalog

For the full-paper run, live IBM/Qiskit Runtime metadata collected in the current execution are used before dataset generation. QPU rows in the catalog are calibrated from `external_qpu_metadata_for_generation`, which must report `anchor_priority = live_current_run` and `source = ibm_qiskit_runtime_live_metadata`.

The resulting dataset remains synthetic because workloads, operational states, and decisions are generated in a controlled way. However, superconducting QPU parameters are externally anchored with live IBM/Qiskit Runtime metadata for the current run. If live metadata are unavailable, the notebook stops rather than using a cached or static fallback.


In [ ]:

# ============================================================
# 3.2 Source-aware external calibration of the QPU catalog
# ============================================================

USE_EXTERNAL_ANCHOR_FOR_QPU_CATALOG = True
ANCHOR_BLEND_ALPHA = 0.45  # 0 = keep original catalog; 1 = fully use the external reference.
ANCHOR_ALPHA_SENSITIVITY_VALUES = [0.0, 0.25, 0.45, 0.75, 1.0]

# Full-paper guard: Section 1.1 must have collected live IBM/Qiskit Runtime metadata.
if "external_qpu_metadata_for_generation" not in globals() or external_qpu_metadata_for_generation.empty:
    raise RuntimeError(
        "Live IBM/Qiskit Runtime metadata are required before QPU catalog calibration. "
        "Rerun Section 1.1 with valid IBM credentials."
    )
if not external_qpu_metadata_for_generation.get("anchor_priority", pd.Series(dtype=str)).astype(str).str.contains("live_current_run", case=False, na=False).any():
    raise RuntimeError(
        "The available external metadata are not from the current live IBM/Qiskit Runtime run. "
        "Cached/static fallback references are disabled for the full-paper configuration."
    )



def _static_qpu_anchor_reference_v9():
    return pd.DataFrame([
        {
            "backend": "static_superconducting_reference_low_error",
            "qpu_modality": "superconducting",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "n_qubits": 127,
            "median_1q_gate_error": 3.0e-4,
            "median_2q_gate_error": 8.0e-3,
            "median_readout_error": 1.5e-2,
            "coupling_density": 0.020,
            "median_t1_us": 180.0,
            "median_t2_us": 120.0,
            "pending_jobs": 18.0,
            "operational": 1.0,
        },
        {
            "backend": "static_superconducting_reference_moderate_error",
            "qpu_modality": "superconducting",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "n_qubits": 127,
            "median_1q_gate_error": 8.0e-4,
            "median_2q_gate_error": 2.0e-2,
            "median_readout_error": 3.0e-2,
            "coupling_density": 0.030,
            "median_t1_us": 90.0,
            "median_t2_us": 70.0,
            "pending_jobs": 35.0,
            "operational": 1.0,
        },
        {
            "backend": "static_iontrap_reference_high_connectivity",
            "qpu_modality": "iontrap",
            "source": "static_plausibility_reference_not_current",
            "collection_status": "fallback_static_reference",
            "n_qubits": 32,
            "median_1q_gate_error": 5.0e-4,
            "median_2q_gate_error": 1.2e-2,
            "median_readout_error": 2.0e-2,
            "coupling_density": 0.95,
            "median_t1_us": 1.0e6,
            "median_t2_us": 1.0e5,
            "pending_jobs": 8.0,
            "operational": 1.0,
        },
    ])


# Anchor priority for catalog calibration:
# 1) canonical variable generated in Section 1.1;
# 2) external variable from Section 1.1, if the canonical one does not exist;
# 3) live cache saved on disk;
# 4) static fallback only when explicitly allowed.
qpu_generation_anchor_reference = pd.DataFrame()
qpu_generation_anchor_origin = "not_available"

if (
    "external_qpu_metadata_for_generation" in globals()
    and isinstance(external_qpu_metadata_for_generation, pd.DataFrame)
    and not external_qpu_metadata_for_generation.empty
):
    qpu_generation_anchor_reference = external_qpu_metadata_for_generation.copy()
    qpu_generation_anchor_origin = "variable_external_qpu_metadata_for_generation"
elif (
    "external_qpu_metadata" in globals()
    and isinstance(external_qpu_metadata, pd.DataFrame)
    and not external_qpu_metadata.empty
):
    qpu_generation_anchor_reference = external_qpu_metadata.copy()
    qpu_generation_anchor_origin = "variable_external_qpu_metadata"
else:
    candidate_anchor_paths = [
        Path("external_qpu_cache/external_qpu_metadata_for_generation_v9_6.csv"),
        Path("external_qpu_cache/ibm_backend_metadata_live_v9_6.csv"),
        Path("external_qpu_cache/external_qpu_metadata_for_generation_v9_6.csv"),
        Path("external_qpu_cache/ibm_backend_metadata_live_v9_6.csv"),
    ]
    for candidate_path in candidate_anchor_paths:
        try:
            if candidate_path.exists() and candidate_path.stat().st_size > 0:
                tmp = pd.read_csv(candidate_path)
                if not tmp.empty:
                    qpu_generation_anchor_reference = tmp.copy()
                    qpu_generation_anchor_origin = str(candidate_path)
                    break
        except Exception:
            pass

if qpu_generation_anchor_reference.empty:
    if FULL_PAPER_ALLOW_STATIC_FALLBACK_FOR_DRY_RUN:
        qpu_generation_anchor_reference = _static_qpu_anchor_reference_v9()
        qpu_generation_anchor_origin = "static_dry_run_reference"
    else:
        raise RuntimeError(
            "No external IBM/Qiskit anchor was found to calibrate the QPU catalog. "
            "Run Section 1.1 with live IBM Runtime collection or enable static fallback only for debugging."
        )

# Normalize expected column names.
if "backend" in qpu_generation_anchor_reference.columns and "anchor_backend" not in qpu_generation_anchor_reference.columns:
    qpu_generation_anchor_reference["anchor_backend"] = qpu_generation_anchor_reference["backend"].astype(str)
if "qpu_modality" not in qpu_generation_anchor_reference.columns:
    qpu_generation_anchor_reference["qpu_modality"] = "superconducting"
if "source" not in qpu_generation_anchor_reference.columns:
    qpu_generation_anchor_reference["source"] = "unknown_external_anchor"
if "collection_status" not in qpu_generation_anchor_reference.columns:
    qpu_generation_anchor_reference["collection_status"] = "unknown"
if "evidence_level" not in qpu_generation_anchor_reference.columns:
    qpu_generation_anchor_reference["evidence_level"] = np.where(
        qpu_generation_anchor_reference["source"].astype(str).str.contains("ibm_qiskit_runtime_live_metadata", case=False, na=False),
        "live_runtime_metadata",
        "plausibility_or_static_reference",
    )
qpu_generation_anchor_reference["anchor_origin"] = qpu_generation_anchor_origin

if "classify_external_anchor_evidence" in globals():
    qpu_generation_anchor_evidence_scope = classify_external_anchor_evidence(qpu_generation_anchor_reference)
else:
    qpu_generation_anchor_evidence_scope = pd.DataFrame([{
        "anchor_evidence_scope": "unclassified_external_reference",
        "live_ibm_runtime_available": False,
        "recommended_claim": "externally checked, evidence level not classified",
        "paper_wording": "The external anchor evidence level was not classified in this run.",
    }])
qpu_generation_anchor_reference["anchor_evidence_scope"] = qpu_generation_anchor_evidence_scope["anchor_evidence_scope"].iloc[0]
qpu_generation_anchor_reference["live_ibm_runtime_available"] = bool(qpu_generation_anchor_evidence_scope["live_ibm_runtime_available"].iloc[0])

if FULL_PAPER_REQUIRE_IBM_RUNTIME_ANCHOR:
    has_live_ibm_anchor = qpu_generation_anchor_reference["source"].astype(str).str.contains(
        "ibm_qiskit_runtime_live_metadata", case=False, na=False
    ).any()
    if not has_live_ibm_anchor:
        raise RuntimeError(
            "Section 3.2 is not using live IBM/Qiskit Runtime metadata. "
            "Run Section 1.1 and confirm source=ibm_qiskit_runtime_live_metadata."
        )

numeric_anchor_cols = [
    "n_qubits", "median_1q_gate_error", "median_2q_gate_error",
    "median_readout_error", "coupling_density", "median_t1_us", "median_t2_us",
    "pending_jobs", "operational"
]
for col in numeric_anchor_cols:
    if col in qpu_generation_anchor_reference.columns:
        qpu_generation_anchor_reference[col] = pd.to_numeric(qpu_generation_anchor_reference[col], errors="coerce")


def _blend_numeric(original, reference, alpha=ANCHOR_BLEND_ALPHA):
    try:
        if pd.isna(original) or pd.isna(reference):
            return original
        return float((1 - alpha) * float(original) + alpha * float(reference))
    except Exception:
        return original


def _default_qpu_state_values(backend_name):
    backend_name = str(backend_name).lower()
    if "iontrap" in backend_name:
        return {"base_t1_us": 8.0e5, "base_t2_us": 6.5e5, "base_queue_depth": 11.0}
    return {"base_t1_us": 90.0, "base_t2_us": 70.0, "base_queue_depth": 18.0}


def _relative_log_distance(a, b):
    try:
        if pd.isna(a) or pd.isna(b) or a <= 0 or b <= 0:
            return np.nan
        return abs(np.log10(float(a) / float(b)))
    except Exception:
        return np.nan


def calibrate_qpu_backend_catalog(base_catalog, anchor_reference, alpha=ANCHOR_BLEND_ALPHA):
    """Calibrates QPU catalog rows with external metadata before synthetic generation.

    The goal is not to reproduce a specific backend, but to align orders of magnitude
    of qubits, errors, readout, connectivity, T1/T2, and queueing with IBM/Qiskit data ou fontes
    of the reference. IBM metadata anchor only superconducting backends; the ion-trap profile
    remains anchored in a static/snapshot reference when there is no equivalent IBM source.
    """
    calibrated = base_catalog.copy()
    calibrated["generation_anchor_source"] = "synthetic_default"
    calibrated["generation_anchor_alpha"] = 0.0
    calibrated["generation_anchor_n"] = 0

    # New calibratable operational columns used by synthetic generation.
    for idx, row in calibrated.iterrows():
        defaults = _default_qpu_state_values(row.get("backend", ""))
        for col, value in defaults.items():
            if col not in calibrated.columns:
                calibrated[col] = np.nan
            if pd.isna(calibrated.loc[idx, col]):
                calibrated.loc[idx, col] = value if row.get("backend_family") == "qpu" else (0 if col == "base_queue_depth" else 1e9)

    if not USE_EXTERNAL_ANCHOR_FOR_QPU_CATALOG or anchor_reference.empty:
        return calibrated

    for idx, row in calibrated.iterrows():
        if row.get("backend_family") != "qpu":
            continue

        modality = str(row.get("qpu_modality", "")).lower().strip()
        if modality in ("", "nan", "none", "not_applicable"):
            modality = "iontrap" if "iontrap" in str(row.get("backend", "")).lower() else "superconducting"

        ref = anchor_reference[anchor_reference["qpu_modality"].astype(str).str.lower().eq(modality)].copy()

        # If there is no external reference from the same modality, preserve the synthetic profile
        # and explicitly mark the absence of a specific anchor.
        if ref.empty:
            calibrated.loc[idx, "generation_anchor_source"] = f"no_modality_specific_anchor::{modality}"
            calibrated.loc[idx, "generation_anchor_alpha"] = 0.0
            calibrated.loc[idx, "generation_anchor_n"] = 0
            continue

        ref_stats = ref[[c for c in [
            "n_qubits", "median_1q_gate_error", "median_2q_gate_error",
            "median_readout_error", "coupling_density", "median_t1_us", "median_t2_us",
            "pending_jobs", "operational"
        ] if c in ref.columns]].median(numeric_only=True)

        calibrated.loc[idx, "max_effective_qubits"] = max(
            4,
            round(_blend_numeric(row["max_effective_qubits"], ref_stats.get("n_qubits", np.nan), alpha * 0.35)),
        )
        calibrated.loc[idx, "base_single_qubit_error"] = _blend_numeric(
            row["base_single_qubit_error"], ref_stats.get("median_1q_gate_error", np.nan), alpha
        )
        calibrated.loc[idx, "base_two_qubit_error"] = _blend_numeric(
            row["base_two_qubit_error"], ref_stats.get("median_2q_gate_error", np.nan), alpha
        )
        calibrated.loc[idx, "base_readout_error"] = _blend_numeric(
            row["base_readout_error"], ref_stats.get("median_readout_error", np.nan), alpha
        )
        calibrated.loc[idx, "base_coupling_density"] = _blend_numeric(
            row["base_coupling_density"], ref_stats.get("coupling_density", np.nan), alpha
        )
        calibrated.loc[idx, "base_t1_us"] = _blend_numeric(
            calibrated.loc[idx, "base_t1_us"], ref_stats.get("median_t1_us", np.nan), alpha
        )
        calibrated.loc[idx, "base_t2_us"] = _blend_numeric(
            calibrated.loc[idx, "base_t2_us"], ref_stats.get("median_t2_us", np.nan), alpha
        )
        calibrated.loc[idx, "base_queue_depth"] = max(
            0.0,
            _blend_numeric(calibrated.loc[idx, "base_queue_depth"], ref_stats.get("pending_jobs", np.nan), alpha),
        )
        if not pd.isna(ref_stats.get("operational", np.nan)):
            # Conservative adjustment: synthetic availability does not automatically become 1.0; it only moves closer.
            calibrated.loc[idx, "base_availability"] = np.clip(
                _blend_numeric(row["base_availability"], 0.62 + 0.28 * ref_stats.get("operational", 1.0), alpha * 0.20),
                0.20,
                0.98,
            )
        calibrated.loc[idx, "generation_anchor_source"] = ";".join(sorted(ref["source"].dropna().astype(str).unique()))
        calibrated.loc[idx, "generation_anchor_alpha"] = alpha
        calibrated.loc[idx, "generation_anchor_n"] = len(ref)

    return calibrated


def summarize_anchor_alpha_sensitivity(base_catalog, anchor_reference, alphas):
    rows = []
    for alpha in alphas:
        cat = calibrate_qpu_backend_catalog(base_catalog, anchor_reference, alpha=alpha)
        qpus = cat[cat["backend_family"].eq("qpu")].copy()
        for _, row in qpus.iterrows():
            modality = str(row.get("qpu_modality", "")).lower().strip()
            if modality in ("", "nan", "none", "not_applicable"):
                modality = "iontrap" if "iontrap" in str(row.get("backend", "")).lower() else "superconducting"
            ref = anchor_reference[anchor_reference["qpu_modality"].astype(str).str.lower().eq(modality)].copy()
            if ref.empty:
                rows.append({
                    "alpha": alpha,
                    "backend": row["backend"],
                    "modality": modality,
                    "mean_log10_distance_to_anchor": np.nan,
                    "base_single_qubit_error": row.get("base_single_qubit_error"),
                    "base_two_qubit_error": row.get("base_two_qubit_error"),
                    "base_readout_error": row.get("base_readout_error"),
                    "base_t1_us": row.get("base_t1_us"),
                    "base_t2_us": row.get("base_t2_us"),
                    "base_queue_depth": row.get("base_queue_depth"),
                    "base_coupling_density": row.get("base_coupling_density"),
                    "note": "no external anchor from the same modality"
                })
                continue
            ref_stats = ref[[c for c in [
                "median_1q_gate_error", "median_2q_gate_error", "median_readout_error",
                "coupling_density", "median_t1_us", "median_t2_us", "pending_jobs"
            ] if c in ref.columns]].median(numeric_only=True)
            distances = [
                _relative_log_distance(row.get("base_single_qubit_error"), ref_stats.get("median_1q_gate_error", np.nan)),
                _relative_log_distance(row.get("base_two_qubit_error"), ref_stats.get("median_2q_gate_error", np.nan)),
                _relative_log_distance(row.get("base_readout_error"), ref_stats.get("median_readout_error", np.nan)),
                _relative_log_distance(row.get("base_t1_us"), ref_stats.get("median_t1_us", np.nan)),
                _relative_log_distance(row.get("base_t2_us"), ref_stats.get("median_t2_us", np.nan)),
            ]
            rows.append({
                "alpha": alpha,
                "backend": row["backend"],
                "modality": modality,
                "mean_log10_distance_to_anchor": float(np.nanmean(distances)) if np.any(~pd.isna(distances)) else np.nan,
                "base_single_qubit_error": row.get("base_single_qubit_error"),
                "base_two_qubit_error": row.get("base_two_qubit_error"),
                "base_readout_error": row.get("base_readout_error"),
                "base_t1_us": row.get("base_t1_us"),
                "base_t2_us": row.get("base_t2_us"),
                "base_queue_depth": row.get("base_queue_depth"),
                "base_coupling_density": row.get("base_coupling_density"),
            })
    return pd.DataFrame(rows)


backend_catalog_raw = backend_catalog.copy()
anchor_alpha_sensitivity = summarize_anchor_alpha_sensitivity(
    backend_catalog_raw,
    qpu_generation_anchor_reference,
    ANCHOR_ALPHA_SENSITIVITY_VALUES,
)
backend_catalog = calibrate_qpu_backend_catalog(backend_catalog_raw, qpu_generation_anchor_reference)

qpu_generation_anchor_reference.to_csv(TAB_DIR / "qpu_generation_anchor_reference_v9_6.csv", index=False)
qpu_generation_anchor_evidence_scope.to_csv(TAB_DIR / "external_anchor_evidence_scope_v9_6.csv", index=False)
backend_catalog_raw.to_csv(TAB_DIR / "backend_catalog_raw_v9_6.csv", index=False)
backend_catalog.to_csv(TAB_DIR / "backend_catalog_calibrated_v9_6.csv", index=False)
anchor_alpha_sensitivity.to_csv(TAB_DIR / "anchor_alpha_sensitivity_v9_6.csv", index=False)

print("External reference used for generation calibration:")
print("Source-aware evidence scope:")
display(qpu_generation_anchor_evidence_scope)
display(qpu_generation_anchor_reference[[c for c in ["anchor_backend", "qpu_modality", "source", "collection_status", "evidence_level", "anchor_evidence_scope", "live_ibm_runtime_available", "n_qubits", "median_1q_gate_error", "median_2q_gate_error", "median_readout_error", "median_t1_us", "median_t2_us", "coupling_density", "pending_jobs"] if c in qpu_generation_anchor_reference.columns]].head(30))

print("Backend catalog after source-aware externally anchored calibration:")
display(backend_catalog)

print("Sensitivity analysis of the ANCHOR_BLEND_ALPHA parameter:")
display(anchor_alpha_sensitivity)

print("Methodological note: IBM/Qiskit Runtime metadata anchor only superconducting backends. The ion-trap profile is not calibrated with IBM superconducting data; it remains synthetic/static when no external source of the same modality is available. If live_ibm_runtime_available is False, report this run as a static/cache/snapshot plausibility check according to the evidence-scope table.")




### 3.3 Note on QPU modalities

IBM/Qiskit metadata collected through `QiskitRuntimeService` represent superconducting backends. Therefore, they directly calibrate the `qpu_cloud_superconducting` profile. The `qpu_cloud_iontrap` profile is kept as a synthetic/static reference for an alternative high-connectivity architecture; it should not be interpreted as validated by IBM metadata.



## 4. Algorithm profiles

Profiles control depth, two-qubit gate intensity, entanglement, noise sensitivity, and number of shots.



In [ ]:
algorithm_profiles = pd.DataFrame([
    {
        "algorithm": "GHZ",
        "algorithm_family": "entanglement_kernel",
        "depth_factor": 0.55,
        "two_qubit_factor": 0.95,
        "entanglement_factor": 1.15,
        "noise_sensitivity": 0.85,
        "shot_factor": 0.80,
    },
    {
        "algorithm": "QAOA",
        "algorithm_family": "variational_optimization",
        "depth_factor": 1.15,
        "two_qubit_factor": 1.25,
        "entanglement_factor": 1.05,
        "noise_sensitivity": 1.15,
        "shot_factor": 1.20,
    },
    {
        "algorithm": "VQE",
        "algorithm_family": "variational_simulation",
        "depth_factor": 1.00,
        "two_qubit_factor": 1.05,
        "entanglement_factor": 0.95,
        "noise_sensitivity": 1.05,
        "shot_factor": 1.30,
    },
    {
        "algorithm": "HHL",
        "algorithm_family": "linear_solver",
        "depth_factor": 1.55,
        "two_qubit_factor": 1.20,
        "entanglement_factor": 0.90,
        "noise_sensitivity": 1.35,
        "shot_factor": 0.95,
    },
    {
        "algorithm": "Hamiltonian",
        "algorithm_family": "quantum_simulation",
        "depth_factor": 1.45,
        "two_qubit_factor": 1.35,
        "entanglement_factor": 1.20,
        "noise_sensitivity": 1.25,
        "shot_factor": 1.10,
    },
    {
        "algorithm": "QML",
        "algorithm_family": "quantum_machine_learning",
        "depth_factor": 1.10,
        "two_qubit_factor": 1.10,
        "entanglement_factor": 1.00,
        "noise_sensitivity": 1.10,
        "shot_factor": 1.40,
    },
])

algorithm_profiles



## 5. Operational decision profiles

The weights below simulate different usage objectives. This step reduces the bias of a single recommendation policy.



In [ ]:
decision_profiles = pd.DataFrame([
    {
        "decision_profile": "research_validation",
        "w_fidelity": 0.26,
        "w_time": 0.08,
        "w_cost": 0.03,
        "w_availability": 0.08,
        "w_failure": 0.08,
        "w_hardware": 0.42,
        "w_capacity": 0.05,
    },
    {
        "decision_profile": "cost_sensitive",
        "w_fidelity": 0.22,
        "w_time": 0.14,
        "w_cost": 0.34,
        "w_availability": 0.18,
        "w_failure": 0.08,
        "w_hardware": 0.00,
        "w_capacity": 0.04,
    },
    {
        "decision_profile": "deadline_sensitive",
        "w_fidelity": 0.20,
        "w_time": 0.34,
        "w_cost": 0.10,
        "w_availability": 0.18,
        "w_failure": 0.10,
        "w_hardware": 0.03,
        "w_capacity": 0.05,
    },
    {
        "decision_profile": "fidelity_sensitive",
        "w_fidelity": 0.42,
        "w_time": 0.10,
        "w_cost": 0.06,
        "w_availability": 0.12,
        "w_failure": 0.15,
        "w_hardware": 0.10,
        "w_capacity": 0.05,
    },
    {
        "decision_profile": "hpc_batch",
        "w_fidelity": 0.20,
        "w_time": 0.22,
        "w_cost": 0.18,
        "w_availability": 0.16,
        "w_failure": 0.08,
        "w_hardware": 0.02,
        "w_capacity": 0.14,
    },
    {
        "decision_profile": "educational",
        "w_fidelity": 0.20,
        "w_time": 0.16,
        "w_cost": 0.30,
        "w_availability": 0.24,
        "w_failure": 0.08,
        "w_hardware": 0.00,
        "w_capacity": 0.02,
    },
])

decision_profiles



## 6. Auxiliary functions for realistic synthetic generation



In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def minmax(series):
    series = pd.Series(series)
    min_v = series.min()
    max_v = series.max()
    if max_v == min_v:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_v) / (max_v - min_v)


def within_workload_minmax(df, column):
    """Normalizes a variable among candidate backends of the same workload."""
    if "workload_id" not in df.columns:
        return minmax(df[column])
    def _scale(s):
        min_v, max_v = s.min(), s.max()
        if max_v == min_v:
            return pd.Series(np.zeros(len(s)), index=s.index)
        return (s - min_v) / (max_v - min_v)
    return df.groupby("workload_id")[column].transform(_scale)


def assign_selected_backend(df, utility_col="decision_utility", acceptable_col="recommended_execution"):
    """Selects exactly one backend per workload.

    Operational rule:
    1. If there is at least one acceptable backend, select the acceptable one with the highest utility.
    2. If no backend is acceptable, select the highest-utility backend as fallback.
    """
    df = df.copy()
    df["has_acceptable_backend"] = df.groupby("workload_id")[acceptable_col].transform("sum").gt(0).astype(int)
    df["selection_fallback"] = 0
    df["selected_backend"] = 0

    selected_indices = []
    for _, group in df.groupby("workload_id", sort=False):
        acceptable = group[group[acceptable_col] == 1]
        if len(acceptable) > 0:
            idx = acceptable[utility_col].idxmax()
        else:
            idx = group[utility_col].idxmax()
            df.loc[idx, "selection_fallback"] = 1
        selected_indices.append(idx)

    df.loc[selected_indices, "selected_backend"] = 1
    df["backend_rank_utility"] = df.groupby("workload_id")[utility_col].rank(ascending=False, method="first").astype(int)

    # Operational ranking: acceptable backends are ordered by utility; non-acceptable backends come afterward.
    # When there is no acceptable backend, ranking falls back to utility only.
    rank_values = pd.Series(index=df.index, dtype=int)
    for _, group in df.groupby("workload_id", sort=False):
        if group[acceptable_col].sum() > 0:
            tmp = group.assign(_sort_accept=group[acceptable_col], _sort_utility=group[utility_col])
            ordered_idx = tmp.sort_values(["_sort_accept", "_sort_utility"], ascending=[False, False]).index
        else:
            ordered_idx = group.sort_values(utility_col, ascending=False).index
        rank_values.loc[ordered_idx] = np.arange(1, len(ordered_idx) + 1)

    df["backend_rank_true"] = rank_values.astype(int)
    return df


def add_predicted_selection(test_like_df, prob_col="recommendation_probability", score_col="predicted_decision_score", threshold=0.5):
    """Applies the same selection logic to predicted outputs."""
    out = test_like_df.copy()
    out["predicted_acceptable_backend"] = (out[prob_col] >= threshold).astype(int)
    out["predicted_selection_fallback"] = 0
    out["predicted_selected_backend"] = 0

    selected_indices = []
    rank_values = pd.Series(index=out.index, dtype=int)

    for _, group in out.groupby("workload_id", sort=False):
        acceptable = group[group["predicted_acceptable_backend"] == 1]
        if len(acceptable) > 0:
            idx = acceptable[score_col].idxmax()
        else:
            idx = group[score_col].idxmax()
            out.loc[idx, "predicted_selection_fallback"] = 1
        selected_indices.append(idx)

        if group["predicted_acceptable_backend"].sum() > 0:
            ordered_idx = group.sort_values(["predicted_acceptable_backend", score_col], ascending=[False, False]).index
        else:
            ordered_idx = group.sort_values(score_col, ascending=False).index
        rank_values.loc[ordered_idx] = np.arange(1, len(ordered_idx) + 1)

    out.loc[selected_indices, "predicted_selected_backend"] = 1
    out["predicted_backend_rank"] = rank_values.astype(int)
    return out


def generate_workloads(n_workloads=1000, random_seed=42):
    local_rng = np.random.default_rng(random_seed)
    algorithm_types = algorithm_profiles["algorithm"].tolist()
    profile_types = decision_profiles["decision_profile"].tolist()

    workloads = pd.DataFrame({
        "workload_id": np.arange(n_workloads),
        "algorithm": local_rng.choice(algorithm_types, n_workloads),
        "decision_profile": local_rng.choice(
            profile_types,
            n_workloads,
            p=[0.16, 0.20, 0.18, 0.18, 0.14, 0.14],
        ),
        "base_num_qubits": local_rng.integers(4, 81, n_workloads),
        "base_depth": local_rng.integers(20, 1900, n_workloads),
        "base_two_qubit_gates": local_rng.integers(5, 950, n_workloads),
        "base_shots": local_rng.choice(
            [512, 1024, 2048, 4096, 8192],
            n_workloads,
            p=[0.22, 0.28, 0.25, 0.18, 0.07],
        ),
        "scientific_priority": local_rng.beta(2.2, 2.0, n_workloads),
        "deadline_hours": local_rng.choice([1, 4, 12, 24, 72], n_workloads, p=[0.12, 0.18, 0.25, 0.25, 0.20]),
        "budget_usd": local_rng.lognormal(mean=np.log(5.0), sigma=0.75, size=n_workloads),
    })

    workloads = workloads.merge(algorithm_profiles, on="algorithm", how="left")
    workloads = workloads.merge(decision_profiles, on="decision_profile", how="left")

    workloads["requires_real_hardware"] = local_rng.binomial(
        1,
        sigmoid(2.2 * (workloads["scientific_priority"] - 0.62)),
    )

    mask_research = workloads["decision_profile"] == "research_validation"
    workloads.loc[mask_research, "requires_real_hardware"] = local_rng.binomial(1, 0.78, mask_research.sum())

    workloads["validation_need"] = (
        0.55 * workloads["requires_real_hardware"]
        + 0.45 * workloads["scientific_priority"]
        + local_rng.normal(0, 0.08, n_workloads)
    ).clip(0, 1)

    workloads["num_qubits"] = workloads["base_num_qubits"]
    workloads["circuit_depth"] = (
        workloads["base_depth"] * workloads["depth_factor"]
        + local_rng.normal(0, 60, n_workloads)
    ).round().astype(int).clip(lower=5)

    workloads["two_qubit_gates"] = (
        workloads["base_two_qubit_gates"] * workloads["two_qubit_factor"]
        + local_rng.normal(0, 35, n_workloads)
    ).round().astype(int).clip(lower=1)

    workloads["shots"] = (
        workloads["base_shots"] * workloads["shot_factor"]
    ).round().astype(int).clip(lower=256)

    workloads["entanglement_score"] = (
        0.42 * workloads["two_qubit_gates"] / workloads["two_qubit_gates"].max()
        + 0.28 * workloads["circuit_depth"] / workloads["circuit_depth"].max()
        + 0.30 * workloads["num_qubits"] / workloads["num_qubits"].max()
    ) * workloads["entanglement_factor"]
    workloads["entanglement_score"] = workloads["entanglement_score"].clip(0, 1)

    return workloads.drop(columns=["base_num_qubits", "base_depth", "base_two_qubit_gates", "base_shots"])



In [ ]:
def add_backend_state(df, random_seed=42, stress=False):
    local_rng = np.random.default_rng(random_seed)
    df = df.copy()
    m = len(df)

    state_cols = [
        "calibration_age_hours", "queue_depth", "backend_uptime", "t1_us", "t2_us",
        "single_qubit_error", "two_qubit_error", "readout_error", "coupling_density",
        "transpilation_overhead", "communication_latency_sec",
    ]
    for col in state_cols:
        df[col] = np.nan

    backend_lookup = backend_catalog.set_index("backend")

    for backend_name in backend_catalog["backend"]:
        idx = df["backend"] == backend_name
        k = int(idx.sum())
        row = backend_lookup.loc[backend_name]

        if row["backend_family"] == "qpu":
            modality = str(row.get("qpu_modality", "")).lower()
            default_queue = 11 if modality == "iontrap" or "iontrap" in backend_name else 18
            qpu_queue_mean = row.get("base_queue_depth", default_queue)
            if pd.isna(qpu_queue_mean):
                qpu_queue_mean = default_queue
            qpu_queue_mean = float(qpu_queue_mean) * (1.25 if stress else 1.0)
            qpu_sigma = 0.55 if stress else 0.45

            df.loc[idx, "calibration_age_hours"] = local_rng.gamma(2.0, 4.2 if stress else 3.0, k)
            df.loc[idx, "queue_depth"] = local_rng.poisson(max(0.0, qpu_queue_mean), k)
            df.loc[idx, "backend_uptime"] = np.clip(row["base_availability"] + local_rng.normal(-0.04 if stress else 0, 0.10 if stress else 0.08, k), 0.20, 1.0)

            default_t1 = 8.0e5 if modality == "iontrap" or "iontrap" in backend_name else 90
            default_t2 = 6.5e5 if modality == "iontrap" or "iontrap" in backend_name else 70
            t1_center = row.get("base_t1_us", default_t1)
            t2_center = row.get("base_t2_us", default_t2)
            if pd.isna(t1_center):
                t1_center = default_t1
            if pd.isna(t2_center):
                t2_center = default_t2
            t1_sigma = 0.30 if stress else 0.22
            t2_sigma = 0.32 if stress else 0.24
            df.loc[idx, "t1_us"] = np.clip(local_rng.lognormal(np.log(max(float(t1_center), 1.0)), t1_sigma, k), 10, None)
            df.loc[idx, "t2_us"] = np.clip(local_rng.lognormal(np.log(max(float(t2_center), 1.0)), t2_sigma, k), 10, None)

            df.loc[idx, "single_qubit_error"] = np.clip(row["base_single_qubit_error"] * local_rng.lognormal(0, qpu_sigma, k), 0.00005, 0.010)
            df.loc[idx, "two_qubit_error"] = np.clip(row["base_two_qubit_error"] * local_rng.lognormal(0, qpu_sigma + 0.10, k), 0.001, 0.090)
            df.loc[idx, "readout_error"] = np.clip(row["base_readout_error"] * local_rng.lognormal(0, qpu_sigma, k), 0.002, 0.150)
            df.loc[idx, "coupling_density"] = np.clip(row["base_coupling_density"] + local_rng.normal(0, 0.09 if stress else 0.07, k), 0.01, 1.0)
            df.loc[idx, "transpilation_overhead"] = np.clip(local_rng.lognormal(np.log(1.35 if stress else 1.28), 0.35 if stress else 0.28, k), 0.80, 3.60)
            df.loc[idx, "communication_latency_sec"] = local_rng.lognormal(2.2 if stress else 2.0, 0.65 if stress else 0.50, k)
        else:
            df.loc[idx, "calibration_age_hours"] = 0
            df.loc[idx, "queue_depth"] = local_rng.poisson(8 if (stress and backend_name == "tensor_network_sim") else (6 if backend_name == "tensor_network_sim" else 3), k)
            df.loc[idx, "backend_uptime"] = np.clip(row["base_availability"] + local_rng.normal(-0.02 if stress else 0, 0.05 if stress else 0.03, k), 0.55, 1.0)
            df.loc[idx, "t1_us"] = 1e9
            df.loc[idx, "t2_us"] = 1e9
            df.loc[idx, "single_qubit_error"] = np.clip(row["base_single_qubit_error"] * local_rng.lognormal(0, 0.35 if stress else 0.25, k), 0.000001, 0.020)
            df.loc[idx, "two_qubit_error"] = np.clip(row["base_two_qubit_error"] * local_rng.lognormal(0, 0.42 if stress else 0.30, k), 0.000001, 0.040)
            df.loc[idx, "readout_error"] = np.clip(row["base_readout_error"] * local_rng.lognormal(0, 0.35 if stress else 0.25, k), 0.000001, 0.050)
            df.loc[idx, "coupling_density"] = np.clip(row["base_coupling_density"] + local_rng.normal(0, 0.06 if stress else 0.04, k), 0.10, 1.0)
            df.loc[idx, "transpilation_overhead"] = np.clip(local_rng.lognormal(np.log(1.08 if stress else 1.03), 0.18 if stress else 0.12, k), 0.80, 1.90)
            df.loc[idx, "communication_latency_sec"] = local_rng.lognormal(0.65 if stress else 0.50, 0.35 if stress else 0.25, k)

    return df



In [ ]:
def simulate_operational_outcomes(df, random_seed=42, stress=False, generate_label=True):
    local_rng = np.random.default_rng(random_seed)
    df = df.copy()
    m = len(df)

    depth_norm = df["circuit_depth"] / df["circuit_depth"].max()
    gates_norm = df["two_qubit_gates"] / df["two_qubit_gates"].max()

    df["qubit_pressure"] = df["num_qubits"] / df["max_effective_qubits"]
    df["capacity_penalty"] = np.where(df["qubit_pressure"] > 1, (df["qubit_pressure"] - 1) ** 2, 0)

    queue_shape = 2.0 if not stress else 2.3
    qpu_scale = 90.0 if not stress else 125.0
    sim_scale = 8.0 if not stress else 14.0

    df["queue_time_sec"] = np.where(
        df["backend_family"] == "qpu",
        local_rng.gamma(queue_shape, qpu_scale, m) * (1 + df["queue_depth"] / 30),
        local_rng.gamma(1.5, sim_scale, m) * (1 + df["queue_depth"] / 12),
    )

    df["cost_per_shot"] = (df["base_cost_per_shot"] * local_rng.lognormal(0, 0.30 if stress else 0.22, m)).clip(0.000001, 0.02)
    df["total_cost"] = df["shots"] * df["cost_per_shot"] + np.where(df["backend_family"] == "simulator", 0.0008 * df["queue_time_sec"], 0)

    df["single_qubit_gates"] = (0.25 * df["circuit_depth"] * np.log1p(df["num_qubits"])).round().astype(int)
    df["circuit_duration_us"] = 0.04 * df["single_qubit_gates"] + 0.25 * df["two_qubit_gates"] * df["transpilation_overhead"]

    base_runtime = (
        0.014 * df["circuit_depth"] * np.log1p(df["num_qubits"])
        + 0.0045 * df["two_qubit_gates"]
        + 0.0007 * df["shots"]
    ) * df["speed_factor"] * df["transpilation_overhead"]

    base_runtime *= np.where((df["backend"] == "tensor_network_sim") & (df["entanglement_score"] < 0.45), 0.72, 1.00)
    base_runtime *= np.where((df["backend"] == "tensor_network_sim") & (df["entanglement_score"] > 0.72), 1.55, 1.00)
    base_runtime *= np.where(df["backend_family"] == "qpu", 1.05 + 0.25 * depth_norm, 1.00)

    df["transpilation_time_sec"] = (0.0015 * df["circuit_depth"] * np.log1p(df["num_qubits"]) * df["transpilation_overhead"]).clip(lower=0.1)
    df["postprocessing_time_sec"] = 0.0004 * df["shots"] * np.log1p(df["num_qubits"]) + 0.02 * df["entanglement_score"] * df["circuit_depth"] / 100
    df["execution_time_sec"] = (base_runtime + local_rng.normal(0, 30 if stress else 20, m)).clip(lower=0.5)

    df["total_runtime_sec"] = (
        df["queue_time_sec"]
        + df["transpilation_time_sec"]
        + df["execution_time_sec"]
        + df["communication_latency_sec"]
        + df["postprocessing_time_sec"]
        + 130 * df["capacity_penalty"]
    ).clip(lower=1)

    error_load = (
        0.18 * df["single_qubit_error"] * np.log1p(df["single_qubit_gates"])
        + 0.42 * df["two_qubit_error"] * np.log1p(df["two_qubit_gates"])
        + 0.25 * df["readout_error"] * np.log1p(df["num_qubits"])
        + 0.10 * df["entanglement_score"] * df["noise_sensitivity"]
        + 0.08 * depth_norm
        + 0.18 * df["capacity_penalty"]
    )

    decoherence_penalty = (df["circuit_duration_us"] / df["t1_us"]).clip(0, 10) * np.where(df["backend_family"] == "qpu", 1, 0)
    tensor_approx_penalty = np.where((df["backend"] == "tensor_network_sim") & (df["entanglement_score"] > 0.72), 0.12 * (df["entanglement_score"] - 0.72), 0)
    ideal_bonus = np.where(df["backend"] == "statevector_sim", 0.08, 0)

    df["expected_fidelity"] = (
        np.exp(-error_load - 0.35 * decoherence_penalty - tensor_approx_penalty)
        + ideal_bonus
        + local_rng.normal(0, 0.035 if stress else 0.025, m)
    ).clip(0, 1)

    failure_score = (
        0.26 * minmax(df["queue_depth"])
        + 0.18 * minmax(df["calibration_age_hours"])
        + 0.28 * minmax(df["two_qubit_error"])
        + 0.12 * depth_norm
        + 0.16 * minmax(df["capacity_penalty"])
    )

    df["failure_probability"] = sigmoid(5 * (failure_score - 0.62)) * np.where(df["backend_family"] == "qpu", 1.20, 0.45)
    df["failure_probability"] = df["failure_probability"].clip(0.005, 0.85)
    df["execution_success"] = local_rng.binomial(1, 1 - df["failure_probability"])

    df["hardware_validation_value"] = (
        df["supports_real_hardware"]
        * (0.55 * df["requires_real_hardware"] + 0.30 * df["scientific_priority"] + 0.15 * df["validation_need"])
    ).clip(0, 1)

    # Local comparison between candidate backends for the same workload.
    df["time_score"] = 1 - within_workload_minmax(df, "total_runtime_sec")
    df["cost_score"] = 1 - within_workload_minmax(df, "total_cost")
    df["failure_score_good"] = 1 - df["failure_probability"]
    df["capacity_score"] = 1 - within_workload_minmax(df, "capacity_penalty")

    # Correction : utility must represent scenarios where QPUs are
    # operationally justified, especially when the workload requires validation
    # em hardware real. Sem esse termo, o gerador tendia a transformar QPUs em classe
    # almost always negative, inducing artificially low recall for the QPU family.
    df["hardware_requirement_alignment"] = (
        df["supports_real_hardware"] * df["requires_real_hardware"] * (0.06 + 0.05 * df["validation_need"])
        - (1 - df["supports_real_hardware"]) * df["requires_real_hardware"] * 0.035
    )

    df["decision_utility"] = (
        df["w_fidelity"] * df["expected_fidelity"]
        + df["w_time"] * df["time_score"]
        + df["w_cost"] * df["cost_score"]
        + df["w_availability"] * df["backend_uptime"]
        + df["w_failure"] * df["failure_score_good"]
        + df["w_hardware"] * df["hardware_validation_value"]
        + df["w_capacity"] * df["capacity_score"]
        + df["hardware_requirement_alignment"]
        + local_rng.normal(0, 0.045 if stress else 0.035, m)
    )

    if generate_label:
        #  threshold by workload, not a global threshold.
        # The recommendation now represents relative acceptability among candidates
        # for the same workload, reducing structural bias against QPUs in scenarios of
        # experimental validation.
        threshold = df.groupby("workload_id")["decision_utility"].transform(lambda s: s.quantile(0.50))
        df["recommendation_probability_true"] = (
            sigmoid(11 * (df["decision_utility"] - threshold))
            * (1 - 0.45 * df["failure_probability"])
        ).clip(0.01, 0.99)
        df["recommended_execution"] = local_rng.binomial(1, df["recommendation_probability_true"])

    return df



## 7. Enriched synthetic dataset generation



In [ ]:
n_workloads = N_WORKLOADS

workloads = generate_workloads(n_workloads=n_workloads, random_seed=SEED)
df = workloads.merge(backend_catalog, how="cross")
df = add_backend_state(df, random_seed=SEED)
df = simulate_operational_outcomes(df, random_seed=SEED, generate_label=True)

# Two outputs are retained:
# 1) recommended_execution: acceptable backend.
# 2) selected_backend: backend escolhido como melhor candidato do workload.
#
# Final selection is made only among acceptable backends.
# When no backend is acceptable, the orchestrator selects the highest utility value as fallback.
df = assign_selected_backend(df, utility_col="decision_utility", acceptable_col="recommended_execution")

print("Workloads:", workloads.shape)
print("Workload-backend pairs:", df.shape)
print("Recommended backends per workload:")
display(df.groupby("workload_id")["recommended_execution"].sum().value_counts().sort_index().rename("n_workloads"))

print("Selected backends per workload:")
display(df.groupby("workload_id")["selected_backend"].sum().value_counts().sort_index().rename("n_workloads"))

print("Workloads without an acceptable backend:", int(df.groupby("workload_id")["recommended_execution"].sum().eq(0).sum()))
print("Fallback selections:", int(df["selection_fallback"].sum()))
print("Final selections that are also acceptable:", round(df.loc[df["selected_backend"] == 1, "recommended_execution"].mean(), 4))

selected_backend_distribution = (
    df[df["selected_backend"] == 1]["backend"]
    .value_counts(normalize=True)
    .rename("selected_backend_rate")
    .reset_index()
)
selected_backend_distribution.columns = ["backend", "selected_backend_rate"]
selected_backend_distribution.to_csv(TAB_DIR / "selected_backend_distribution.csv", index=False)
display(selected_backend_distribution)

df.to_csv(DATA_DIR / "synthetic_quantum_orchestration_v3.csv", index=False)
df.head()



### 7.1 Data dictionary and dataset traceability

The cell below creates an automatic dictionary of generated columns. It supports paper review, debugging, and open science documentation. Columns are classified into methodological categories: identifiers, workload attributes, backend attributes, operational state, supervised targets, and auxiliary variables.



In [ ]:

# ============================================================
# 7.1 Data dictionary of the synthetic dataset
# ============================================================

def infer_column_role(col):
    if col in ["workload_id", "backend", "algorithm", "decision_profile", "backend_family", "algorithm_family"]:
        return "identifier_or_category"
    if col in ["total_runtime_sec", "expected_fidelity"]:
        return "regression_target"
    if col in ["recommended_execution", "selected_backend", "selection_fallback"]:
        return "decision_target_or_selection"
    if any(token in col for token in ["error", "failure", "fidelity", "t1", "t2", "noise"]):
        return "quality_or_noise"
    if any(token in col for token in ["runtime", "time", "latency", "queue", "duration"]):
        return "performance_or_time"
    if any(token in col for token in ["cost", "shot"]):
        return "cost_or_sampling"
    if any(token in col for token in ["qubit", "depth", "gate", "entanglement"]):
        return "circuit_or_workload_complexity"
    if any(token in col for token in ["utility", "score", "penalty", "capacity"]):
        return "decision_utility_component"
    return "other"

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_values": [int(df[c].isna().sum()) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "role": [infer_column_role(c) for c in df.columns],
})

data_dictionary.to_csv(TAB_DIR / "data_dictionary_v9_6.csv", index=False)
display(data_dictionary.sort_values(["role", "column"]).reset_index(drop=True))



### 7.2 Source-aware comparison between external QPU metadata and synthetic data

This step compares the best available external reference with the synthetic QPU attributes generated in the benchmark. The comparison is used as a plausibility validation: it checks whether synthetic distributions remain close to, or compatible with, the recorded evidence source.

The section is intentionally source-aware. If the metadata source is `ibm_qiskit_runtime_live_metadata`, the comparison can be described as a live IBM/Qiskit Runtime metadata comparison. If the source is `static_plausibility_reference_not_current`, `fallback_static_reference`, or `plausibility_or_static_reference`, the comparison must be described only as a static plausibility check.

Because IBM/Qiskit Runtime backends accessible in this setup are superconducting QPUs, the main comparison is performed against synthetic rows from the `qpu_cloud_superconducting` family. Synthetic backends from other modalities, such as ion trap, remain explicitly marked as synthetic/static when no modality-compatible external source is available.



In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

try:
    TAB_DIR
except NameError:
    TAB_DIR = Path("outputs_ai_quantum_v9_6/tables")
    TAB_DIR.mkdir(parents=True, exist_ok=True)


def _rs_numeric(series):
    return pd.to_numeric(series, errors="coerce").dropna()


def _rs_pick_numeric_column(dataframe, candidates):
    if dataframe is None or dataframe.empty:
        return None, pd.Series(dtype=float)
    for col in candidates:
        if col in dataframe.columns:
            values = _rs_numeric(dataframe[col])
            if not values.empty:
                return col, values
    return None, pd.Series(dtype=float)


def _normalize_modality(df_like, backend_col="backend"):
    out = df_like.copy()
    if "qpu_modality" not in out.columns:
        out["qpu_modality"] = np.where(
            out.get(backend_col, pd.Series("", index=out.index)).astype(str).str.contains("ion", case=False, na=False),
            "iontrap",
            "superconducting",
        )
    out["qpu_modality"] = out["qpu_modality"].fillna("unknown").astype(str).str.lower()
    return out


def build_modality_aware_real_vs_synthetic_report(real_metadata, synthetic_data, output_prefix="metadata_vs_synthetic_v9_6"):
    """Compares external metadata and synthetic data separately by QPU modality.

    This report is source-aware. Static fallback rows are treated as
    static plausibility references, not as live IBM/Qiskit Runtime validation. IBM/Qiskit
    Runtime backends are superconducting and must not validate ion-trap profiles.
    """
    if not isinstance(synthetic_data, pd.DataFrame) or synthetic_data.empty:
        return pd.DataFrame(), pd.DataFrame()

    synth_qpu = synthetic_data[synthetic_data.get("backend_family", "").eq("qpu")].copy()
    synth_qpu = _normalize_modality(synth_qpu)

    real = real_metadata.copy() if isinstance(real_metadata, pd.DataFrame) else pd.DataFrame()
    if not real.empty:
        real = _normalize_modality(real)
    if "classify_external_anchor_evidence" in globals():
        evidence_scope_df = classify_external_anchor_evidence(real)
    else:
        evidence_scope_df = pd.DataFrame([{
            "anchor_evidence_scope": "unclassified_external_reference",
            "live_ibm_runtime_available": False,
            "recommended_claim": "externally checked, evidence level not classified",
            "paper_wording": "The external evidence level was not classified.",
        }])
    evidence_scope = evidence_scope_df["anchor_evidence_scope"].iloc[0]
    live_available = bool(evidence_scope_df["live_ibm_runtime_available"].iloc[0])

    metric_map = {
        "n_qubits": ["n_qubits", "num_qubits", "max_effective_qubits"],
        "single_qubit_error": ["median_1q_gate_error", "single_qubit_error", "base_single_qubit_error"],
        "two_qubit_error": ["median_2q_gate_error", "two_qubit_error", "base_two_qubit_error"],
        "readout_error": ["median_readout_error", "readout_error", "base_readout_error"],
        "coupling_density": ["coupling_density", "base_coupling_density"],
        "t1_us": ["median_t1_us", "t1_us", "base_t1_us"],
        "t2_us": ["median_t2_us", "t2_us", "base_t2_us"],
        "queue_depth": ["pending_jobs", "queue_depth", "base_queue_depth"],
    }

    rows = []
    modalities = sorted(set(synth_qpu["qpu_modality"].dropna().astype(str)) | set(real.get("qpu_modality", pd.Series(dtype=str)).dropna().astype(str)))
    for modality in modalities:
        synth_mod = synth_qpu[synth_qpu["qpu_modality"].eq(modality)]
        real_mod = real[real["qpu_modality"].eq(modality)] if not real.empty else pd.DataFrame()

        for metric, candidates in metric_map.items():
            synth_col, synth_values = _rs_pick_numeric_column(synth_mod, candidates)
            real_col, real_values = _rs_pick_numeric_column(real_mod, candidates)

            if synth_values.empty:
                status = "synthetic_metric_not_available"
                compatible = np.nan
                synth_within_real_range_pct = np.nan
                real_min = real_max = synth_mean = np.nan
            elif real_values.empty:
                status = "no_external_reference_for_modality"
                compatible = np.nan
                synth_within_real_range_pct = np.nan
                real_min = real_max = np.nan
                synth_mean = float(synth_values.mean())
            else:
                real_min = float(real_values.min())
                real_max = float(real_values.max())
                # Conservative margin for temporal/calibration differences and synthetic noise.
                if real_min >= 0 and real_max > 0:
                    pad = max((real_max - real_min) * 0.50, real_max * 0.25)
                    lo = max(0.0, real_min - pad)
                    hi = real_max + pad
                else:
                    lo, hi = real_min, real_max
                synth_within_real_range_pct = float(synth_values.between(lo, hi).mean() * 100.0)
                compatible = synth_within_real_range_pct >= 50.0
                status = "compatible" if compatible else "outside_external_range"
                synth_mean = float(synth_values.mean())

            rows.append({
                "qpu_modality": modality,
                "metric": metric,
                "real_column": real_col,
                "synthetic_column": synth_col,
                "real_min": real_min,
                "real_max": real_max,
                "synthetic_mean": synth_mean,
                "synthetic_within_external_range_pct": synth_within_real_range_pct,
                "compatible": compatible,
                "status": status,
                "anchor_evidence_scope": evidence_scope,
                "live_ibm_runtime_available": live_available,
                "real_n": int(len(real_values)) if not real_values.empty else 0,
                "synthetic_n": int(len(synth_values)) if not synth_values.empty else 0,
            })

    comparison = pd.DataFrame(rows)
    if comparison.empty:
        score = pd.DataFrame([{
            "compatible_metrics_pct": np.nan,
            "metrics_with_external_reference": 0,
            "metrics_without_external_reference": 0,
            "mean_synthetic_within_external_range_pct": np.nan,
            "anchor_evidence_scope": evidence_scope,
            "live_ibm_runtime_available": live_available,
            "interpretation": "no_qpu_comparison_available",
            "recommended_claim": evidence_scope_df["recommended_claim"].iloc[0] if "recommended_claim" in evidence_scope_df.columns else "not_available",
        }])
    else:
        with_ref = comparison[comparison["status"].isin(["compatible", "outside_external_range"])]
        score = pd.DataFrame([{
            "compatible_metrics_pct": float(with_ref["compatible"].mean() * 100.0) if not with_ref.empty else np.nan,
            "metrics_with_external_reference": int(len(with_ref)),
            "metrics_without_external_reference": int(comparison["status"].eq("no_external_reference_for_modality").sum()),
            "mean_synthetic_within_external_range_pct": float(with_ref["synthetic_within_external_range_pct"].mean()) if not with_ref.empty else np.nan,
            "anchor_evidence_scope": evidence_scope,
            "live_ibm_runtime_available": live_available,
            "interpretation": "modality_aware_live_runtime_check" if live_available else "modality_aware_static_or_cached_plausibility_check",
            "recommended_claim": evidence_scope_df["recommended_claim"].iloc[0] if "recommended_claim" in evidence_scope_df.columns else "not_available",
        }])

    comparison.to_csv(TAB_DIR / f"{output_prefix}_ranges.csv", index=False)
    score.to_csv(TAB_DIR / f"{output_prefix}_score.csv", index=False)
    return comparison, score

_real_metadata_for_comparison = globals().get("external_qpu_metadata_for_generation", globals().get("external_qpu_metadata", pd.DataFrame()))
metadata_vs_synthetic_ranges, metadata_vs_synthetic_score = build_modality_aware_real_vs_synthetic_report(
    _real_metadata_for_comparison,
    df,
    output_prefix="metadata_vs_synthetic_v9_6",
)

print("External-vs-synthetic metadata comparison by modality:")
display(metadata_vs_synthetic_ranges)
print("Compatibility summary:")
display(metadata_vs_synthetic_score)




## 8. Dataset bias diagnostics

This section inspects recommendation rates by backend, algorithm, and decision profile to identify possible imbalance in the synthetic policy.



In [ ]:
class_balance = df["recommended_execution"].value_counts(normalize=True).rename("proportion").reset_index()
class_balance.columns = ["recommended_execution", "proportion"]
class_balance.to_csv(TAB_DIR / "class_balance.csv", index=False)
class_balance



In [ ]:
bias_backend = df.groupby("backend").agg(
    n=("recommended_execution", "size"),
    recommendation_rate=("recommended_execution", "mean"),
    mean_fidelity=("expected_fidelity", "mean"),
    mean_runtime_sec=("total_runtime_sec", "mean"),
    mean_cost=("total_cost", "mean"),
    mean_failure_probability=("failure_probability", "mean"),
    mean_hardware_value=("hardware_validation_value", "mean"),
).sort_values("recommendation_rate", ascending=False)

bias_backend.to_csv(TAB_DIR / "bias_by_backend.csv")
bias_backend



In [ ]:
bias_algorithm = df.groupby("algorithm").agg(
    n=("recommended_execution", "size"),
    recommendation_rate=("recommended_execution", "mean"),
    mean_fidelity=("expected_fidelity", "mean"),
    mean_runtime_sec=("total_runtime_sec", "mean"),
    mean_entanglement=("entanglement_score", "mean"),
).sort_values("recommendation_rate", ascending=False)

bias_algorithm.to_csv(TAB_DIR / "bias_by_algorithm.csv")
bias_algorithm



In [ ]:
bias_profile = df.groupby("decision_profile").agg(
    n=("recommended_execution", "size"),
    recommendation_rate=("recommended_execution", "mean"),
    mean_hardware_value=("hardware_validation_value", "mean"),
    mean_budget=("budget_usd", "mean"),
    mean_deadline=("deadline_hours", "mean"),
).sort_values("recommendation_rate", ascending=False)

bias_profile.to_csv(TAB_DIR / "bias_by_decision_profile.csv")
bias_profile



In [ ]:
plt.figure(figsize=(8, 5))
bias_backend["recommendation_rate"].sort_values().plot(kind="barh")
plt.xlabel("Recommendation rate")
plt.ylabel("Backend")
plt.title("Recommendation rate by backend")
plt.tight_layout()
plt.savefig(FIG_DIR / "recommendation_rate_by_backend.png", dpi=300)
plt.show()
plt.close()



In [ ]:
plt.figure(figsize=(8, 5))
bias_algorithm["recommendation_rate"].sort_values().plot(kind="barh")
plt.xlabel("Recommendation rate")
plt.ylabel("Algorithm")
plt.title("Recommendation rate by algorithm")
plt.tight_layout()
plt.savefig(FIG_DIR / "recommendation_rate_by_algorithm.png", dpi=300)
plt.show()
plt.close()



In [ ]:
plt.figure(figsize=(8, 5))
bias_profile["recommendation_rate"].sort_values().plot(kind="barh")
plt.xlabel("Recommendation rate")
plt.ylabel("Decision profile")
plt.title("Recommendation rate by decision profile")
plt.tight_layout()
plt.savefig(FIG_DIR / "recommendation_rate_by_decision_profile.png", dpi=300)
plt.show()
plt.close()



## 8.1 Final backend selection diagnostics



In [ ]:
selected_backend_by_backend = df.groupby("backend").agg(
    selection_rate=("selected_backend", "mean"),
    mean_rank=("backend_rank_true", "mean"),
    utility_rank_mean=("backend_rank_utility", "mean"),
    recommendation_rate=("recommended_execution", "mean"),
    avg_decision_utility=("decision_utility", "mean"),
    fallback_selection_rate=("selection_fallback", "mean"),
).sort_values("selection_rate", ascending=False)

selected_backend_by_profile = (
    df[df["selected_backend"] == 1]
    .groupby("decision_profile")["backend"]
    .value_counts(normalize=True)
    .rename("selection_share")
    .reset_index()
)

selected_backend_diagnostics = pd.DataFrame({
    "metric": [
        "workloads_total",
        "workloads_with_at_least_one_acceptable_backend",
        "workloads_without_acceptable_backend",
        "selected_backend_is_acceptable_rate",
        "fallback_selection_count",
    ],
    "value": [
        df["workload_id"].nunique(),
        int(df.groupby("workload_id")["recommended_execution"].sum().gt(0).sum()),
        int(df.groupby("workload_id")["recommended_execution"].sum().eq(0).sum()),
        df.loc[df["selected_backend"] == 1, "recommended_execution"].mean(),
        int(df["selection_fallback"].sum()),
    ],
})

selected_backend_by_backend.to_csv(TAB_DIR / "selected_backend_by_backend.csv")
selected_backend_by_profile.to_csv(TAB_DIR / "selected_backend_by_profile.csv", index=False)
selected_backend_diagnostics.to_csv(TAB_DIR / "selected_backend_diagnostics.csv", index=False)

display(selected_backend_diagnostics)
selected_backend_by_backend



In [ ]:
plt.figure(figsize=(8, 5))
selected_backend_by_backend["selection_rate"].sort_values().plot(kind="barh")
plt.xlabel("Final selection rate")
plt.ylabel("Backend")
plt.title("Distribution of selected backend by workload")
plt.tight_layout()
plt.savefig(FIG_DIR / "selected_backend_rate_by_backend.png", dpi=300)
plt.show()
plt.close()


## 9. Exploratory analysis of operational targets



In [ ]:
summary_backend = df.groupby("backend").agg(
    avg_total_runtime_sec=("total_runtime_sec", "mean"),
    median_total_runtime_sec=("total_runtime_sec", "median"),
    avg_expected_fidelity=("expected_fidelity", "mean"),
    avg_failure_probability=("failure_probability", "mean"),
    avg_total_cost=("total_cost", "mean"),
    avg_decision_utility=("decision_utility", "mean"),
    recommendation_rate=("recommended_execution", "mean"),
).sort_values("avg_decision_utility", ascending=False)

summary_algorithm = df.groupby("algorithm").agg(
    avg_total_runtime_sec=("total_runtime_sec", "mean"),
    avg_expected_fidelity=("expected_fidelity", "mean"),
    avg_failure_probability=("failure_probability", "mean"),
    avg_decision_utility=("decision_utility", "mean"),
    recommendation_rate=("recommended_execution", "mean"),
).sort_values("avg_decision_utility", ascending=False)

summary_backend.to_csv(TAB_DIR / "summary_backend.csv")
summary_algorithm.to_csv(TAB_DIR / "summary_algorithm.csv")

summary_backend



In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["total_runtime_sec"], df["expected_fidelity"], alpha=0.25)
plt.xscale("log")
plt.xlabel("Total runtime (s, log scale)")
plt.ylabel("Expected fidelity")
plt.title("Relationship between total runtime and expected fidelity")
plt.tight_layout()
plt.savefig(FIG_DIR / "runtime_vs_fidelity.png", dpi=300)
plt.show()
plt.close()



In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["failure_probability"], df["decision_utility"], alpha=0.25)
plt.xlabel("Failure probability")
plt.ylabel("Decision utility")
plt.title("Relationship between operational risk and utility")
plt.tight_layout()
plt.savefig(FIG_DIR / "failure_vs_utility.png", dpi=300)
plt.show()
plt.close()



In [ ]:
plt.figure(figsize=(8, 5))
for backend_name, group in df.groupby("backend"):
    plt.scatter(group["num_qubits"], group["total_runtime_sec"], alpha=0.16, label=backend_name)
plt.yscale("log")
plt.xlabel("Number of qubits")
plt.ylabel("Total runtime (s, log scale)")
plt.title("Synthetic scalability by backend")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "runtime_by_qubits_backend.png", dpi=300)
plt.show()
plt.close()



## 10. Modeling preparation with workload-level split



### 10.1 Experimental leakage control

The train/test split is performed by `workload_id`. This prevents the same workload from appearing in both training and testing through different candidate backends.



In [ ]:
base_features = [
    "algorithm", "algorithm_family", "decision_profile",
    "backend", "backend_family", "native_strength",
    "num_qubits", "circuit_depth", "two_qubit_gates", "shots",
    "entanglement_score", "noise_sensitivity",
    "scientific_priority", "requires_real_hardware", "validation_need",
    "deadline_hours", "budget_usd",
    "calibration_age_hours", "queue_depth", "backend_uptime",
    "single_qubit_error", "two_qubit_error", "readout_error",
    "coupling_density", "transpilation_overhead", "communication_latency_sec",
    "qubit_pressure", "capacity_penalty", "queue_time_sec", "total_cost",
    "failure_probability", "hardware_validation_value",
]

classifier_features = base_features + ["expected_fidelity", "total_runtime_sec"]

categorical_features = [c for c in base_features if df[c].dtype == "object"]
numeric_features = [c for c in base_features if c not in categorical_features]

categorical_features_clf = [c for c in classifier_features if df[c].dtype == "object"]
numeric_features_clf = [c for c in classifier_features if c not in categorical_features_clf]

preprocess_reg = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numeric_features),
])

preprocess_clf = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_clf),
    ("num", StandardScaler(), numeric_features_clf),
])

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(df, df["recommended_execution"], groups=df["workload_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", train_df.shape[0])
print("Test rows:", test_df.shape[0])
print("Workloads in training:", train_df["workload_id"].nunique())
print("Workloads in test:", test_df["workload_id"].nunique())
print("Workload intersection:", len(set(train_df["workload_id"]) & set(test_df["workload_id"])))



## 11. Regression for total runtime



### 11.0 Interpretation of regression metrics

MAE and RMSE quantify prediction error in seconds, while R² measures explained variance.



In [ ]:
X_train_reg = train_df[base_features]
X_test_reg = test_df[base_features]
y_train_runtime = train_df["total_runtime_sec"]
y_test_runtime = test_df["total_runtime_sec"]

runtime_model = Pipeline([
    ("prep", preprocess_reg),
    ("model", RandomForestRegressor(
        n_estimators=50,
        random_state=SEED,
        min_samples_leaf=3,
        n_jobs=1,
    )),
])

runtime_model.fit(X_train_reg, y_train_runtime)
y_pred_runtime = runtime_model.predict(X_test_reg)

runtime_metrics = {
    "model": "RandomForestRegressor",
    "target": "total_runtime_sec",
    "evaluation": "holdout_group_split",
    "MAE": mean_absolute_error(y_test_runtime, y_pred_runtime),
    "RMSE": mean_squared_error(y_test_runtime, y_pred_runtime) ** 0.5,
    "R2": r2_score(y_test_runtime, y_pred_runtime),
}
runtime_metrics



In [ ]:
runtime_baselines = []
for strategy in ["mean", "median"]:
    dummy = DummyRegressor(strategy=strategy)
    dummy.fit(X_train_reg, y_train_runtime)
    pred = dummy.predict(X_test_reg)
    runtime_baselines.append({
        "model": f"DummyRegressor_{strategy}",
        "target": "total_runtime_sec",
        "MAE": mean_absolute_error(y_test_runtime, pred),
        "RMSE": mean_squared_error(y_test_runtime, pred) ** 0.5,
        "R2": r2_score(y_test_runtime, pred),
    })

pd.DataFrame([runtime_metrics] + runtime_baselines)



## 11.1 Comparison with other runtime regressors



In [ ]:
runtime_regressors = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.001, l1_ratio=0.20, max_iter=5000, random_state=SEED),
    "SGD Regressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=SEED),
    "Decision Tree": DecisionTreeRegressor(random_state=SEED, min_samples_leaf=8),
    "Extra Trees": ExtraTreesRegressor(n_estimators=50, random_state=SEED, min_samples_leaf=3, n_jobs=1),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=SEED, min_samples_leaf=3, n_jobs=1),
}

runtime_model_comparison = []
for name, model in runtime_regressors.items():
    pipe = Pipeline([
        ("prep", preprocess_reg),
        ("model", model),
    ])
    pipe.fit(X_train_reg, y_train_runtime)
    pred = pipe.predict(X_test_reg)
    runtime_model_comparison.append({
        "model": name,
        "target": "total_runtime_sec",
        "MAE": mean_absolute_error(y_test_runtime, pred),
        "RMSE": mean_squared_error(y_test_runtime, pred) ** 0.5,
        "R2": r2_score(y_test_runtime, pred),
    })

runtime_model_comparison_df = pd.DataFrame(runtime_model_comparison).sort_values("RMSE", ascending=True)
runtime_model_comparison_df.to_csv(TAB_DIR / "runtime_regressor_model_comparison.csv", index=False)
runtime_model_comparison_df


In [ ]:
plt.figure(figsize=(9, 5))
runtime_model_comparison_df.set_index("model")["R2"].sort_values().plot(kind="barh")
plt.xlabel("R²")
plt.title("Runtime regressor comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "runtime_regressor_model_comparison.png", dpi=180)
plt.show()
plt.close()


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_runtime, y_pred_runtime, alpha=0.35)
min_v = min(y_test_runtime.min(), y_pred_runtime.min())
max_v = max(y_test_runtime.max(), y_pred_runtime.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Synthetic true runtime (s)")
plt.ylabel("Predicted runtime (s)")
plt.title("Total runtime prediction")
plt.tight_layout()
plt.savefig(FIG_DIR / "runtime_prediction.png", dpi=300)
plt.show()
plt.close()



## 12. Regression for expected fidelity



In [ ]:
y_train_fidelity = train_df["expected_fidelity"]
y_test_fidelity = test_df["expected_fidelity"]

fidelity_model = Pipeline([
    ("prep", preprocess_reg),
    ("model", RandomForestRegressor(
        n_estimators=50,
        random_state=SEED,
        min_samples_leaf=3,
        n_jobs=1,
    )),
])

fidelity_model.fit(X_train_reg, y_train_fidelity)
y_pred_fidelity = fidelity_model.predict(X_test_reg).clip(0, 1)

fidelity_metrics = {
    "model": "RandomForestRegressor",
    "target": "expected_fidelity",
    "evaluation": "holdout_group_split",
    "MAE": mean_absolute_error(y_test_fidelity, y_pred_fidelity),
    "RMSE": mean_squared_error(y_test_fidelity, y_pred_fidelity) ** 0.5,
    "R2": r2_score(y_test_fidelity, y_pred_fidelity),
}
fidelity_metrics



In [ ]:
fidelity_baselines = []
for strategy in ["mean", "median"]:
    dummy = DummyRegressor(strategy=strategy)
    dummy.fit(X_train_reg, y_train_fidelity)
    pred = dummy.predict(X_test_reg)
    fidelity_baselines.append({
        "model": f"DummyRegressor_{strategy}",
        "target": "expected_fidelity",
        "MAE": mean_absolute_error(y_test_fidelity, pred),
        "RMSE": mean_squared_error(y_test_fidelity, pred) ** 0.5,
        "R2": r2_score(y_test_fidelity, pred),
    })

pd.DataFrame([fidelity_metrics] + fidelity_baselines)



## 12.1 Comparison with other expected-fidelity regressors



In [ ]:
fidelity_regressors = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.001, l1_ratio=0.20, max_iter=5000, random_state=SEED),
    "SGD Regressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=SEED),
    "Decision Tree": DecisionTreeRegressor(random_state=SEED, min_samples_leaf=8),
    "Extra Trees": ExtraTreesRegressor(n_estimators=50, random_state=SEED, min_samples_leaf=3, n_jobs=1),
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=SEED, min_samples_leaf=3, n_jobs=1),
}

fidelity_model_comparison = []
for name, model in fidelity_regressors.items():
    pipe = Pipeline([
        ("prep", preprocess_reg),
        ("model", model),
    ])
    pipe.fit(X_train_reg, y_train_fidelity)
    pred = pipe.predict(X_test_reg).clip(0, 1)
    fidelity_model_comparison.append({
        "model": name,
        "target": "expected_fidelity",
        "MAE": mean_absolute_error(y_test_fidelity, pred),
        "RMSE": mean_squared_error(y_test_fidelity, pred) ** 0.5,
        "R2": r2_score(y_test_fidelity, pred),
    })

fidelity_model_comparison_df = pd.DataFrame(fidelity_model_comparison).sort_values("RMSE", ascending=True)
fidelity_model_comparison_df.to_csv(TAB_DIR / "fidelity_regressor_model_comparison.csv", index=False)
fidelity_model_comparison_df


In [ ]:
plt.figure(figsize=(9, 5))
fidelity_model_comparison_df.set_index("model")["R2"].sort_values().plot(kind="barh")
plt.xlabel("R²")
plt.title("Expected-fidelity regressor comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "fidelity_regressor_model_comparison.png", dpi=180)
plt.show()
plt.close()


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_fidelity, y_pred_fidelity, alpha=0.35)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Synthetic true fidelity")
plt.ylabel("Predicted fidelity")
plt.title("Expected fidelity prediction")
plt.tight_layout()
plt.savefig(FIG_DIR / "fidelity_prediction.png", dpi=300)
plt.show()
plt.close()



## 13. Execution recommendation classification



### 13.0 Interpretation of classification metrics

Accuracy, precision, recall, and F1 summarize the ability to reproduce the synthetic recommendation policy.



In [ ]:
X_train_clf = train_df[classifier_features]
X_test_clf = test_df[classifier_features]
y_train_clf = train_df["recommended_execution"]
y_test_clf = test_df["recommended_execution"]

clf_model = Pipeline([
    ("prep", preprocess_clf),
    ("model", RandomForestClassifier(
        n_estimators=50,
        random_state=SEED,
        class_weight="balanced",
        min_samples_leaf=3,
        n_jobs=1,
    )),
])

clf_model.fit(X_train_clf, y_train_clf)
y_pred_clf = clf_model.predict(X_test_clf)
y_prob_clf = clf_model.predict_proba(X_test_clf)[:, 1]

clf_metrics = {
    "model": "RandomForestClassifier",
    "task": "acceptable_backend_classification",
    "evaluation": "holdout_observed_runtime_fidelity",
    "Accuracy": accuracy_score(y_test_clf, y_pred_clf),
    "Precision": precision_score(y_test_clf, y_pred_clf, zero_division=0),
    "Recall": recall_score(y_test_clf, y_pred_clf, zero_division=0),
    "F1": f1_score(y_test_clf, y_pred_clf, zero_division=0),
}
clf_metrics



In [ ]:
print(classification_report(y_test_clf, y_pred_clf, target_names=["not recommended", "recommended"], zero_division=0))

cm = confusion_matrix(y_test_clf, y_pred_clf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not recommended", "recommended"])
disp.plot(values_format="d")
plt.title("Confusion matrix — recommendation")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_recommendation.png", dpi=300)
plt.show()
plt.close()



## 14. Comparison with baselines



In [ ]:
classification_baselines = []

for strategy in ["most_frequent", "stratified"]:
    dummy = DummyClassifier(strategy=strategy, random_state=SEED)
    dummy.fit(X_train_clf, y_train_clf)
    pred = dummy.predict(X_test_clf)
    classification_baselines.append({
        "model": f"DummyClassifier_{strategy}",
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(y_test_clf, pred, zero_division=0),
        "Recall": recall_score(y_test_clf, pred, zero_division=0),
        "F1": f1_score(y_test_clf, pred, zero_division=0),
    })

heuristic_pred = np.where(
    (test_df["expected_fidelity"] >= test_df["expected_fidelity"].quantile(0.55))
    & (test_df["total_runtime_sec"] <= test_df["total_runtime_sec"].quantile(0.60))
    & (test_df["failure_probability"] <= test_df["failure_probability"].quantile(0.70))
    & (test_df["backend_uptime"] >= 0.60),
    1,
    0,
)
classification_baselines.append({
    "model": "heuristic_fidelity_time_failure",
    "Accuracy": accuracy_score(y_test_clf, heuristic_pred),
    "Precision": precision_score(y_test_clf, heuristic_pred, zero_division=0),
    "Recall": recall_score(y_test_clf, heuristic_pred, zero_division=0),
    "F1": f1_score(y_test_clf, heuristic_pred, zero_division=0),
})

baseline_results = pd.DataFrame([clf_metrics] + classification_baselines)
baseline_results.to_csv(TAB_DIR / "classification_baselines.csv", index=False)
baseline_results



In [ ]:
plt.figure(figsize=(9, 5))
baseline_results.set_index("model")["F1"].sort_values().plot(kind="barh")
plt.xlabel("F1-score")
plt.title("Comparison between model and classification baselines")
plt.tight_layout()
plt.savefig(FIG_DIR / "classification_baseline_comparison.png", dpi=300)
plt.show()
plt.close()



## 15. Comparison with other classifiers



In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1200, class_weight="balanced"),
    "SGD Logistic": SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3, class_weight="balanced", random_state=SEED),
    "Linear SVM": LinearSVC(class_weight="balanced", max_iter=3000, dual=False, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED, class_weight="balanced", min_samples_leaf=8),
    "AdaBoost": AdaBoostClassifier(n_estimators=60, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=50,
        random_state=SEED,
        class_weight="balanced",
        min_samples_leaf=3,
        n_jobs=1,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        random_state=SEED,
        class_weight="balanced",
        min_samples_leaf=3,
        n_jobs=1,
    ),
}

model_comparison = []
for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess_clf),
        ("model", model),
    ])
    pipe.fit(X_train_clf, y_train_clf)
    pred = pipe.predict(X_test_clf)
    model_comparison.append({
        "model": name,
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(y_test_clf, pred, zero_division=0),
        "Recall": recall_score(y_test_clf, pred, zero_division=0),
        "F1": f1_score(y_test_clf, pred, zero_division=0),
    })

model_comparison_df = pd.DataFrame(model_comparison).sort_values("F1", ascending=False)
model_comparison_df.to_csv(TAB_DIR / "classifier_model_comparison.csv", index=False)
model_comparison_df


In [ ]:
plt.figure(figsize=(8, 5))
model_comparison_df.set_index("model")["F1"].sort_values().plot(kind="barh")
plt.xlabel("F1-score")
plt.title("Classifier comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "classifier_model_comparison.png", dpi=300)
plt.show()
plt.close()



## 15.1 End-to-end evaluation with predicted runtime and fidelity



In [ ]:
# Operational test set: replaces observed synthetic variables with predictions
X_test_clf_e2e = X_test_clf.copy()
X_test_clf_e2e["total_runtime_sec"] = y_pred_runtime
X_test_clf_e2e["expected_fidelity"] = y_pred_fidelity

y_pred_clf_e2e = clf_model.predict(X_test_clf_e2e)
y_prob_clf_e2e = clf_model.predict_proba(X_test_clf_e2e)[:, 1]

clf_metrics_e2e = {
    "model": "RandomForestClassifier_end_to_end",
    "task": "acceptable_backend_classification",
    "evaluation": "end_to_end_predicted_runtime_fidelity",
    "Accuracy": accuracy_score(y_test_clf, y_pred_clf_e2e),
    "Precision": precision_score(y_test_clf, y_pred_clf_e2e, zero_division=0),
    "Recall": recall_score(y_test_clf, y_pred_clf_e2e, zero_division=0),
    "F1": f1_score(y_test_clf, y_pred_clf_e2e, zero_division=0),
}

clf_metrics_observed = {
    "model": "RandomForestClassifier_observed",
    "task": "acceptable_backend_classification",
    "evaluation": "isolated_observed_synthetic_runtime_fidelity",
    "Accuracy": clf_metrics["Accuracy"],
    "Precision": clf_metrics["Precision"],
    "Recall": clf_metrics["Recall"],
    "F1": clf_metrics["F1"],
}

classification_end_to_end_df = pd.DataFrame([clf_metrics_observed, clf_metrics_e2e])
classification_end_to_end_df.to_csv(TAB_DIR / "classification_end_to_end.csv", index=False)
classification_end_to_end_df


### 15.2 End-to-end comparison between classifiers



In [ ]:
end_to_end_model_comparison = []
for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess_clf),
        ("model", model),
    ])
    pipe.fit(X_train_clf, y_train_clf)
    pred = pipe.predict(X_test_clf_e2e)
    end_to_end_model_comparison.append({
        "model": name,
        "evaluation": "end_to_end_predicted_runtime_fidelity",
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(y_test_clf, pred, zero_division=0),
        "Recall": recall_score(y_test_clf, pred, zero_division=0),
        "F1": f1_score(y_test_clf, pred, zero_division=0),
    })

end_to_end_model_comparison_df = pd.DataFrame(end_to_end_model_comparison).sort_values("F1", ascending=False)
end_to_end_model_comparison_df.to_csv(TAB_DIR / "classifier_end_to_end_model_comparison.csv", index=False)
end_to_end_model_comparison_df


In [ ]:
plt.figure(figsize=(8, 5))
end_to_end_model_comparison_df.set_index("model")["F1"].sort_values().plot(kind="barh")
plt.xlabel("F1-score")
plt.title("End-to-end classifier comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "classifier_end_to_end_model_comparison.png", dpi=300)
plt.show()
plt.close()


## 16. Feature importance



In [ ]:
perm = permutation_importance(
    clf_model,
    X_test_clf,
    y_test_clf,
    scoring="f1",
    n_repeats=3,
    random_state=SEED,
    n_jobs=1,
    max_samples=0.5,
)

importance_df = pd.DataFrame({
    "feature": classifier_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.to_csv(TAB_DIR / "feature_importance_permutation.csv", index=False)
importance_df.head(20)



In [ ]:
plt.figure(figsize=(8, 6))
importance_df.head(15).sort_values("importance_mean").plot(
    x="feature",
    y="importance_mean",
    kind="barh",
    legend=False,
    xerr="importance_std",
    ax=plt.gca(),
)
plt.xlabel("Mean permutation importance")
plt.ylabel("Feature")
plt.title("Main features for recommendation")
plt.tight_layout()
plt.savefig(FIG_DIR / "feature_importance.png", dpi=300)
plt.show()
plt.close()



## 16.1 Backend selection and ranking evaluation



### 16.1.0 Why evaluate ranking instead of classification only?

In operational orchestration, the main output is not only whether a backend is acceptable. The system must also choose one backend per workload. Ranking metrics evaluate whether the selected backend matches, or remains close to, the synthetic policy optimum.



In [ ]:
def score_candidates_for_selection(candidate_df):
    scored = candidate_df.copy()

    scored["predicted_total_runtime_sec"] = runtime_model.predict(scored[base_features])
    scored["predicted_fidelity"] = fidelity_model.predict(scored[base_features]).clip(0, 1)

    scored_for_clf = scored.copy()
    scored_for_clf["total_runtime_sec"] = scored["predicted_total_runtime_sec"]
    scored_for_clf["expected_fidelity"] = scored["predicted_fidelity"]

    scored["recommendation_probability"] = clf_model.predict_proba(scored_for_clf[classifier_features])[:, 1]
    scored["predicted_acceptable_backend"] = (scored["recommendation_probability"] >= 0.5).astype(int)

    scored["pred_time_score"] = 1 - within_workload_minmax(scored, "predicted_total_runtime_sec")
    scored["pred_cost_score"] = 1 - within_workload_minmax(scored, "total_cost")
    scored["pred_failure_score"] = 1 - scored["failure_probability"]
    scored["pred_capacity_score"] = 1 - within_workload_minmax(scored, "capacity_penalty")

    scored["predicted_decision_score"] = (
        scored["w_fidelity"] * scored["predicted_fidelity"]
        + scored["w_time"] * scored["pred_time_score"]
        + scored["w_cost"] * scored["pred_cost_score"]
        + scored["w_availability"] * scored["backend_uptime"]
        + scored["w_failure"] * scored["pred_failure_score"]
        + scored["w_hardware"] * scored["hardware_validation_value"]
        + scored["w_capacity"] * scored["pred_capacity_score"]
    )

    scored = add_predicted_selection(scored, prob_col="recommendation_probability", score_col="predicted_decision_score", threshold=0.5)
    return scored


selection_eval_df = score_candidates_for_selection(test_df)

true_selected = (
    selection_eval_df[selection_eval_df["selected_backend"] == 1]
    [["workload_id", "backend"]]
    .rename(columns={"backend": "true_backend"})
)

pred_selected = (
    selection_eval_df[selection_eval_df["predicted_selected_backend"] == 1]
    [["workload_id", "backend"]]
    .rename(columns={"backend": "predicted_backend"})
)

selection_pair_eval = true_selected.merge(pred_selected, on="workload_id", how="inner")
selection_pair_eval["top1_correct"] = selection_pair_eval["true_backend"].eq(selection_pair_eval["predicted_backend"])

rank_eval = selection_eval_df.merge(true_selected, on="workload_id", how="left")
rank_eval["is_true_backend"] = rank_eval["backend"].eq(rank_eval["true_backend"])

top1_accuracy = selection_pair_eval["top1_correct"].mean()
top2_accuracy = rank_eval.loc[rank_eval["predicted_backend_rank"] <= 2].groupby("workload_id")["is_true_backend"].max().mean()
top3_accuracy = rank_eval.loc[rank_eval["predicted_backend_rank"] <= 3].groupby("workload_id")["is_true_backend"].max().mean()

ranking_metrics = pd.DataFrame([{
    "Top-1 accuracy": top1_accuracy,
    "Top-2 accuracy": top2_accuracy,
    "Top-3 accuracy": top3_accuracy,
    "predicted_fallback_rate": selection_eval_df.loc[selection_eval_df["predicted_selected_backend"] == 1, "predicted_selection_fallback"].mean(),
    "true_fallback_rate": selection_eval_df.loc[selection_eval_df["selected_backend"] == 1, "selection_fallback"].mean(),
}])

selection_confusion = pd.crosstab(
    selection_pair_eval["true_backend"],
    selection_pair_eval["predicted_backend"],
    rownames=["true_backend"],
    colnames=["predicted_backend"],
    normalize="index",
)

selection_distribution_comparison = (
    pd.concat([
        selection_pair_eval["true_backend"].value_counts(normalize=True).rename("true_selection_rate"),
        selection_pair_eval["predicted_backend"].value_counts(normalize=True).rename("predicted_selection_rate"),
    ], axis=1)
    .fillna(0)
)
selection_distribution_comparison["diff_pred_minus_true"] = (
    selection_distribution_comparison["predicted_selection_rate"]
    - selection_distribution_comparison["true_selection_rate"]
)

ranking_metrics.to_csv(TAB_DIR / "backend_ranking_metrics.csv", index=False)
selection_confusion.to_csv(TAB_DIR / "backend_selection_confusion.csv")
selection_distribution_comparison.to_csv(TAB_DIR / "selection_distribution_comparison.csv")

display(ranking_metrics)
display(selection_distribution_comparison)
selection_confusion



## 17. Backend recommendation function



In [ ]:
def build_candidate_pairs_realistic(workload, random_seed=2026):
    """Combines one workload with all backends and generates the current operational state."""
    workload_df = pd.DataFrame([workload])
    workload_df = workload_df.merge(algorithm_profiles, on="algorithm", how="left")
    workload_df = workload_df.merge(decision_profiles, on="decision_profile", how="left")

    if "validation_need" not in workload_df.columns:
        workload_df["validation_need"] = (
            0.55 * workload_df["requires_real_hardware"]
            + 0.45 * workload_df["scientific_priority"]
        ).clip(0, 1)

    candidates = workload_df.merge(backend_catalog, how="cross")
    candidates = add_backend_state(candidates, random_seed=random_seed)
    candidates = simulate_operational_outcomes(candidates, random_seed=random_seed, generate_label=False)
    return candidates



def recommend_backend_v3(workload, random_seed=2026):
    candidates = build_candidate_pairs_realistic(workload, random_seed=random_seed)

    candidates["predicted_total_runtime_sec"] = runtime_model.predict(candidates[base_features])
    candidates["predicted_fidelity"] = fidelity_model.predict(candidates[base_features]).clip(0, 1)

    candidates_for_clf = candidates.copy()
    candidates_for_clf["total_runtime_sec"] = candidates["predicted_total_runtime_sec"]
    candidates_for_clf["expected_fidelity"] = candidates["predicted_fidelity"]

    candidates["recommendation_probability"] = clf_model.predict_proba(candidates_for_clf[classifier_features])[:, 1]
    candidates["acceptable_backend"] = clf_model.predict(candidates_for_clf[classifier_features])

    candidates["pred_time_score"] = 1 - minmax(candidates["predicted_total_runtime_sec"])
    candidates["pred_cost_score"] = 1 - minmax(candidates["total_cost"])
    candidates["pred_failure_score"] = 1 - candidates["failure_probability"]
    candidates["pred_capacity_score"] = 1 - minmax(candidates["capacity_penalty"])

    candidates["predicted_decision_score"] = (
        candidates["w_fidelity"] * candidates["predicted_fidelity"]
        + candidates["w_time"] * candidates["pred_time_score"]
        + candidates["w_cost"] * candidates["pred_cost_score"]
        + candidates["w_availability"] * candidates["backend_uptime"]
        + candidates["w_failure"] * candidates["pred_failure_score"]
        + candidates["w_hardware"] * candidates["hardware_validation_value"]
        + candidates["w_capacity"] * candidates["pred_capacity_score"]
    )

    # Operational selection: choose the best acceptable backend when available.
    # Otherwise, use the best score as fallback.
    candidates["workload_id"] = 0
    candidates = candidates.rename(columns={"acceptable_backend": "predicted_acceptable_backend"})
    candidates = add_predicted_selection(
        candidates,
        prob_col="recommendation_probability",
        score_col="predicted_decision_score",
        threshold=0.5,
    )
    candidates = candidates.rename(columns={"predicted_acceptable_backend": "acceptable_backend"})
    candidates["selected_backend"] = candidates["predicted_selected_backend"]

    output_cols = [
        "backend", "backend_family", "native_strength",
        "predicted_backend_rank", "selected_backend", "acceptable_backend",
        "predicted_selection_fallback",
        "predicted_total_runtime_sec", "predicted_fidelity",
        "recommendation_probability", "predicted_decision_score",
        "backend_uptime", "queue_depth", "queue_time_sec",
        "failure_probability", "total_cost", "hardware_validation_value",
        "capacity_penalty", "single_qubit_error", "two_qubit_error", "readout_error",
    ]

    return candidates[output_cols].sort_values(
        by=["predicted_backend_rank", "acceptable_backend", "recommendation_probability"],
        ascending=[True, False, False],
    ).reset_index(drop=True)



In [ ]:
example_workload = {
    "workload_id": 999999,
    "algorithm": "QAOA",
    "decision_profile": "research_validation",
    "scientific_priority": 0.88,
    "requires_real_hardware": 1,
    "deadline_hours": 24,
    "budget_usd": 20.0,
    "num_qubits": 18,
    "circuit_depth": 420,
    "two_qubit_gates": 260,
    "shots": 2048,
    "entanglement_score": 0.52,
}

recommendation_example = recommend_backend_v3(example_workload, random_seed=2026)
recommendation_example.to_csv(TAB_DIR / "example_backend_recommendation.csv", index=False)
recommendation_example



## 18. Noise sensitivity in QPUs



In [ ]:
noise_values = np.linspace(0.002, 0.060, 12)
sensitivity_results = []
original_catalog = backend_catalog.copy()

for noise in noise_values:
    backend_catalog.loc[backend_catalog["backend"].str.contains("qpu_cloud"), "base_two_qubit_error"] = noise
    candidates = build_candidate_pairs_realistic(example_workload, random_seed=2026)
    candidates["predicted_total_runtime_sec"] = runtime_model.predict(candidates[base_features])
    candidates["predicted_fidelity"] = fidelity_model.predict(candidates[base_features]).clip(0, 1)
    candidates["noise_level"] = noise
    candidates["pred_time_score"] = 1 - minmax(candidates["predicted_total_runtime_sec"])
    candidates["pred_cost_score"] = 1 - minmax(candidates["total_cost"])
    candidates["pred_failure_score"] = 1 - candidates["failure_probability"]
    candidates["pred_capacity_score"] = 1 - minmax(candidates["capacity_penalty"])
    candidates["predicted_decision_score"] = (
        candidates["w_fidelity"] * candidates["predicted_fidelity"]
        + candidates["w_time"] * candidates["pred_time_score"]
        + candidates["w_cost"] * candidates["pred_cost_score"]
        + candidates["w_availability"] * candidates["backend_uptime"]
        + candidates["w_failure"] * candidates["pred_failure_score"]
        + candidates["w_hardware"] * candidates["hardware_validation_value"]
        + candidates["w_capacity"] * candidates["pred_capacity_score"]
    )
    sensitivity_results.append(candidates)

backend_catalog = original_catalog.copy()
sensitivity_df = pd.concat(sensitivity_results, ignore_index=True)
sensitivity_df.to_csv(TAB_DIR / "sensitivity_qpu_noise.csv", index=False)
sensitivity_df.head()


In [ ]:
qpu_sensitivity = sensitivity_df[sensitivity_df["backend"].str.contains("qpu_cloud")].copy()

for metric, ylabel, filename, title in [
    ("predicted_fidelity", "Predicted fidelity", "sensitivity_qpu_noise_fidelity.png", "Effect of noise on predicted fidelity"),
    ("failure_probability", "Failure probability", "sensitivity_qpu_noise_failure.png", "Effect of noise on operational risk"),
    ("predicted_decision_score", "Predicted utility", "sensitivity_qpu_noise_utility.png", "Effect of noise on decision utility"),
]:
    plt.figure(figsize=(8, 5))
    for backend_name, group in qpu_sensitivity.groupby("backend"):
        group = group.sort_values("noise_level")
        plt.plot(group["noise_level"], group[metric], marker="o", label=backend_name)
    plt.xlabel("Synthetic two-qubit gate error on QPU")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300)
    plt.show()
plt.close()


## 19. Scalability by number of qubits



In [ ]:
scale_results = []
for q in range(4, 81, 4):
    workload = example_workload.copy()
    workload["num_qubits"] = q
    workload["circuit_depth"] = int(28 * q * 1.15)
    workload["two_qubit_gates"] = int(14 * q * 1.25)
    workload["entanglement_score"] = min(1.0, 0.012 * q)

    rec = recommend_backend_v3(workload, random_seed=2026)
    rec["num_qubits"] = q
    scale_results.append(rec)

scale_df = pd.concat(scale_results, ignore_index=True)
scale_df.to_csv(TAB_DIR / "scalability_by_qubits.csv", index=False)
scale_df.head()



In [ ]:
plt.figure(figsize=(8, 5))
for backend_name, group in scale_df.groupby("backend"):
    plt.plot(group["num_qubits"], group["predicted_total_runtime_sec"], label=backend_name)
plt.yscale("log")
plt.xlabel("Number of qubits")
plt.ylabel("Predicted total runtime (s, log scale)")
plt.title("Predicted scalability by backend")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "scalability_runtime_by_backend.png", dpi=300)
plt.show()
plt.close()



In [ ]:
plt.figure(figsize=(8, 5))
for backend_name, group in scale_df.groupby("backend"):
    plt.plot(group["num_qubits"], group["recommendation_probability"], label=backend_name)
plt.xlabel("Number of qubits")
plt.ylabel("Recommendation probability")
plt.title("Recommendation probability by workload scale")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "scalability_recommendation_probability.png", dpi=300)
plt.show()
plt.close()



## 20. Stress test under higher operational instability



In [ ]:
stress_workloads = generate_workloads(n_workloads=220, random_seed=123)
stress_df = stress_workloads.merge(backend_catalog, how="cross")
stress_df = add_backend_state(stress_df, random_seed=123, stress=True)
stress_df = simulate_operational_outcomes(stress_df, random_seed=123, stress=True, generate_label=True)
stress_df = assign_selected_backend(stress_df, utility_col="decision_utility", acceptable_col="recommended_execution")

stress_X = stress_df[classifier_features]
stress_y = stress_df["recommended_execution"]
stress_pred = clf_model.predict(stress_X)
stress_prob = clf_model.predict_proba(stress_X)[:, 1]

stress_metrics = {
    "Accuracy": accuracy_score(stress_y, stress_pred),
    "Precision": precision_score(stress_y, stress_pred, zero_division=0),
    "Recall": recall_score(stress_y, stress_pred, zero_division=0),
    "F1": f1_score(stress_y, stress_pred, zero_division=0),
    "positive_rate_stress": stress_y.mean(),
}

pd.DataFrame([stress_metrics]).to_csv(TAB_DIR / "stress_test_metrics.csv", index=False)
stress_metrics



In [ ]:
stress_summary_backend = stress_df.groupby("backend").agg(
    recommendation_rate=("recommended_execution", "mean"),
    mean_fidelity=("expected_fidelity", "mean"),
    mean_runtime_sec=("total_runtime_sec", "mean"),
    mean_failure_probability=("failure_probability", "mean"),
).sort_values("recommendation_rate", ascending=False)

stress_summary_backend.to_csv(TAB_DIR / "stress_summary_backend.csv")
stress_summary_backend



## 21. Consolidated tables for the paper



In [ ]:
table_dataset = pd.DataFrame({
    "metric": [
        "n_workloads",
        "n_backend_candidates",
        "n_pairs",
        "n_algorithms",
        "n_decision_profiles",
        "global_recommendation_rate",
        "workloads_with_one_selected_backend",
        "workloads_with_at_least_one_acceptable_backend",
        "workloads_without_acceptable_backend",
        "selected_backend_is_acceptable_rate",
        "fallback_selection_count",
        "group_split_train_workloads",
        "group_split_test_workloads",
    ],
    "value": [
        workloads["workload_id"].nunique(),
        backend_catalog["backend"].nunique(),
        df.shape[0],
        workloads["algorithm"].nunique(),
        workloads["decision_profile"].nunique(),
        df["recommended_execution"].mean(),
        int(df.groupby("workload_id")["selected_backend"].sum().eq(1).sum()),
        int(df.groupby("workload_id")["recommended_execution"].sum().gt(0).sum()),
        int(df.groupby("workload_id")["recommended_execution"].sum().eq(0).sum()),
        df.loc[df["selected_backend"] == 1, "recommended_execution"].mean(),
        int(df["selection_fallback"].sum()),
        train_df["workload_id"].nunique(),
        test_df["workload_id"].nunique(),
    ],
})

table_model_results = pd.DataFrame([
    runtime_metrics,
    fidelity_metrics,
    clf_metrics,
    clf_metrics_e2e,
    {"model": "Stress test", **stress_metrics},
])

table_dataset.to_csv(TAB_DIR / "table_dataset_description.csv", index=False)
table_model_results.to_csv(TAB_DIR / "table_model_results.csv", index=False)
ranking_metrics.to_csv(TAB_DIR / "table_backend_ranking_metrics.csv", index=False)
summary_backend.reset_index().to_csv(TAB_DIR / "table_backend_results.csv", index=False)
summary_algorithm.reset_index().to_csv(TAB_DIR / "table_algorithm_results.csv", index=False)
selected_backend_by_backend.reset_index().to_csv(TAB_DIR / "table_selected_backend_results.csv", index=False)

print("Tables saved in:", TAB_DIR)
display(table_dataset)
display(table_model_results)



## 22. Cross-validation by `workload_id`



In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

n_group_folds = N_GROUP_FOLDS
gkf = GroupKFold(n_splits=n_group_folds)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df, df["recommended_execution"], groups=df["workload_id"]), start=1):
    fold_train = df.iloc[tr_idx].copy()
    fold_val = df.iloc[va_idx].copy()

    fold_runtime_model = clone(runtime_model)
    fold_fidelity_model = clone(fidelity_model)
    fold_clf_model = clone(clf_model)

    fold_runtime_model.fit(fold_train[base_features], fold_train["total_runtime_sec"])
    fold_runtime_pred = fold_runtime_model.predict(fold_val[base_features])

    cv_rows.append({
        "fold": fold,
        "task": "runtime_regression",
        "target": "total_runtime_sec",
        "evaluation": "group_cv_by_workload",
        "MAE": mean_absolute_error(fold_val["total_runtime_sec"], fold_runtime_pred),
        "RMSE": mean_squared_error(fold_val["total_runtime_sec"], fold_runtime_pred) ** 0.5,
        "R2": r2_score(fold_val["total_runtime_sec"], fold_runtime_pred),
    })

    fold_fidelity_model.fit(fold_train[base_features], fold_train["expected_fidelity"])
    fold_fidelity_pred = fold_fidelity_model.predict(fold_val[base_features]).clip(0, 1)

    cv_rows.append({
        "fold": fold,
        "task": "fidelity_regression",
        "target": "expected_fidelity",
        "evaluation": "group_cv_by_workload",
        "MAE": mean_absolute_error(fold_val["expected_fidelity"], fold_fidelity_pred),
        "RMSE": mean_squared_error(fold_val["expected_fidelity"], fold_fidelity_pred) ** 0.5,
        "R2": r2_score(fold_val["expected_fidelity"], fold_fidelity_pred),
    })

    fold_clf_model.fit(fold_train[classifier_features], fold_train["recommended_execution"])
    fold_clf_pred_obs = fold_clf_model.predict(fold_val[classifier_features])

    cv_rows.append({
        "fold": fold,
        "task": "acceptable_backend_classification",
        "evaluation": "group_cv_observed_runtime_fidelity",
        "Accuracy": accuracy_score(fold_val["recommended_execution"], fold_clf_pred_obs),
        "Precision": precision_score(fold_val["recommended_execution"], fold_clf_pred_obs, zero_division=0),
        "Recall": recall_score(fold_val["recommended_execution"], fold_clf_pred_obs, zero_division=0),
        "F1": f1_score(fold_val["recommended_execution"], fold_clf_pred_obs, zero_division=0),
    })

    fold_val_e2e = fold_val[classifier_features].copy()
    fold_val_e2e["total_runtime_sec"] = fold_runtime_pred
    fold_val_e2e["expected_fidelity"] = fold_fidelity_pred
    fold_clf_pred_e2e = fold_clf_model.predict(fold_val_e2e)

    cv_rows.append({
        "fold": fold,
        "task": "acceptable_backend_classification",
        "evaluation": "group_cv_end_to_end_predicted_runtime_fidelity",
        "Accuracy": accuracy_score(fold_val["recommended_execution"], fold_clf_pred_e2e),
        "Precision": precision_score(fold_val["recommended_execution"], fold_clf_pred_e2e, zero_division=0),
        "Recall": recall_score(fold_val["recommended_execution"], fold_clf_pred_e2e, zero_division=0),
        "F1": f1_score(fold_val["recommended_execution"], fold_clf_pred_e2e, zero_division=0),
    })

cv_results_df = pd.DataFrame(cv_rows)

regression_metric_cols = ["MAE", "RMSE", "R2"]
classification_metric_cols = ["Accuracy", "Precision", "Recall", "F1"]

cv_results_regression = (
    cv_results_df.dropna(how="all", subset=regression_metric_cols)
    [["fold", "task", "target", "evaluation"] + regression_metric_cols]
    .reset_index(drop=True)
)
cv_results_classification = (
    cv_results_df.dropna(how="all", subset=classification_metric_cols)
    [["fold", "task", "evaluation"] + classification_metric_cols]
    .reset_index(drop=True)
)

group_cv_summary_regression = (
    cv_results_regression
    .groupby(["task", "target", "evaluation"], as_index=False)
    .agg({metric: ["mean", "std"] for metric in regression_metric_cols})
)
group_cv_summary_regression.columns = ["_".join([part for part in col if part]).strip("_") for col in group_cv_summary_regression.columns]

group_cv_summary_classification = (
    cv_results_classification
    .groupby(["task", "evaluation"], as_index=False)
    .agg({metric: ["mean", "std"] for metric in classification_metric_cols})
)
group_cv_summary_classification.columns = ["_".join([part for part in col if part]).strip("_") for col in group_cv_summary_classification.columns]

# Keep a combined table for auditing only, without main display.
cv_summary_df = pd.concat([
    group_cv_summary_regression.assign(metric_family="regression"),
    group_cv_summary_classification.assign(metric_family="classification"),
], ignore_index=True, sort=False)

cv_results_df.to_csv(TAB_DIR / "group_cv_fold_results_combined_audit.csv", index=False)
cv_results_regression.to_csv(TAB_DIR / "group_cv_fold_results_regression.csv", index=False)
cv_results_classification.to_csv(TAB_DIR / "group_cv_fold_results_classification.csv", index=False)
group_cv_summary_regression.to_csv(TAB_DIR / "group_cv_summary_regression.csv", index=False)
group_cv_summary_classification.to_csv(TAB_DIR / "group_cv_summary_classification.csv", index=False)
cv_summary_df.to_csv(TAB_DIR / "group_cv_summary_combined_audit.csv", index=False)

print("Workload-level cross-validation - regression")
display(cv_results_regression)
display(group_cv_summary_regression)

print("Workload-level cross-validation - classification")
display(cv_results_classification)
display(group_cv_summary_classification)



In [ ]:
plt.figure(figsize=(8, 5))
cv_plot = cv_results_df[cv_results_df["task"].eq("acceptable_backend_classification")]
cv_plot.groupby("evaluation")["F1"].mean().sort_values().plot(kind="barh")
plt.xlabel("Mean F1 across folds")
plt.title("Workload-level cross-validation: classification")
plt.tight_layout()
plt.savefig(FIG_DIR / "group_cv_classification_f1.png", dpi=300)
plt.show()
plt.close()


## 23. End-to-end evaluation with out-of-fold predictions in training



In [ ]:
def build_oof_performance_predictions(train_data, test_data, n_splits=N_OOF_FOLDS):
    """Generates out-of-fold runtime/fidelity predictions in training and predictions in test."""
    train_aug = train_data.copy()
    test_aug = test_data.copy()

    train_aug["oof_total_runtime_sec"] = np.nan
    train_aug["oof_expected_fidelity"] = np.nan

    gkf_inner = GroupKFold(n_splits=n_splits)
    for tr_idx_inner, va_idx_inner in gkf_inner.split(
        train_aug,
        train_aug["recommended_execution"],
        groups=train_aug["workload_id"],
    ):
        inner_train = train_aug.iloc[tr_idx_inner]
        inner_val = train_aug.iloc[va_idx_inner]

        rt = clone(runtime_model)
        fd = clone(fidelity_model)
        rt.fit(inner_train[base_features], inner_train["total_runtime_sec"])
        fd.fit(inner_train[base_features], inner_train["expected_fidelity"])

        train_aug.loc[inner_val.index, "oof_total_runtime_sec"] = rt.predict(inner_val[base_features])
        train_aug.loc[inner_val.index, "oof_expected_fidelity"] = fd.predict(inner_val[base_features]).clip(0, 1)

    # Final models trained on the full training set to predict the test set.
    rt_final = clone(runtime_model)
    fd_final = clone(fidelity_model)
    rt_final.fit(train_aug[base_features], train_aug["total_runtime_sec"])
    fd_final.fit(train_aug[base_features], train_aug["expected_fidelity"])

    test_aug["pred_total_runtime_sec"] = rt_final.predict(test_aug[base_features])
    test_aug["pred_expected_fidelity"] = fd_final.predict(test_aug[base_features]).clip(0, 1)

    return train_aug, test_aug, rt_final, fd_final


train_oof_df, test_oof_df, runtime_model_oof_final, fidelity_model_oof_final = build_oof_performance_predictions(
    train_df,
    test_df,
    n_splits=3,
)

X_train_clf_oof = train_oof_df[classifier_features].copy()
X_train_clf_oof["total_runtime_sec"] = train_oof_df["oof_total_runtime_sec"]
X_train_clf_oof["expected_fidelity"] = train_oof_df["oof_expected_fidelity"]

y_train_clf_oof = train_oof_df["recommended_execution"]

X_test_clf_oof = test_oof_df[classifier_features].copy()
X_test_clf_oof["total_runtime_sec"] = test_oof_df["pred_total_runtime_sec"]
X_test_clf_oof["expected_fidelity"] = test_oof_df["pred_expected_fidelity"]

y_test_clf_oof = test_oof_df["recommended_execution"]

clf_model_oof = clone(clf_model)
clf_model_oof.fit(X_train_clf_oof, y_train_clf_oof)

y_pred_clf_oof_e2e = clf_model_oof.predict(X_test_clf_oof)
y_prob_clf_oof_e2e = clf_model_oof.predict_proba(X_test_clf_oof)[:, 1]

oof_e2e_metrics = {
    "model": "RandomForestClassifier_oof_end_to_end",
    "task": "acceptable_backend_classification",
    "evaluation": "train_oof_test_predicted_runtime_fidelity",
    "Accuracy": accuracy_score(y_test_clf_oof, y_pred_clf_oof_e2e),
    "Precision": precision_score(y_test_clf_oof, y_pred_clf_oof_e2e, zero_division=0),
    "Recall": recall_score(y_test_clf_oof, y_pred_clf_oof_e2e, zero_division=0),
    "F1": f1_score(y_test_clf_oof, y_pred_clf_oof_e2e, zero_division=0),
}

oof_e2e_comparison_df = pd.DataFrame([
    clf_metrics_observed,
    clf_metrics_e2e,
    oof_e2e_metrics,
])

oof_e2e_comparison_df.to_csv(TAB_DIR / "classification_oof_end_to_end.csv", index=False)
oof_e2e_comparison_df


In [ ]:
plt.figure(figsize=(9, 5))
oof_e2e_comparison_df.set_index("evaluation")["F1"].sort_values().plot(kind="barh")
plt.xlabel("F1-score")
plt.title("Classification: observed, end-to-end, and OOF end-to-end")
plt.tight_layout()
plt.savefig(FIG_DIR / "classification_oof_end_to_end.png", dpi=300)
plt.show()
plt.close()


## 23.1 Selection and ranking with the OOF end-to-end pipeline



In [ ]:
def score_candidates_for_selection_oof(candidate_df):
    """Ranks candidates using the final end-to-end OOF pipeline."""
    scored = candidate_df.copy()

    scored["predicted_total_runtime_sec"] = runtime_model_oof_final.predict(scored[base_features])
    scored["predicted_fidelity"] = fidelity_model_oof_final.predict(scored[base_features]).clip(0, 1)

    scored_for_clf = scored[classifier_features].copy()
    scored_for_clf["total_runtime_sec"] = scored["predicted_total_runtime_sec"]
    scored_for_clf["expected_fidelity"] = scored["predicted_fidelity"]

    scored["recommendation_probability"] = clf_model_oof.predict_proba(scored_for_clf)[:, 1]
    scored["predicted_acceptable_backend"] = (scored["recommendation_probability"] >= 0.5).astype(int)

    scored["pred_time_score"] = 1 - within_workload_minmax(scored, "predicted_total_runtime_sec")
    scored["pred_cost_score"] = 1 - within_workload_minmax(scored, "total_cost")
    scored["pred_failure_score"] = 1 - scored["failure_probability"]
    scored["pred_capacity_score"] = 1 - within_workload_minmax(scored, "capacity_penalty")

    scored["predicted_decision_score"] = (
        scored["w_fidelity"] * scored["predicted_fidelity"]
        + scored["w_time"] * scored["pred_time_score"]
        + scored["w_cost"] * scored["pred_cost_score"]
        + scored["w_availability"] * scored["backend_uptime"]
        + scored["w_failure"] * scored["pred_failure_score"]
        + scored["w_hardware"] * scored["hardware_validation_value"]
        + scored["w_capacity"] * scored["pred_capacity_score"]
    )

    scored = add_predicted_selection(
        scored,
        prob_col="recommendation_probability",
        score_col="predicted_decision_score",
        threshold=0.5,
    )
    return scored


# From this point onward, selection metrics use the OOF end-to-end pipeline.
selection_eval_df = score_candidates_for_selection_oof(test_df)
selection_eval_df.to_csv(TAB_DIR / "selection_eval_oof_end_to_end.csv", index=False)

true_selected = (
    selection_eval_df[selection_eval_df["selected_backend"] == 1]
    [["workload_id", "backend"]]
    .rename(columns={"backend": "true_backend"})
)

pred_selected = (
    selection_eval_df[selection_eval_df["predicted_selected_backend"] == 1]
    [["workload_id", "backend"]]
    .rename(columns={"backend": "predicted_backend"})
)

selection_pair_eval = true_selected.merge(pred_selected, on="workload_id", how="inner")
selection_pair_eval["top1_correct"] = selection_pair_eval["true_backend"].eq(selection_pair_eval["predicted_backend"])

rank_eval = selection_eval_df.merge(true_selected, on="workload_id", how="left")
rank_eval["is_true_backend"] = rank_eval["backend"].eq(rank_eval["true_backend"])

top1_accuracy = selection_pair_eval["top1_correct"].mean()
top2_accuracy = rank_eval.loc[rank_eval["predicted_backend_rank"] <= 2].groupby("workload_id")["is_true_backend"].max().mean()
top3_accuracy = rank_eval.loc[rank_eval["predicted_backend_rank"] <= 3].groupby("workload_id")["is_true_backend"].max().mean()

ranking_metrics = pd.DataFrame([{
    "evaluation": "oof_end_to_end_selection",
    "Top-1 accuracy": top1_accuracy,
    "Top-2 accuracy": top2_accuracy,
    "Top-3 accuracy": top3_accuracy,
    "predicted_fallback_rate": selection_eval_df.loc[selection_eval_df["predicted_selected_backend"] == 1, "predicted_selection_fallback"].mean(),
    "true_fallback_rate": selection_eval_df.loc[selection_eval_df["selected_backend"] == 1, "selection_fallback"].mean(),
}])

selection_confusion = pd.crosstab(
    selection_pair_eval["true_backend"],
    selection_pair_eval["predicted_backend"],
    rownames=["true_backend"],
    colnames=["predicted_backend"],
    normalize="index",
)

selection_distribution_comparison = (
    pd.concat([
        selection_pair_eval["true_backend"].value_counts(normalize=True).rename("true_selection_rate"),
        selection_pair_eval["predicted_backend"].value_counts(normalize=True).rename("predicted_selection_rate"),
    ], axis=1)
    .fillna(0)
)
selection_distribution_comparison["diff_pred_minus_true"] = (
    selection_distribution_comparison["predicted_selection_rate"]
    - selection_distribution_comparison["true_selection_rate"]
)

ranking_metrics.to_csv(TAB_DIR / "backend_ranking_metrics_oof_end_to_end.csv", index=False)
ranking_metrics.to_csv(TAB_DIR / "backend_ranking_metrics.csv", index=False)
selection_confusion.to_csv(TAB_DIR / "backend_selection_confusion_oof_end_to_end.csv")
selection_confusion.to_csv(TAB_DIR / "backend_selection_confusion.csv")
selection_distribution_comparison.to_csv(TAB_DIR / "selection_distribution_comparison_oof_end_to_end.csv")
selection_distribution_comparison.to_csv(TAB_DIR / "selection_distribution_comparison.csv")

print("Ranking metrics recalculated with the OOF end-to-end pipeline.")
display(ranking_metrics)
display(selection_distribution_comparison)
selection_confusion


## 23.2 Comparison of final selection with operational heuristic policies



In [ ]:
# ============================================================
# 23.1 Comparison of final selection with heuristic policies
# ============================================================

def evaluate_selection_strategy(scored_df, strategy_name, selected_index_by_workload):
    selected = (
        scored_df.loc[list(selected_index_by_workload), ["workload_id", "backend", "decision_utility", "selection_fallback"]]
        .rename(columns={
            "backend": "strategy_backend",
            "decision_utility": "strategy_true_utility",
            "selection_fallback": "strategy_fallback_flag",
        })
    )
    true_sel = (
        scored_df[scored_df["selected_backend"].eq(1)]
        [["workload_id", "backend", "decision_utility"]]
        .rename(columns={"backend": "true_backend", "decision_utility": "true_selected_utility"})
    )
    merged = true_sel.merge(selected, on="workload_id", how="inner")
    merged["same_backend"] = merged["true_backend"].eq(merged["strategy_backend"])
    merged["utility_regret"] = (merged["true_selected_utility"] - merged["strategy_true_utility"]).clip(lower=0)
    merged["relative_utility_regret"] = merged["utility_regret"] / (merged["true_selected_utility"].abs() + 1e-9)
    return {
        "strategy": strategy_name,
        "Top-1 accuracy": merged["same_backend"].mean(),
        "mean_regret": merged["utility_regret"].mean(),
        "median_regret": merged["utility_regret"].median(),
        "p90_regret": merged["utility_regret"].quantile(0.90),
        "mean_relative_regret": merged["relative_utility_regret"].mean(),
        "low_regret_rate_le_0_03": (merged["utility_regret"] <= 0.03).mean(),
        "n_workloads": merged["workload_id"].nunique(),
    }


def _idxmax_by_workload(df_obj, col):
    return df_obj.groupby("workload_id")[col].idxmax()


def _idxmin_by_workload(df_obj, col):
    return df_obj.groupby("workload_id")[col].idxmin()

selection_strategy_rows = []

# Final OOF ML pipeline strategy.
ml_selected_idx = selection_eval_df[selection_eval_df["predicted_selected_backend"].eq(1)].index
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "ML_OOF_end_to_end", ml_selected_idx))

# Direct heuristics.
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "heuristic_min_predicted_runtime", _idxmin_by_workload(selection_eval_df, "predicted_total_runtime_sec")))
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "heuristic_max_predicted_fidelity", _idxmax_by_workload(selection_eval_df, "predicted_fidelity")))
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "heuristic_min_total_cost", _idxmin_by_workload(selection_eval_df, "total_cost")))
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "heuristic_min_failure_probability", _idxmin_by_workload(selection_eval_df, "failure_probability")))

# Multicriteria rule without classifier: uses only the predicted operational score.
selection_strategy_rows.append(evaluate_selection_strategy(selection_eval_df, "heuristic_weighted_score_no_classifier", _idxmax_by_workload(selection_eval_df, "predicted_decision_score")))

selection_strategy_comparison = pd.DataFrame(selection_strategy_rows).sort_values(["mean_regret", "Top-1 accuracy"], ascending=[True, False])
selection_strategy_comparison.to_csv(TAB_DIR / "selection_strategy_comparison_v9_6.csv", index=False)

print("Comparison between ML and heuristic selection policies:")
display(selection_strategy_comparison)



## 23.3 Feature-group ablation

This section measures how much each variable group contributes to the classification of acceptable backends. The ablation helps answer whether the decision depends more on workload attributes, backend attributes, operational state, or decision profile.



In [ ]:
# ============================================================
# 23.3 Feature-group ablation for classification
# ============================================================

def existing_columns(cols, data):
    return [c for c in cols if c in data.columns]

ablation_feature_sets = {
    "workload_only": existing_columns([
        "algorithm", "algorithm_family", "num_qubits", "circuit_depth", "two_qubit_gates", "shots",
        "entanglement_score", "noise_sensitivity", "scientific_priority", "requires_real_hardware", "validation_need",
        "deadline_hours", "budget_usd",
    ], df),
    "backend_static_only": existing_columns([
        "backend", "backend_family", "native_strength", "max_effective_qubits", "supports_real_hardware",
        "single_qubit_error", "two_qubit_error", "readout_error", "coupling_density", "hardware_validation_value",
    ], df),
    "operational_state_only": existing_columns([
        "calibration_age_hours", "queue_depth", "backend_uptime", "queue_time_sec", "total_cost",
        "failure_probability", "transpilation_overhead", "communication_latency_sec", "capacity_penalty", "qubit_pressure",
    ], df),
    "decision_profile_only": existing_columns([
        "decision_profile", "w_fidelity", "w_time", "w_cost", "w_availability", "w_failure", "w_hardware", "w_capacity",
    ], df),
    "predicted_performance_oracle": existing_columns([
        "expected_fidelity", "total_runtime_sec",
    ], df),
    "full_classifier_features": classifier_features,
}

ablation_rows = []
for config_name, features in ablation_feature_sets.items():
    if not features:
        continue
    cat_cols = [c for c in features if train_df[c].dtype == "object"]
    num_cols = [c for c in features if c not in cat_cols]
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ])
    model = Pipeline([
        ("prep", prep),
        ("model", RandomForestClassifier(
            n_estimators=40 if FAST_MODE else 70,
            random_state=SEED,
            class_weight="balanced",
            min_samples_leaf=3,
            n_jobs=1,
        )),
    ])
    model.fit(train_df[features], y_train_clf)
    pred = model.predict(test_df[features])
    ablation_rows.append({
        "feature_set": config_name,
        "n_features": len(features),
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(y_test_clf, pred, zero_division=0),
        "Recall": recall_score(y_test_clf, pred, zero_division=0),
        "F1": f1_score(y_test_clf, pred, zero_division=0),
    })

feature_ablation_results = pd.DataFrame(ablation_rows).sort_values("F1", ascending=False)
feature_ablation_results.to_csv(TAB_DIR / "feature_ablation_results_v9_6.csv", index=False)

print("Feature-group ablation results:")
display(feature_ablation_results)



## 23.4 Strong multicriteria and learning-to-rank baselines for full-paper validation

This section extends the benchmark beyond unidimensional heuristics. It adds classical multicriteria decision-making baselines and direct utility rankers, which are important for a full-paper evaluation because backend selection is inherently multicriteria rather than purely predictive.

The added baselines are:

- **TOPSIS**, using predicted runtime and fidelity plus operational cost, failure, availability, capacity, and hardware-alignment criteria;
- **Pareto dominance**, selecting alternatives that dominate more candidates within the same workload;
- **Random Forest utility ranker**, trained to predict the synthetic utility and rank candidates directly;
- **Gradient Boosting utility ranker**, an additional nonlinear utility-ranking baseline.

These baselines are evaluated with the same grouped test split and the same regret-based decision metrics used by the end-to-end ML pipeline.


In [ ]:
# ============================================================
# 23.4 Strong multicriteria and learning-to-rank baselines for full-paper validation
# ============================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


def _safe_group_minmax(values, benefit=True):
    values = pd.to_numeric(values, errors="coerce")
    vmin, vmax = values.min(), values.max()
    if pd.isna(vmin) or pd.isna(vmax) or vmax == vmin:
        return pd.Series(np.ones(len(values)) * 0.5, index=values.index)
    scaled = (values - vmin) / (vmax - vmin)
    return scaled if benefit else 1 - scaled


def add_topsis_scores(candidate_df):
    """Adds a TOPSIS score computed within each workload.

    Benefit criteria are larger-is-better. Cost criteria are converted to larger-is-better
    through workload-level min-max normalization.
    """
    scored = candidate_df.copy()

    if "predicted_total_runtime_sec" not in scored.columns:
        scored["predicted_total_runtime_sec"] = scored.get("total_runtime_sec", np.nan)
    if "predicted_fidelity" not in scored.columns:
        scored["predicted_fidelity"] = scored.get("expected_fidelity", np.nan)

    rows = []
    criterion_cols = [
        "topsis_fidelity", "topsis_time", "topsis_cost", "topsis_failure",
        "topsis_availability", "topsis_hardware", "topsis_capacity"
    ]

    for _, group in scored.groupby("workload_id", sort=False):
        g = group.copy()
        g["topsis_fidelity"] = _safe_group_minmax(g["predicted_fidelity"], benefit=True)
        g["topsis_time"] = _safe_group_minmax(g["predicted_total_runtime_sec"], benefit=False)
        g["topsis_cost"] = _safe_group_minmax(g["total_cost"], benefit=False)
        g["topsis_failure"] = _safe_group_minmax(g["failure_probability"], benefit=False)
        g["topsis_availability"] = _safe_group_minmax(g["backend_uptime"], benefit=True)
        g["topsis_hardware"] = _safe_group_minmax(g["hardware_validation_value"], benefit=True)
        g["topsis_capacity"] = _safe_group_minmax(g["capacity_penalty"], benefit=False)

        weights = np.array([
            float(g["w_fidelity"].iloc[0]),
            float(g["w_time"].iloc[0]),
            float(g["w_cost"].iloc[0]),
            float(g["w_failure"].iloc[0]),
            float(g["w_availability"].iloc[0]),
            float(g["w_hardware"].iloc[0]),
            float(g["w_capacity"].iloc[0]),
        ], dtype=float)
        weights = weights / (weights.sum() + 1e-12)

        matrix = g[criterion_cols].to_numpy(dtype=float)
        weighted = matrix * weights
        ideal = np.nanmax(weighted, axis=0)
        anti_ideal = np.nanmin(weighted, axis=0)
        dist_ideal = np.sqrt(np.nansum((weighted - ideal) ** 2, axis=1))
        dist_anti = np.sqrt(np.nansum((weighted - anti_ideal) ** 2, axis=1))
        g["topsis_score"] = dist_anti / (dist_ideal + dist_anti + 1e-12)
        rows.append(g)

    return pd.concat(rows, axis=0).sort_index()


def add_pareto_scores(candidate_df):
    """Adds a simple Pareto-dominance score within each workload."""
    scored = candidate_df.copy()
    if "predicted_total_runtime_sec" not in scored.columns:
        scored["predicted_total_runtime_sec"] = scored.get("total_runtime_sec", np.nan)
    if "predicted_fidelity" not in scored.columns:
        scored["predicted_fidelity"] = scored.get("expected_fidelity", np.nan)

    out = []
    for _, group in scored.groupby("workload_id", sort=False):
        g = group.copy()
        criteria = pd.DataFrame({
            "fidelity": _safe_group_minmax(g["predicted_fidelity"], benefit=True),
            "runtime": _safe_group_minmax(g["predicted_total_runtime_sec"], benefit=False),
            "cost": _safe_group_minmax(g["total_cost"], benefit=False),
            "failure": _safe_group_minmax(g["failure_probability"], benefit=False),
            "availability": _safe_group_minmax(g["backend_uptime"], benefit=True),
            "hardware": _safe_group_minmax(g["hardware_validation_value"], benefit=True),
            "capacity": _safe_group_minmax(g["capacity_penalty"], benefit=False),
        }, index=g.index)

        values = criteria.to_numpy(dtype=float)
        dominance_count = []
        dominated_flag = []
        for i in range(len(g)):
            vi = values[i]
            dominates_i = 0
            is_dominated = False
            for j in range(len(g)):
                if i == j:
                    continue
                vj = values[j]
                i_dominates_j = np.all(vi >= vj - 1e-12) and np.any(vi > vj + 1e-12)
                j_dominates_i = np.all(vj >= vi - 1e-12) and np.any(vj > vi + 1e-12)
                dominates_i += int(i_dominates_j)
                is_dominated = is_dominated or bool(j_dominates_i)
            dominance_count.append(dominates_i)
            dominated_flag.append(int(is_dominated))

        g["pareto_dominance_count"] = dominance_count
        g["pareto_non_dominated"] = 1 - pd.Series(dominated_flag, index=g.index)
        # Tie-break with the already available weighted predicted decision score.
        if "predicted_decision_score" in g.columns:
            tie_break = _safe_group_minmax(g["predicted_decision_score"], benefit=True)
        else:
            tie_break = criteria.mean(axis=1)
        g["pareto_score"] = g["pareto_non_dominated"] + 0.1 * g["pareto_dominance_count"] + 0.01 * tie_break
        out.append(g)
    return pd.concat(out, axis=0).sort_index()


# Multicriteria decision baselines.
selection_eval_df = add_topsis_scores(selection_eval_df)
selection_eval_df = add_pareto_scores(selection_eval_df)

selection_strategy_rows_extended = list(selection_strategy_rows)
selection_strategy_rows_extended.append(
    evaluate_selection_strategy(selection_eval_df, "topsis_multicriteria_predicted", _idxmax_by_workload(selection_eval_df, "topsis_score"))
)
selection_strategy_rows_extended.append(
    evaluate_selection_strategy(selection_eval_df, "pareto_dominance_predicted", _idxmax_by_workload(selection_eval_df, "pareto_score"))
)

# Direct utility rankers. They learn the synthetic utility directly and rank candidate backends.
# Test-time runtime and fidelity are replaced by end-to-end predictions to avoid using observed synthetic targets.
ranker_train = train_df.copy()
ranker_test = selection_eval_df.copy()
ranker_test_for_model = ranker_test.copy()
ranker_test_for_model["total_runtime_sec"] = ranker_test_for_model["predicted_total_runtime_sec"]
ranker_test_for_model["expected_fidelity"] = ranker_test_for_model["predicted_fidelity"]

utility_rankers = {
    "rf_direct_utility_ranker": RandomForestRegressor(
        n_estimators=80,
        random_state=SEED,
        min_samples_leaf=3,
        n_jobs=1,
    ),
    "gbr_direct_utility_ranker": GradientBoostingRegressor(
        random_state=SEED,
        n_estimators=120,
        learning_rate=0.05,
        max_depth=3,
    ),
}

utility_ranker_predictions = []
for ranker_name, ranker in utility_rankers.items():
    model = Pipeline([
        ("prep", preprocess_clf),
        ("model", ranker),
    ])
    model.fit(ranker_train[classifier_features], ranker_train["decision_utility"])
    pred_col = f"{ranker_name}_predicted_utility"
    selection_eval_df[pred_col] = model.predict(ranker_test_for_model[classifier_features])
    selection_strategy_rows_extended.append(
        evaluate_selection_strategy(selection_eval_df, ranker_name, _idxmax_by_workload(selection_eval_df, pred_col))
    )
    utility_ranker_predictions.append({"strategy": ranker_name, "prediction_column": pred_col})

selection_strategy_comparison_extended = (
    pd.DataFrame(selection_strategy_rows_extended)
    .drop_duplicates(subset=["strategy"], keep="last")
    .sort_values(["mean_regret", "Top-1 accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

selection_strategy_comparison_extended.to_csv(TAB_DIR / "selection_strategy_comparison_full_paper_v9_6.csv", index=False)
selection_eval_df.to_csv(TAB_DIR / "selection_eval_with_full_paper_baselines_v9_6.csv", index=False)

print("Extended comparison with multicriteria and utility-ranker baselines:")
display(selection_strategy_comparison_extended)



## 23.5 Training-size sensitivity for full-paper robustness

This section evaluates whether the recommendation classifier is stable when trained with fewer workloads. The goal is not to claim scalability of real hardware execution, but to test whether the synthetic benchmark produces stable decision boundaries as the amount of training data changes.

The test set remains fixed and grouped by `workload_id`; only the number of training workloads is varied.


In [ ]:
# ============================================================
# 23.5 Training-size sensitivity for full-paper robustness
# ============================================================

training_size_candidates = [100, 200, 400, train_df["workload_id"].nunique()]
training_size_candidates = sorted(set(int(x) for x in training_size_candidates if int(x) <= train_df["workload_id"].nunique()))

training_size_rows = []
train_workload_ids = np.array(sorted(train_df["workload_id"].unique()))

for n_train_workloads in training_size_candidates:
    local_rng = np.random.default_rng(SEED + n_train_workloads)
    sampled_workloads = local_rng.choice(train_workload_ids, size=n_train_workloads, replace=False)
    sub_train = train_df[train_df["workload_id"].isin(sampled_workloads)].copy()

    size_model = Pipeline([
        ("prep", preprocess_clf),
        ("model", RandomForestClassifier(
            n_estimators=60,
            random_state=SEED + n_train_workloads,
            class_weight="balanced",
            min_samples_leaf=3,
            n_jobs=1,
        )),
    ])
    size_model.fit(sub_train[classifier_features], sub_train["recommended_execution"])
    y_size_pred = size_model.predict(test_df[classifier_features])
    y_size_prob = size_model.predict_proba(test_df[classifier_features])[:, 1]

    training_size_rows.append({
        "n_train_workloads": n_train_workloads,
        "n_train_pairs": len(sub_train),
        "accuracy": accuracy_score(y_test_clf, y_size_pred),
        "precision": precision_score(y_test_clf, y_size_pred, zero_division=0),
        "recall": recall_score(y_test_clf, y_size_pred, zero_division=0),
        "f1": f1_score(y_test_clf, y_size_pred, zero_division=0),
        "mean_recommendation_probability": float(np.mean(y_size_prob)),
    })

training_size_sensitivity = pd.DataFrame(training_size_rows)
training_size_sensitivity.to_csv(TAB_DIR / "training_size_sensitivity_v9_6.csv", index=False)

print("Training-size sensitivity of the recommendation classifier:")
display(training_size_sensitivity)

plt.figure(figsize=(8, 5))
plt.plot(training_size_sensitivity["n_train_workloads"], training_size_sensitivity["f1"], marker="o", label="F1")
plt.plot(training_size_sensitivity["n_train_workloads"], training_size_sensitivity["accuracy"], marker="o", label="Accuracy")
plt.xlabel("Number of training workloads")
plt.ylabel("Score")
plt.title("Training-size sensitivity of recommendation classification")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "training_size_sensitivity_v9_6.png", dpi=180)
plt.show()



## 24. Utility loss in backend ranking

Top-k metrics indicate whether the correct backend appears among the first candidates, but they do not measure the practical cost of an error. In orchestration, missing the Top-1 backend may be operationally minor if the chosen backend has utility close to the best option. Therefore, this section computes utility regret:

\[
\text{regret} = U(\text{backend selected by the synthetic policy}) - U(\text{backend selected by the predictive pipeline}).
\]

Values close to zero indicate low operational impact from the selection error.



In [ ]:
true_selection_utility = (
    selection_eval_df[selection_eval_df["selected_backend"] == 1]
    [["workload_id", "backend", "decision_utility"]]
    .rename(columns={"backend": "true_backend", "decision_utility": "true_selected_utility"})
)

pred_selection_utility = (
    selection_eval_df[selection_eval_df["predicted_selected_backend"] == 1]
    [["workload_id", "backend", "decision_utility", "predicted_decision_score", "backend_rank_true"]]
    .rename(columns={
        "backend": "predicted_backend",
        "decision_utility": "predicted_selected_true_utility",
        "backend_rank_true": "predicted_backend_true_rank",
    })
)

utility_regret_df = true_selection_utility.merge(pred_selection_utility, on="workload_id", how="inner")
utility_regret_df["utility_regret"] = (
    utility_regret_df["true_selected_utility"] - utility_regret_df["predicted_selected_true_utility"]
)
utility_regret_df["utility_regret"] = utility_regret_df["utility_regret"].clip(lower=0)
utility_regret_df["relative_utility_regret"] = utility_regret_df["utility_regret"] / (
    utility_regret_df["true_selected_utility"].abs() + 1e-9
)
utility_regret_df["same_backend"] = utility_regret_df["true_backend"].eq(utility_regret_df["predicted_backend"])

utility_regret_summary = pd.DataFrame([{
    "mean_regret": utility_regret_df["utility_regret"].mean(),
    "median_regret": utility_regret_df["utility_regret"].median(),
    "p90_regret": utility_regret_df["utility_regret"].quantile(0.90),
    "mean_relative_regret": utility_regret_df["relative_utility_regret"].mean(),
    "median_relative_regret": utility_regret_df["relative_utility_regret"].median(),
    "low_regret_rate_le_0_01": (utility_regret_df["utility_regret"] <= 0.01).mean(),
    "low_regret_rate_le_0_03": (utility_regret_df["utility_regret"] <= 0.03).mean(),
    "low_regret_rate_le_0_05": (utility_regret_df["utility_regret"] <= 0.05).mean(),
    "exact_backend_match_rate": utility_regret_df["same_backend"].mean(),
}])

utility_regret_df.to_csv(TAB_DIR / "utility_regret_by_workload.csv", index=False)
utility_regret_summary.to_csv(TAB_DIR / "utility_regret_summary.csv", index=False)

display(utility_regret_summary)
display(utility_regret_df.head())


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(utility_regret_df["utility_regret"], bins=25)
plt.xlabel("Utility loss")
plt.ylabel("Number of workloads")
plt.title("Distribution of utility loss in backend selection")
plt.tight_layout()
plt.savefig(FIG_DIR / "utility_regret_distribution.png", dpi=300)
plt.show()
plt.close()


## 25. Subgroup performance analysis

Global analysis may hide performance differences across backends, algorithms, and decision profiles. This section computes subgroup metrics for end-to-end classification with OOF predictions and for backend selection/ranking.



In [ ]:
def subgroup_classification_metrics(data, y_true, y_pred, group_col):
    rows = []
    temp = data.copy()
    temp["y_true"] = np.asarray(y_true)
    temp["y_pred"] = np.asarray(y_pred)
    for group_value, group in temp.groupby(group_col):
        rows.append({
            "group_type": group_col,
            "group_value": group_value,
            "n": len(group),
            "positive_rate": group["y_true"].mean(),
            "Accuracy": accuracy_score(group["y_true"], group["y_pred"]),
            "Precision": precision_score(group["y_true"], group["y_pred"], zero_division=0),
            "Recall": recall_score(group["y_true"], group["y_pred"], zero_division=0),
            "F1": f1_score(group["y_true"], group["y_pred"], zero_division=0),
        })
    return pd.DataFrame(rows)

subgroup_clf_results = pd.concat([
    subgroup_classification_metrics(test_oof_df, y_test_clf_oof, y_pred_clf_oof_e2e, "backend"),
    subgroup_classification_metrics(test_oof_df, y_test_clf_oof, y_pred_clf_oof_e2e, "backend_family"),
    subgroup_classification_metrics(test_oof_df, y_test_clf_oof, y_pred_clf_oof_e2e, "algorithm"),
    subgroup_classification_metrics(test_oof_df, y_test_clf_oof, y_pred_clf_oof_e2e, "decision_profile"),
], ignore_index=True)

subgroup_clf_results.to_csv(TAB_DIR / "subgroup_classification_metrics.csv", index=False)
subgroup_clf_results


In [ ]:
workload_metadata_test = (
    test_df[["workload_id", "algorithm", "algorithm_family", "decision_profile"]]
    .drop_duplicates("workload_id")
)

selection_workload_eval = (
    selection_pair_eval
    .merge(workload_metadata_test, on="workload_id", how="left")
    .merge(utility_regret_df[["workload_id", "utility_regret", "relative_utility_regret", "predicted_backend_true_rank"]], on="workload_id", how="left")
)

ranking_subgroup_rows = []
for group_col in ["algorithm", "algorithm_family", "decision_profile", "true_backend"]:
    for group_value, group in selection_workload_eval.groupby(group_col):
        ranking_subgroup_rows.append({
            "group_type": group_col,
            "group_value": group_value,
            "n_workloads": len(group),
            "Top1_accuracy": group["top1_correct"].mean(),
            "mean_utility_regret": group["utility_regret"].mean(),
            "median_utility_regret": group["utility_regret"].median(),
            "mean_relative_utility_regret": group["relative_utility_regret"].mean(),
            "low_regret_rate_le_0_03": (group["utility_regret"] <= 0.03).mean(),
            "mean_true_rank_of_predicted_backend": group["predicted_backend_true_rank"].mean(),
        })

subgroup_ranking_results = pd.DataFrame(ranking_subgroup_rows)
subgroup_ranking_results.to_csv(TAB_DIR / "subgroup_ranking_metrics.csv", index=False)
subgroup_ranking_results


In [ ]:
plt.figure(figsize=(9, 5))
plot_df = subgroup_clf_results[subgroup_clf_results["group_type"].eq("backend")].sort_values("F1")
plt.barh(plot_df["group_value"], plot_df["F1"])
plt.xlabel("F1 end-to-end OOF")
plt.ylabel("Backend")
plt.title("Classification performance by backend")
plt.tight_layout()
plt.savefig(FIG_DIR / "subgroup_classification_f1_by_backend.png", dpi=300)
plt.show()
plt.close()

plt.figure(figsize=(9, 5))
plot_df = subgroup_ranking_results[subgroup_ranking_results["group_type"].eq("decision_profile")].sort_values("mean_utility_regret")
plt.barh(plot_df["group_value"], plot_df["mean_utility_regret"])
plt.xlabel("Mean utility loss")
plt.ylabel("Decision profile")
plt.title("Utility loss by decision profile")
plt.tight_layout()
plt.savefig(FIG_DIR / "subgroup_utility_regret_by_profile.png", dpi=300)
plt.show()
plt.close()


## 26. External data validation and plausibility anchoring

This section is included so the notebook does **not depend only on synthetic data**. The main dataset remains synthetic and controlled, but its operating ranges are compared with an external reference.

The validation follows three priority levels:

1. **IBM Quantum/Qiskit metadata collected through the API**, when `IBM_QUANTUM_TOKEN` is configured.
2. **Qiskit snapshots/fake backends**, when available in the environment.
3. **Static plausibility reference**, embedded in the notebook and used only when neither API access nor fake providers are available.

The section also runs small circuits with **Qiskit/Aer** when the package is available. This does not replace validation on real QPUs, but it adds a reproducible operational check for GHZ, Hamiltonian-like, and QAOA-like circuits.

The purpose is an external plausibility validation: checking whether simulated qubit counts, gate errors, readout errors, connectivity, and coherence times are compatible with reference QPU ranges. This step is **not used for model training**, avoiding contamination of the supervised evaluation.



### 26.0 Methodological objective of external validation

External validation does not change model training. It acts as an independent plausibility check. The question answered is:

> Are synthetic QPU variables in orders of magnitude compatible with external references?

This reduces the purely synthetic character of the study, but it does not turn the dataset into a complete empirical dataset. When the source is a fake backend or a static reference, that fact is explicitly recorded in the tables.



In [ ]:
# ============================================================
# 26.0 External validation: packages, directories, and execution mode
# ============================================================

from pathlib import Path
import os
import sys
import time
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("outputs_ai_quantum_v9_6")
DATA_DIR = OUTPUT_DIR / "data"
TAB_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
REAL_VALIDATION_DIR = OUTPUT_DIR / "real_validation"
for _d in [DATA_DIR, TAB_DIR, FIG_DIR, REAL_VALIDATION_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("External validation directory:", REAL_VALIDATION_DIR.resolve())

# Optional installation in pure Python to keep the cell valid outside Colab/IPython.
INSTALL_VALIDATION_PACKAGES_IF_MISSING = False

qiskit_available = False
qiskit_aer_available = False
qiskit_ibm_available = False

def _install_validation_package_if_missing(import_name, pip_name):
    try:
        __import__(import_name)
        return "already_installed"
    except Exception:
        if not INSTALL_VALIDATION_PACKAGES_IF_MISSING:
            return "missing_not_installed"
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pip_name])
        __import__(import_name)
        return "installed_now"

for _import_name, _pip_name in [("qiskit", "qiskit"), ("qiskit_aer", "qiskit-aer"), ("qiskit_ibm_runtime", "qiskit-ibm-runtime")]:
    print(_pip_name, _install_validation_package_if_missing(_import_name, _pip_name))


try:
    import qiskit
    from qiskit import QuantumCircuit, transpile
    qiskit_available = True
    print("qiskit:", getattr(qiskit, "__version__", "unidentified version"))
except Exception as e:
    print("qiskit unavailable:", type(e).__name__, str(e))

try:
    import qiskit_aer
    from qiskit_aer import AerSimulator
    qiskit_aer_available = True
    print("qiskit-aer:", getattr(qiskit_aer, "__version__", "unidentified version"))
except Exception as e:
    print("qiskit-aer unavailable:", type(e).__name__, str(e))

try:
    import qiskit_ibm_runtime
    from qiskit_ibm_runtime import QiskitRuntimeService
    qiskit_ibm_available = True
    print("qiskit-ibm-runtime:", getattr(qiskit_ibm_runtime, "__version__", "unidentified version"))
except Exception as e:
    print("qiskit-ibm-runtime unavailable:", type(e).__name__, str(e))

package_status = pd.DataFrame([
    {"package": "qiskit", "available": qiskit_available},
    {"package": "qiskit-aer", "available": qiskit_aer_available},
    {"package": "qiskit-ibm-runtime", "available": qiskit_ibm_available},
])

display(package_status)
package_status.to_csv(REAL_VALIDATION_DIR / "package_status_v9_6.csv", index=False)



In [ ]:
# ============================================================
# 26.1 Loading the main synthetic dataset
# ============================================================

synthetic_df_for_validation = None
for candidate_name in ["df", "data", "dataset"]:
    if candidate_name in globals() and isinstance(globals()[candidate_name], pd.DataFrame):
        synthetic_df_for_validation = globals()[candidate_name].copy()
        print(f"Synthetic dataset found in variable `{candidate_name}`:", synthetic_df_for_validation.shape)
        break

if synthetic_df_for_validation is None:
    candidate_csv = DATA_DIR / "synthetic_quantum_orchestration_v3.csv"
    if candidate_csv.exists():
        synthetic_df_for_validation = pd.read_csv(candidate_csv)
        print("Synthetic dataset loaded from:", candidate_csv, synthetic_df_for_validation.shape)

if synthetic_df_for_validation is None:
    raise RuntimeError(
        "The synthetic dataset was not found. Run the dataset-generation cells before external validation."
    )

# Focus of the external comparison: QPU backends.
if "backend_family" in synthetic_df_for_validation.columns:
    synthetic_qpu_df = synthetic_df_for_validation[
        synthetic_df_for_validation["backend_family"].astype(str).str.lower().eq("qpu")
    ].copy()
else:
    synthetic_qpu_df = synthetic_df_for_validation.copy()

if synthetic_qpu_df.empty:
    synthetic_qpu_df = synthetic_df_for_validation.copy()

print("Synthetic records used in QPU validation:", synthetic_qpu_df.shape)
print("Synthetic backends:", sorted(synthetic_qpu_df.get("backend", pd.Series(dtype=str)).astype(str).unique().tolist()))



In [ ]:
# Robust installation/import of Qiskit Aer to ensure that
# local validation is not skipped because of a missing package.
import sys, subprocess, importlib

def _ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", pip_name])
        return importlib.import_module(import_name)

try:
    from qiskit import QuantumCircuit, transpile
except Exception:
    _ensure_import("qiskit", "qiskit")
    from qiskit import QuantumCircuit, transpile

try:
    from qiskit_aer import AerSimulator
except Exception:
    _ensure_import("qiskit_aer", "qiskit-aer")
    from qiskit_aer import AerSimulator

# ============================================================
# 26.2 Local execution of small circuits with Qiskit/Aer
# ============================================================

def build_ghz_circuit(n_qubits: int):
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.h(0)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc


def build_hamiltonian_like_circuit(n_qubits: int, layers: int = 2):
    qc = QuantumCircuit(n_qubits, n_qubits)
    for q in range(n_qubits):
        qc.h(q)
    for _ in range(layers):
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
            qc.rz(0.15, q + 1)
            qc.cx(q, q + 1)
        for q in range(n_qubits):
            qc.rx(0.25, q)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc


def build_qaoa_like_circuit(n_qubits: int, p: int = 1):
    qc = QuantumCircuit(n_qubits, n_qubits)
    for q in range(n_qubits):
        qc.h(q)
    for _ in range(p):
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
            qc.rz(0.30, q + 1)
            qc.cx(q, q + 1)
        for q in range(n_qubits):
            qc.rx(0.40, q)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

qiskit_aer_small_circuit_runs = pd.DataFrame()

if qiskit_available and qiskit_aer_available:
    simulator = AerSimulator()
    small_circuits = [
        ("GHZ_3", build_ghz_circuit(3)),
        ("GHZ_5", build_ghz_circuit(5)),
        ("HAM_like_4", build_hamiltonian_like_circuit(4, layers=2)),
        ("QAOA_like_4", build_qaoa_like_circuit(4, p=1)),
    ]

    rows = []
    for circuit_name, qc in small_circuits:
        try:
            tqc = transpile(qc, simulator, optimization_level=1)
            start = time.perf_counter()
            result = simulator.run(tqc, shots=1024).result()
            runtime_sec = time.perf_counter() - start
            counts = result.get_counts()
            ops = tqc.count_ops()
            top_outcome = max(counts, key=counts.get) if counts else None

            rows.append({
                "circuit": circuit_name,
                "num_qubits": qc.num_qubits,
                "original_depth": qc.depth(),
                "transpiled_depth": tqc.depth(),
                "original_size": qc.size(),
                "transpiled_size": tqc.size(),
                "two_qubit_gates": int(sum(v for k, v in ops.items() if k in ["cx", "cz", "ecr", "swap"])),
                "num_outcomes": len(counts),
                "runtime_sec": runtime_sec,
                "top_outcome": top_outcome,
                "top_outcome_probability": counts.get(top_outcome, 0) / 1024 if top_outcome is not None else np.nan,
                "ops": json.dumps(dict(ops)),
                "status": "ok",
                "source": "qiskit_aer_local_simulator",
            })
        except Exception as e:
            rows.append({
                "circuit": circuit_name,
                "num_qubits": getattr(qc, "num_qubits", np.nan),
                "original_depth": getattr(qc, "depth", lambda: np.nan)(),
                "transpiled_depth": np.nan,
                "original_size": getattr(qc, "size", lambda: np.nan)(),
                "transpiled_size": np.nan,
                "two_qubit_gates": np.nan,
                "num_outcomes": np.nan,
                "runtime_sec": np.nan,
                "top_outcome": None,
                "top_outcome_probability": np.nan,
                "ops": "{}",
                "status": f"error: {type(e).__name__}: {str(e)}",
                "source": "qiskit_aer_local_simulator",
            })

    qiskit_aer_small_circuit_runs = pd.DataFrame(rows)
else:
    qiskit_aer_small_circuit_runs = pd.DataFrame([{
        "circuit": "not_run",
        "num_qubits": np.nan,
        "original_depth": np.nan,
        "transpiled_depth": np.nan,
        "original_size": np.nan,
        "transpiled_size": np.nan,
        "two_qubit_gates": np.nan,
        "num_outcomes": np.nan,
        "runtime_sec": np.nan,
        "top_outcome": None,
        "top_outcome_probability": np.nan,
        "ops": "{}",
        "status": "qiskit_or_aer_unavailable",
        "source": "qiskit_aer_local_simulator",
    }])

display(qiskit_aer_small_circuit_runs)
qiskit_aer_small_circuit_runs.to_csv(REAL_VALIDATION_DIR / "qiskit_aer_small_circuit_runs_v9_6.csv", index=False)



### 26.3 Source-aware external QPU metadata synchronization

This cell synchronizes the external metadata variable used in the validation section. It may use live IBM/Qiskit Runtime metadata, cached metadata, or a static/fallback reference, but the selected source is classified explicitly.

**Important:** only rows whose evidence scope is `live_ibm_qiskit_runtime_metadata` should be reported as live IBM/Qiskit Runtime validation. If the scope is `static_plausibility_reference_not_current`, the correct claim is static plausibility anchoring.



In [ ]:
# ============================================================
# 26.3 Source-aware synchronization of external QPU metadata
# ============================================================
# The main collection occurs in Section 1.1. This cell does not convert static
# references into live metadata. It synchronizes the best available source and
# records the evidence scope used by the external validation tables.

from pathlib import Path
import numpy as np
import pandas as pd

REAL_VALIDATION_DIR = Path("outputs_ai_quantum_v9_6/real_validation")
REAL_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_QPU_METADATA_CSV = REAL_VALIDATION_DIR / "external_qpu_metadata_v9_6.csv"
EXTERNAL_QPU_SCOPE_CSV = REAL_VALIDATION_DIR / "external_anchor_evidence_scope_v9_6.csv"


def _safe_read_csv_v26(path):
    path = Path(path)
    try:
        if not path.exists() or path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

selected_external_metadata = pd.DataFrame()
selected_origin = "not_available"

candidate_sources = []
if (
    "external_qpu_metadata_for_generation" in globals()
    and isinstance(external_qpu_metadata_for_generation, pd.DataFrame)
    and not external_qpu_metadata_for_generation.empty
):
    candidate_sources.append(("variable_external_qpu_metadata_for_generation", external_qpu_metadata_for_generation.copy()))
if (
    "external_qpu_metadata" in globals()
    and isinstance(external_qpu_metadata, pd.DataFrame)
    and not external_qpu_metadata.empty
):
    candidate_sources.append(("variable_external_qpu_metadata", external_qpu_metadata.copy()))

for candidate_path in [
    Path("external_qpu_cache/external_qpu_metadata_for_generation_v9_6.csv"),
    Path("external_qpu_cache/ibm_backend_metadata_live_v9_6.csv"),
    Path("external_qpu_cache/external_qpu_metadata_for_generation_v9_5.csv"),
    Path("external_qpu_cache/ibm_backend_metadata_live_v9_5.csv"),
]:
    tmp = _safe_read_csv_v26(candidate_path)
    if not tmp.empty:
        candidate_sources.append((str(candidate_path), tmp))

if candidate_sources:
    # Prefer live metadata, then cached IBM/Qiskit metadata, then static/snapshot references.
    ranked = []
    for origin, data in candidate_sources:
        scope_df = classify_external_anchor_evidence(data) if "classify_external_anchor_evidence" in globals() else pd.DataFrame([{"anchor_evidence_scope": "unclassified_external_reference", "live_ibm_runtime_available": False}])
        scope = scope_df["anchor_evidence_scope"].iloc[0]
        live = bool(scope_df["live_ibm_runtime_available"].iloc[0]) if "live_ibm_runtime_available" in scope_df.columns else False
        if live:
            rank = 0
        elif "cached_ibm" in scope:
            rank = 1
        elif "snapshot" in scope:
            rank = 2
        elif "static" in scope:
            rank = 3
        else:
            rank = 4
        ranked.append((rank, origin, data, scope_df))
    ranked.sort(key=lambda x: x[0])
    _, selected_origin, selected_external_metadata, selected_scope = ranked[0]
else:
    selected_scope = classify_external_anchor_evidence(pd.DataFrame()) if "classify_external_anchor_evidence" in globals() else pd.DataFrame()

if selected_external_metadata.empty:
    external_qpu_metadata = pd.DataFrame()
    external_qpu_metadata_for_generation = pd.DataFrame()
else:
    if "collection_status" not in selected_external_metadata.columns:
        selected_external_metadata["collection_status"] = "unknown"
    if "evidence_level" not in selected_external_metadata.columns:
        selected_external_metadata["evidence_level"] = np.where(
            selected_external_metadata.get("source", pd.Series(dtype=str)).astype(str).str.contains("ibm_qiskit_runtime_live_metadata", case=False, na=False),
            "live_runtime_metadata",
            "plausibility_or_static_reference",
        )
    if "backend_type" not in selected_external_metadata.columns:
        selected_external_metadata["backend_type"] = "qpu"
    if "backend_family" not in selected_external_metadata.columns:
        selected_external_metadata["backend_family"] = np.where(
            selected_external_metadata.get("qpu_modality", pd.Series("", index=selected_external_metadata.index)).astype(str).str.contains("ion", case=False, na=False),
            "qpu_cloud_iontrap",
            "qpu_cloud_superconducting",
        )

    selected_scope = classify_external_anchor_evidence(selected_external_metadata) if "classify_external_anchor_evidence" in globals() else selected_scope
    selected_external_metadata["anchor_evidence_scope"] = selected_scope["anchor_evidence_scope"].iloc[0]
    selected_external_metadata["live_ibm_runtime_available"] = bool(selected_scope["live_ibm_runtime_available"].iloc[0])

    external_qpu_metadata = selected_external_metadata.copy()
    external_qpu_metadata_for_generation = selected_external_metadata.copy()
    external_qpu_metadata.to_csv(EXTERNAL_QPU_METADATA_CSV, index=False)
    selected_scope.to_csv(EXTERNAL_QPU_SCOPE_CSV, index=False)

print("External QPU metadata synchronized for validation.")
print("Origin:", selected_origin)
print("Rows:", len(external_qpu_metadata))
print("Source-aware evidence scope:")
display(selected_scope)
if isinstance(external_qpu_metadata, pd.DataFrame) and not external_qpu_metadata.empty:
    display(
        external_qpu_metadata[
            [c for c in [
                "backend", "source", "collection_status", "evidence_level", "anchor_evidence_scope",
                "live_ibm_runtime_available", "anchor_priority", "n_qubits", "operational", "pending_jobs",
                "median_1q_gate_error", "median_2q_gate_error", "median_readout_error"
            ] if c in external_qpu_metadata.columns]
        ]
    )
    if not bool(selected_scope["live_ibm_runtime_available"].iloc[0]):
        print("IMPORTANT: this validation is not live IBM/Qiskit Runtime validation. Use the recommended_claim in the scope table.")
print("External metadata file:", EXTERNAL_QPU_METADATA_CSV)
print("Evidence scope file:", EXTERNAL_QPU_SCOPE_CSV)



In [ ]:

# ============================================================
# 26.4 Real-vs-synthetic comparison for QPUs — corrected by modality
# ============================================================
# - IBM/Qiskit Runtime provides metadata for superconducting backends;
# - these metadata are compared only with synthetic superconducting rows;
# - ion-trap profiles are not validated against IBM superconducting data;
# - modalities without external reference are explicitly marked as unanchored.

import numpy as np
import pandas as pd
from pathlib import Path

REAL_VALIDATION_DIR = Path("outputs_ai_quantum_v9_6/real_validation")
REAL_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

if "df" in globals():
    synthetic_df_for_validation = df.copy()
elif (DATA_DIR / "synthetic_quantum_orchestration_v3.csv").exists():
    synthetic_df_for_validation = pd.read_csv(DATA_DIR / "synthetic_quantum_orchestration_v3.csv")
else:
    synthetic_df_for_validation = pd.DataFrame()

if "external_qpu_metadata" not in globals():
    external_qpu_metadata = pd.DataFrame()

# Reuse the function defined in Section 7.2 when available; otherwise define a minimal fallback.
if "build_modality_aware_real_vs_synthetic_report" not in globals():
    def build_modality_aware_real_vs_synthetic_report(real_metadata, synthetic_data, output_prefix="real_vs_synthetic_qpu_v9_6"):
        return pd.DataFrame(), pd.DataFrame([{"compatible_metrics_pct": np.nan, "metrics_with_external_reference": 0}])

real_vs_synthetic_qpu_ranges, real_data_validation_score = build_modality_aware_real_vs_synthetic_report(
    external_qpu_metadata,
    synthetic_df_for_validation,
    output_prefix="real_vs_synthetic_qpu_v9_6",
)

real_vs_synthetic_qpu_ranges.to_csv(REAL_VALIDATION_DIR / "real_vs_synthetic_qpu_ranges_v9_6.csv", index=False)
real_data_validation_score.to_csv(REAL_VALIDATION_DIR / "real_data_validation_score_v9_6.csv", index=False)

print("Real-vs-synthetic comparison by QPU modality:")
display(real_vs_synthetic_qpu_ranges)
print("External validation summary:")
display(real_data_validation_score)



In [ ]:
# ============================================================
# 26.5 Internal consistency validation of the synthetic dataset
# ============================================================

required_columns = [
    "workload_id", "algorithm", "backend", "backend_family", "num_qubits", "circuit_depth",
    "two_qubit_gates", "shots", "total_runtime_sec", "expected_fidelity",
    "failure_probability", "decision_utility", "recommended_execution", "selected_backend"
]

internal_rows = []
for col in required_columns:
    internal_rows.append({
        "check": f"required_column::{col}",
        "status": "pass" if col in synthetic_df_for_validation.columns else "fail",
        "value": col in synthetic_df_for_validation.columns,
        "details": "required column present" if col in synthetic_df_for_validation.columns else "required column absent",
    })

range_checks = [
    ("expected_fidelity", 0.0, 1.0),
    ("failure_probability", 0.0, 1.0),
    ("recommendation_probability_true", 0.0, 1.0),
    ("backend_uptime", 0.0, 1.0),
    ("coupling_density", 0.0, 1.0),
]

for col, lo, hi in range_checks:
    if col in synthetic_df_for_validation.columns:
        vals = pd.to_numeric(synthetic_df_for_validation[col], errors="coerce")
        ok = vals.dropna().between(lo, hi).all()
        internal_rows.append({
            "check": f"range::{col}",
            "status": "pass" if ok else "fail",
            "value": float(vals.dropna().between(lo, hi).mean() * 100) if vals.notna().any() else np.nan,
            "details": f"percentage within [{lo}, {hi}]",
        })

non_negative_cols = [
    "num_qubits", "circuit_depth", "two_qubit_gates", "shots", "queue_time_sec", "total_cost",
    "execution_time_sec", "total_runtime_sec", "single_qubit_error", "two_qubit_error", "readout_error", "decision_utility"
]
for col in non_negative_cols:
    if col in synthetic_df_for_validation.columns:
        vals = pd.to_numeric(synthetic_df_for_validation[col], errors="coerce")
        ok = (vals.dropna() >= 0).all()
        internal_rows.append({
            "check": f"non_negative::{col}",
            "status": "pass" if ok else "fail",
            "value": float((vals.dropna() >= 0).mean() * 100) if vals.notna().any() else np.nan,
            "details": "percentage of non-negative values",
        })

# Checks the correct rule: exactly one backend selected per workload.
if {"workload_id", "selected_backend"}.issubset(synthetic_df_for_validation.columns):
    selected_sum_per_workload = synthetic_df_for_validation.groupby("workload_id")["selected_backend"].sum()
    ok = selected_sum_per_workload.eq(1).all()
    internal_rows.append({
        "check": "selected_backend_exactly_one_by_workload",
        "status": "pass" if ok else "fail",
        "value": float(selected_sum_per_workload.eq(1).mean() * 100),
        "details": "percentage of workloads with exactly one selected backend",
    })

# Checks whether fallback occurs only when there is no acceptable backend.
required_for_fallback = {"workload_id", "recommended_execution", "selection_fallback", "selected_backend"}
if required_for_fallback.issubset(synthetic_df_for_validation.columns):
    acceptable_sum = synthetic_df_for_validation.groupby("workload_id")["recommended_execution"].sum()
    fallback_by_workload = synthetic_df_for_validation.groupby("workload_id")["selection_fallback"].sum()
    fallback_expected = acceptable_sum.eq(0).astype(int)
    ok = fallback_by_workload.eq(fallback_expected).all()
    internal_rows.append({
        "check": "fallback_only_when_no_acceptable_backend",
        "status": "pass" if ok else "fail",
        "value": float(fallback_by_workload.eq(fallback_expected).mean() * 100),
        "details": "percentage of workloads where fallback coincides with no acceptable backend",
    })

synthetic_data_quality_checks = pd.DataFrame(internal_rows)
synthetic_data_quality_checks.to_csv(REAL_VALIDATION_DIR / "synthetic_data_quality_checks_v9_6.csv", index=False)

print("Internal consistency validation of the dataset:")
display(synthetic_data_quality_checks)



In [ ]:
# ============================================================
# 26.6 Final summary of source-aware external validation
# ============================================================

def _component_status(df_obj, positive_col=None, positive_value=None):
    if not isinstance(df_obj, pd.DataFrame) or df_obj.empty:
        return "not_available_or_not_run"
    if positive_col is not None and positive_col in df_obj.columns:
        return "executed" if df_obj[positive_col].astype(str).eq(str(positive_value)).any() else "not_available_or_not_run"
    return "executed"


def external_metadata_status(df_obj):
    if "classify_external_anchor_evidence" in globals():
        scope = classify_external_anchor_evidence(df_obj)
        return scope["anchor_evidence_scope"].iloc[0]
    if not isinstance(df_obj, pd.DataFrame) or df_obj.empty:
        return "not_available_or_not_run"
    statuses = set(df_obj.get("collection_status", pd.Series(dtype=str)).astype(str))
    evidence = set(df_obj.get("evidence_level", pd.Series(dtype=str)).astype(str))
    if "live_runtime_metadata" in evidence:
        return "live_ibm_qiskit_runtime_metadata"
    if "fallback_static_reference" in statuses or "plausibility_or_static_reference" in evidence:
        return "static_plausibility_reference_not_current"
    return "external_metadata_available_unknown_scope"

internal_pass_rate = np.nan
if isinstance(synthetic_data_quality_checks, pd.DataFrame) and not synthetic_data_quality_checks.empty:
    internal_pass_rate = float(synthetic_data_quality_checks["status"].eq("pass").mean() * 100.0)

external_compatible_rate = np.nan
if isinstance(real_data_validation_score, pd.DataFrame) and not real_data_validation_score.empty:
    external_compatible_rate = real_data_validation_score["compatible_metrics_pct"].iloc[0]

external_scope_df = classify_external_anchor_evidence(external_qpu_metadata) if "classify_external_anchor_evidence" in globals() else pd.DataFrame([{
    "anchor_evidence_scope": external_metadata_status(external_qpu_metadata),
    "live_ibm_runtime_available": False,
    "recommended_claim": "external metadata available, source not classified",
    "paper_wording": "The external metadata source was not classified.",
}])
external_scope = external_scope_df["anchor_evidence_scope"].iloc[0]
live_available = bool(external_scope_df["live_ibm_runtime_available"].iloc[0]) if "live_ibm_runtime_available" in external_scope_df.columns else False
recommended_claim = external_scope_df["recommended_claim"].iloc[0] if "recommended_claim" in external_scope_df.columns else "not_available"
paper_wording = external_scope_df["paper_wording"].iloc[0] if "paper_wording" in external_scope_df.columns else "not_available"

real_validation_summary = pd.DataFrame([
    {
        "component": "synthetic_data_internal_quality_checks",
        "status": "executed",
        "rows": len(synthetic_data_quality_checks),
        "score": internal_pass_rate,
        "output_file": str(REAL_VALIDATION_DIR / "synthetic_data_quality_checks_v9_6.csv"),
        "interpretation": "Checks schema, ranges, non-negative variables, and selected-backend consistency.",
    },
    {
        "component": "qiskit_aer_small_circuit_execution",
        "status": _component_status(qiskit_aer_small_circuit_runs, "status", "ok"),
        "rows": len(qiskit_aer_small_circuit_runs),
        "score": float(qiskit_aer_small_circuit_runs["status"].eq("ok").mean() * 100.0) if "status" in qiskit_aer_small_circuit_runs.columns else np.nan,
        "output_file": str(REAL_VALIDATION_DIR / "qiskit_aer_small_circuit_runs_v9_6.csv"),
        "interpretation": "Local simulator execution of representative small circuits.",
    },
    {
        "component": "external_qpu_metadata",
        "status": external_scope,
        "rows": len(external_qpu_metadata),
        "score": np.nan,
        "output_file": str(REAL_VALIDATION_DIR / "external_qpu_metadata_v9_6.csv"),
        "live_ibm_runtime_available": live_available,
        "recommended_claim": recommended_claim,
        "interpretation": paper_wording,
    },
    {
        "component": "external_vs_synthetic_qpu_range_comparison",
        "status": "executed" if isinstance(real_vs_synthetic_qpu_ranges, pd.DataFrame) and not real_vs_synthetic_qpu_ranges.empty else "not_available_or_not_run",
        "rows": len(real_vs_synthetic_qpu_ranges),
        "score": external_compatible_rate,
        "output_file": str(REAL_VALIDATION_DIR / "real_vs_synthetic_qpu_ranges_v9_6.csv"),
        "live_ibm_runtime_available": live_available,
        "recommended_claim": recommended_claim,
        "interpretation": "Live IBM/Qiskit Runtime comparison." if live_available else "Static/cache/snapshot plausibility comparison; not live Runtime validation.",
    },
])

external_scope_df.to_csv(REAL_VALIDATION_DIR / "external_anchor_evidence_scope_v9_6.csv", index=False)
real_validation_summary.to_csv(REAL_VALIDATION_DIR / "real_validation_summary_v9_6.csv", index=False)

print("Final source-aware external validation summary:")
display(real_validation_summary)
print("Evidence scope to use in the paper:")
display(external_scope_df)
if not live_available:
    print("IMPORTANT: Do not claim live IBM/Qiskit Runtime validation for this run. Use the recommended_claim above.")



## 26.7 Extended real-metadata calibration and anchoring sensitivity

This section expands the external plausibility analysis for the full-paper version. It evaluates how the synthetic QPU catalog moves toward the external metadata as the anchoring coefficient changes.

The reported quantity is the mean absolute log-distance between calibrated synthetic backend parameters and external metadata medians. Lower values indicate closer order-of-magnitude alignment. Modalities without compatible external references are explicitly marked instead of being treated as validation failures.


In [ ]:
# ============================================================
# 26.7 Extended real-metadata calibration and anchoring sensitivity
# ============================================================


def build_extended_anchor_calibration_report(base_catalog, anchor_reference, alphas):
    if not isinstance(base_catalog, pd.DataFrame) or base_catalog.empty:
        return pd.DataFrame(), pd.DataFrame()
    if not isinstance(anchor_reference, pd.DataFrame):
        anchor_reference = pd.DataFrame()

    metric_map = {
        "single_qubit_error": ("base_single_qubit_error", "median_1q_gate_error"),
        "two_qubit_error": ("base_two_qubit_error", "median_2q_gate_error"),
        "readout_error": ("base_readout_error", "median_readout_error"),
        "coupling_density": ("base_coupling_density", "coupling_density"),
        "t1_us": ("base_t1_us", "median_t1_us"),
        "t2_us": ("base_t2_us", "median_t2_us"),
        "queue_depth": ("base_queue_depth", "pending_jobs"),
    }

    rows = []
    summary_rows = []
    for alpha in alphas:
        calibrated = calibrate_qpu_backend_catalog(base_catalog, anchor_reference, alpha=alpha)
        qpus = calibrated[calibrated["backend_family"].eq("qpu")].copy()
        distances_for_alpha = []
        with_ref_count = 0
        without_ref_count = 0

        for _, backend_row in qpus.iterrows():
            modality = str(backend_row.get("qpu_modality", "")).lower().strip()
            if modality in ("", "nan", "none", "not_applicable"):
                modality = "iontrap" if "ion" in str(backend_row.get("backend", "")).lower() else "superconducting"
            ref_mod = anchor_reference[anchor_reference.get("qpu_modality", pd.Series(dtype=str)).astype(str).str.lower().eq(modality)].copy()

            if ref_mod.empty:
                without_ref_count += 1
                rows.append({
                    "alpha": alpha,
                    "backend": backend_row.get("backend"),
                    "qpu_modality": modality,
                    "metric": "all",
                    "synthetic_value": np.nan,
                    "external_median": np.nan,
                    "abs_log10_distance": np.nan,
                    "status": "no_external_reference_for_modality",
                })
                continue

            with_ref_count += 1
            ref_medians = ref_mod.median(numeric_only=True)
            for metric, (synthetic_col, real_col) in metric_map.items():
                synthetic_value = backend_row.get(synthetic_col, np.nan)
                external_median = ref_medians.get(real_col, np.nan)
                dist = _relative_log_distance(synthetic_value, external_median)
                if not pd.isna(dist):
                    distances_for_alpha.append(dist)
                rows.append({
                    "alpha": alpha,
                    "backend": backend_row.get("backend"),
                    "qpu_modality": modality,
                    "metric": metric,
                    "synthetic_column": synthetic_col,
                    "external_column": real_col,
                    "synthetic_value": synthetic_value,
                    "external_median": external_median,
                    "abs_log10_distance": dist,
                    "status": "with_external_reference" if not pd.isna(dist) else "metric_not_available",
                })

        summary_rows.append({
            "alpha": alpha,
            "mean_abs_log10_distance": float(np.nanmean(distances_for_alpha)) if len(distances_for_alpha) else np.nan,
            "median_abs_log10_distance": float(np.nanmedian(distances_for_alpha)) if len(distances_for_alpha) else np.nan,
            "metrics_with_distance": int(len(distances_for_alpha)),
            "qpu_backends_with_reference": int(with_ref_count),
            "qpu_backends_without_reference": int(without_ref_count),
        })

    return pd.DataFrame(rows), pd.DataFrame(summary_rows)


extended_anchor_detail, extended_anchor_summary = build_extended_anchor_calibration_report(
    backend_catalog_raw if "backend_catalog_raw" in globals() else backend_catalog,
    qpu_generation_anchor_reference if "qpu_generation_anchor_reference" in globals() else external_qpu_metadata,
    ANCHOR_ALPHA_SENSITIVITY_VALUES if "ANCHOR_ALPHA_SENSITIVITY_VALUES" in globals() else [0, 0.25, 0.45, 0.75, 1.0],
)

extended_anchor_detail.to_csv(REAL_VALIDATION_DIR / "extended_anchor_calibration_detail_v9_6.csv", index=False)
extended_anchor_summary.to_csv(REAL_VALIDATION_DIR / "extended_anchor_calibration_summary_v9_6.csv", index=False)

print("Extended anchoring sensitivity summary:")
display(extended_anchor_summary)

if not extended_anchor_summary.empty and extended_anchor_summary["mean_abs_log10_distance"].notna().any():
    plt.figure(figsize=(8, 5))
    plt.plot(extended_anchor_summary["alpha"], extended_anchor_summary["mean_abs_log10_distance"], marker="o")
    plt.xlabel("Anchoring coefficient alpha")
    plt.ylabel("Mean absolute log10 distance")
    plt.title("Sensitivity of synthetic QPU catalog to real-metadata anchoring")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "anchor_alpha_distance_sensitivity_v9_6.png", dpi=180)
    plt.show()
else:
    print("No numeric external-reference distance could be computed for the available metadata.")



## 26.8 Optional small-scale hardware execution protocol

This optional section documents and, when explicitly enabled, executes a limited hardware sanity check on IBM Quantum Runtime. It is disabled by default to preserve open-science reproducibility without credentials, queue usage, or paid access.

For a full-paper experimental run, enable this cell only after confirming backend availability, quota, and institutional authorization. The experiment is intentionally small and should be reported as a limited sanity check, not as full operational validation.


In [ ]:
# ============================================================
# 26.8 Optional small-scale hardware execution protocol
# ============================================================

RUN_OPTIONAL_IBM_HARDWARE_SANITY_CHECK = False
HARDWARE_SANITY_SHOTS = 256
HARDWARE_SANITY_MAX_CIRCUITS = 3

hardware_sanity_protocol = pd.DataFrame([
    {"circuit": "GHZ_3", "qubits": 3, "purpose": "entanglement-sensitive sanity check"},
    {"circuit": "GHZ_5", "qubits": 5, "purpose": "larger entanglement sanity check"},
    {"circuit": "QAOA_like_4", "qubits": 4, "purpose": "small variational-workload proxy"},
    {"circuit": "HAM_like_4", "qubits": 4, "purpose": "small Hamiltonian-like workload proxy"},
])
hardware_sanity_protocol.to_csv(REAL_VALIDATION_DIR / "hardware_sanity_protocol_v9_6.csv", index=False)

hardware_sanity_results = []

if not RUN_OPTIONAL_IBM_HARDWARE_SANITY_CHECK:
    hardware_sanity_results.append({
        "status": "skipped_by_default",
        "reason": "Open-science execution avoids credentialed hardware usage unless explicitly enabled.",
        "shots": HARDWARE_SANITY_SHOTS,
        "max_circuits": HARDWARE_SANITY_MAX_CIRCUITS,
    })
else:
    try:
        from qiskit import QuantumCircuit, transpile
        from qiskit_ibm_runtime import QiskitRuntimeService

        token = os.environ.get("IBM_QUANTUM_TOKEN") or os.environ.get("QISKIT_IBM_TOKEN") or ""
        instance = os.environ.get("IBM_QUANTUM_INSTANCE") or os.environ.get("QISKIT_IBM_INSTANCE") or ""
        if token and instance:
            service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token, instance=instance)
        elif token:
            service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
        else:
            service = QiskitRuntimeService(channel="ibm_quantum_platform")

        candidate_backends = []
        for b in service.backends(simulator=False, operational=True):
            try:
                n_qubits = getattr(b, "num_qubits", None) or b.configuration().n_qubits
                status = b.status()
                pending = getattr(status, "pending_jobs", np.nan)
                candidate_backends.append((pending, n_qubits, b))
            except Exception:
                pass
        if not candidate_backends:
            raise RuntimeError("No operational hardware backend was available.")
        candidate_backends = sorted(candidate_backends, key=lambda x: (x[0], -x[1]))
        backend = candidate_backends[0][2]
        backend_name = backend.name() if callable(getattr(backend, "name", None)) else str(getattr(backend, "name", backend))

        circuits = []
        names = []
        for circuit_name in hardware_sanity_protocol["circuit"].head(HARDWARE_SANITY_MAX_CIRCUITS):
            if circuit_name == "GHZ_3":
                qc = QuantumCircuit(3, 3)
                qc.h(0); qc.cx(0, 1); qc.cx(1, 2); qc.measure(range(3), range(3))
            elif circuit_name == "GHZ_5":
                qc = QuantumCircuit(5, 5)
                qc.h(0)
                for i in range(4): qc.cx(i, i + 1)
                qc.measure(range(5), range(5))
            elif circuit_name == "QAOA_like_4":
                qc = QuantumCircuit(4, 4)
                qc.h(range(4))
                qc.cx(0, 1); qc.rz(0.7, 1); qc.cx(0, 1)
                qc.cx(2, 3); qc.rz(0.7, 3); qc.cx(2, 3)
                qc.rx(0.4, range(4)); qc.measure(range(4), range(4))
            else:
                qc = QuantumCircuit(4, 4)
                qc.h(range(4)); qc.cx(0, 1); qc.rz(0.3, 1); qc.cx(1, 2); qc.rz(0.3, 2); qc.measure(range(4), range(4))
            circuits.append(qc)
            names.append(circuit_name)

        transpiled = transpile(circuits, backend=backend, optimization_level=1, seed_transpiler=SEED)
        job = backend.run(transpiled, shots=HARDWARE_SANITY_SHOTS)
        result = job.result()

        for name, circuit, tqc in zip(names, circuits, transpiled):
            counts = result.get_counts(tqc)
            hardware_sanity_results.append({
                "status": "ok",
                "backend": backend_name,
                "job_id": job.job_id(),
                "circuit": name,
                "original_depth": circuit.depth(),
                "transpiled_depth": tqc.depth(),
                "shots": HARDWARE_SANITY_SHOTS,
                "top_outcome": max(counts, key=counts.get) if counts else None,
                "top_outcome_count": max(counts.values()) if counts else np.nan,
                "num_observed_bitstrings": len(counts) if counts else 0,
            })
    except Exception as e:
        hardware_sanity_results.append({
            "status": "failed",
            "error_type": type(e).__name__,
            "error_message": str(e)[:500],
        })

hardware_sanity_results = pd.DataFrame(hardware_sanity_results)
hardware_sanity_results.to_csv(REAL_VALIDATION_DIR / "hardware_sanity_results_v9_6.csv", index=False)

print("Optional hardware sanity-check status:")
display(hardware_sanity_protocol)
display(hardware_sanity_results)



### How to report this validation in the paper

Use the evidence scope generated by the notebook before choosing the wording.

If `external_anchor_evidence_scope_v9_6.csv` reports `live_ibm_qiskit_runtime_metadata`, a suitable formulation is:

> As a complementary step, the synthetic dataset was subjected to an external plausibility validation using live IBM/Qiskit Runtime metadata for superconducting QPU profiles. This validation compared synthetic attributes such as qubit count, gate errors, readout error, coherence times, connectivity, and queue-related metadata with the live metadata available to the configured account. IBM/Qiskit Runtime metadata are treated as a superconducting reference and are not used to validate ion-trap profiles. This step is not used for training and does not constitute full operational validation on production QPUs.

If the scope reports `static_plausibility_reference_not_current`, `qiskit_snapshot_or_fake_backend_reference`, or `cached_ibm_qiskit_metadata`, use the more cautious formulation:

> As a complementary step, the synthetic dataset was subjected to a source-aware external plausibility check. In this execution, the comparison used cached, snapshot, or static reference data rather than live IBM/Qiskit Runtime metadata. Therefore, the check supports only order-of-magnitude plausibility of selected QPU parameters and does not constitute live Runtime validation or operational validation on production QPUs.

In the paper, use the expression **external plausibility validation** or **external sanity check**, and avoid claiming complete operational validation on a real QPU when the source is a snapshot, cache, or static reference.



## 27. Final tables for the paper

The tables below consolidate the definitive results after cross-validation by `workload_id`, end-to-end evaluation with out-of-fold predictions, ranking recalculation using the final OOF pipeline, and limited complementary validation with Qiskit/Aer and source-aware external QPU metadata comparison by modality when available.

Before writing the paper, inspect the final evidence-scope table. The wording must follow the actual source used in the run: live IBM/Qiskit Runtime metadata, cached metadata, Qiskit snapshot/fake backend, or static plausibility reference.



In [ ]:
# ============================================================
# 27. Final tables for the paper
# ============================================================

# Preserve compatibility with variables from previous sections.
table_dataset_final = table_dataset.copy() if "table_dataset" in globals() else pd.DataFrame()
table_ranking_final = ranking_metrics.copy() if "ranking_metrics" in globals() else pd.DataFrame()
table_utility_regret_final = utility_regret_summary.copy() if "utility_regret_summary" in globals() else pd.DataFrame()
table_group_cv_final = cv_summary_df.copy() if "cv_summary_df" in globals() else pd.DataFrame()

# Separate tables to avoid displaying NaN in incompatible metrics.
model_result_rows = []
for name in ["runtime_metrics", "fidelity_metrics", "clf_metrics", "clf_metrics_e2e"]:
    if name in globals():
        model_result_rows.append(globals()[name])
base_model_results = pd.DataFrame(model_result_rows)

if "oof_e2e_comparison_df" in globals() and isinstance(oof_e2e_comparison_df, pd.DataFrame):
    table_model_results_combined_audit = pd.concat([base_model_results, oof_e2e_comparison_df], ignore_index=True, sort=False)
else:
    table_model_results_combined_audit = base_model_results.copy()

def _ensure_column(df_obj, col, value):
    if isinstance(df_obj, pd.DataFrame) and not df_obj.empty and col not in df_obj.columns:
        df_obj[col] = value
    return df_obj

# Regression results.
regression_metric_cols = ["MAE", "RMSE", "R2"]
reg_cols = ["model", "target", "evaluation"] + regression_metric_cols
table_model_results_regression = table_model_results_combined_audit.dropna(
    how="all", subset=[c for c in regression_metric_cols if c in table_model_results_combined_audit.columns]
).copy()
table_model_results_regression = _ensure_column(table_model_results_regression, "evaluation", "holdout_group_split")
table_model_results_regression = table_model_results_regression[[c for c in reg_cols if c in table_model_results_regression.columns]]
table_model_results_regression = table_model_results_regression.drop_duplicates().reset_index(drop=True)

# Classification results.
classification_metric_cols = ["Accuracy", "Precision", "Recall", "F1"]
clf_cols = ["model", "task", "evaluation"] + classification_metric_cols
table_model_results_classification = table_model_results_combined_audit.dropna(
    how="all", subset=[c for c in classification_metric_cols if c in table_model_results_combined_audit.columns]
).copy()
table_model_results_classification = _ensure_column(table_model_results_classification, "task", "acceptable_backend_classification")
table_model_results_classification = _ensure_column(table_model_results_classification, "evaluation", "holdout_or_oof")
table_model_results_classification = table_model_results_classification[[c for c in clf_cols if c in table_model_results_classification.columns]]
table_model_results_classification = table_model_results_classification.drop_duplicates(subset=[c for c in ["model", "task", "evaluation"] if c in table_model_results_classification.columns]).reset_index(drop=True)

# Separate cross-validation tables.
if "group_cv_summary_regression" in globals():
    table_group_cv_regression = group_cv_summary_regression.copy()
elif isinstance(table_group_cv_final, pd.DataFrame) and not table_group_cv_final.empty:
    metric_subset = [c for c in ["MAE_mean", "MAE_std", "RMSE_mean", "RMSE_std", "R2_mean", "R2_std"] if c in table_group_cv_final.columns]
    table_group_cv_regression = table_group_cv_final.dropna(how="all", subset=metric_subset).copy()
else:
    table_group_cv_regression = pd.DataFrame()

if not table_group_cv_regression.empty:
    desired = ["task", "target", "evaluation", "MAE_mean", "MAE_std", "RMSE_mean", "RMSE_std", "R2_mean", "R2_std"]
    table_group_cv_regression = table_group_cv_regression[[c for c in desired if c in table_group_cv_regression.columns]].drop_duplicates().reset_index(drop=True)

if "group_cv_summary_classification" in globals():
    table_group_cv_classification = group_cv_summary_classification.copy()
elif isinstance(table_group_cv_final, pd.DataFrame) and not table_group_cv_final.empty:
    metric_subset = [c for c in ["Accuracy_mean", "Accuracy_std", "Precision_mean", "Precision_std", "Recall_mean", "Recall_std", "F1_mean", "F1_std"] if c in table_group_cv_final.columns]
    table_group_cv_classification = table_group_cv_final.dropna(how="all", subset=metric_subset).copy()
else:
    table_group_cv_classification = pd.DataFrame()

if not table_group_cv_classification.empty:
    desired = ["task", "evaluation", "Accuracy_mean", "Accuracy_std", "Precision_mean", "Precision_std", "Recall_mean", "Recall_std", "F1_mean", "F1_std"]
    table_group_cv_classification = table_group_cv_classification[[c for c in desired if c in table_group_cv_classification.columns]].drop_duplicates().reset_index(drop=True)

final_tables = {
    "final_dataset_description_v9_6.csv": table_dataset_final,
    "final_model_results_regression_v9_6.csv": table_model_results_regression,
    "final_model_results_classification_v9_6.csv": table_model_results_classification,
    "final_model_results_combined_audit_v9_6.csv": table_model_results_combined_audit,
    "final_group_cv_summary_regression_v9_6.csv": table_group_cv_regression,
    "final_group_cv_summary_classification_v9_6.csv": table_group_cv_classification,
    "final_group_cv_summary_combined_audit_v9_6.csv": table_group_cv_final,
    "final_backend_ranking_metrics_v9_6.csv": table_ranking_final,
    "final_utility_regret_summary_v9_6.csv": table_utility_regret_final,
}

optional_tables = {
    "final_backend_catalog_calibrated_v9_6.csv": globals().get("backend_catalog"),
    "final_qpu_generation_anchor_reference_v9_6.csv": globals().get("qpu_generation_anchor_reference"),
    "final_external_anchor_evidence_scope_v9_6.csv": globals().get("qpu_generation_anchor_evidence_scope", globals().get("external_anchor_evidence_scope")),
    "final_selection_strategy_comparison_v9_6.csv": globals().get("selection_strategy_comparison"),
    "final_feature_ablation_results_v9_6.csv": globals().get("feature_ablation_results"),
    "final_subgroup_classification_metrics_v9_6.csv": globals().get("subgroup_clf_results"),
    "final_subgroup_ranking_metrics_v9_6.csv": globals().get("subgroup_ranking_results"),
    "final_selection_distribution_comparison_v9_6.csv": globals().get("selection_distribution_comparison"),
    "final_real_validation_aer_small_circuits_v9_6.csv": globals().get("qiskit_aer_small_circuit_runs"),
    "final_external_qpu_metadata_v9_6.csv": globals().get("external_qpu_metadata"),
    "final_real_vs_synthetic_qpu_ranges_v9_6.csv": globals().get("real_vs_synthetic_qpu_ranges"),
    "final_real_data_validation_score_v9_6.csv": globals().get("real_data_validation_score"),
    "final_synthetic_data_quality_checks_v9_6.csv": globals().get("synthetic_data_quality_checks"),
    "final_real_validation_summary_v9_6.csv": globals().get("real_validation_summary"),
}

for filename, table in optional_tables.items():
    if isinstance(table, pd.DataFrame):
        final_tables[filename] = table.reset_index(drop=False) if filename == "final_selection_distribution_comparison_v9_6.csv" else table

for filename, table in final_tables.items():
    if isinstance(table, pd.DataFrame):
        table.to_csv(TAB_DIR / filename, index=False)

print("Final tables saved in:", TAB_DIR)

# Clean display: only main tables, separated by metric type.
if not table_model_results_regression.empty:
    print("Final results - regression")
    display(table_model_results_regression)
if not table_model_results_classification.empty:
    print("Final results - classification")
    display(table_model_results_classification)
if not table_group_cv_regression.empty:
    print("Workload-level cross-validation - regression")
    display(table_group_cv_regression)
if not table_group_cv_classification.empty:
    print("Workload-level cross-validation - classification")
    display(table_group_cv_classification)
if isinstance(table_ranking_final, pd.DataFrame) and not table_ranking_final.empty:
    print("Ranking/selection metrics")
    display(table_ranking_final)
if "selection_strategy_comparison" in globals() and isinstance(selection_strategy_comparison, pd.DataFrame):
    print("Comparison with heuristic policies")
    display(selection_strategy_comparison)
if "feature_ablation_results" in globals() and isinstance(feature_ablation_results, pd.DataFrame):
    print("Feature-group ablation")
    display(feature_ablation_results)
if isinstance(table_utility_regret_final, pd.DataFrame) and not table_utility_regret_final.empty:
    print("Utility regret summary")
    display(table_utility_regret_final)
if "real_validation_summary" in globals() and isinstance(real_validation_summary, pd.DataFrame):
    print("External data validation summary")
    display(real_validation_summary)




## 27.1 Generated artifact inventory

The cell below creates an index of all files exported by the notebook. This inventory supports supplementary-material submission and preserves traceability between figures, tables, and data.



In [ ]:

# ============================================================
# Generated artifact inventory
# ============================================================

artifact_rows = []
for folder, artifact_type in [(DATA_DIR, "data"), (TAB_DIR, "table"), (FIG_DIR, "figure"), (REAL_VALIDATION_DIR, "real_validation")]:
    if folder.exists():
        for p in sorted(folder.glob("*")):
            if p.is_file():
                artifact_rows.append({
                    "type": artifact_type,
                    "file": p.name,
                    "relative_path": str(p),
                    "size_kb": round(p.stat().st_size / 1024, 2),
                })

artifact_inventory = pd.DataFrame(artifact_rows)
artifact_inventory.to_csv(TAB_DIR / "artifact_inventory_v9_6.csv", index=False)

print("Generated artifact inventory:")
display(artifact_inventory)



## 28. Methodological appendix: how to use this notebook in the paper

### 28.1 Recommended framing

Use this notebook as evidence of a methodological proposal, not as conclusive evidence of performance in real quantum infrastructure. The paper should state that:

- the main experiment is synthetic;
- the decision policy is synthetic and probabilistic;
- the external validation anchors QPU ranges only according to the reported evidence scope;
- live IBM/Qiskit Runtime validation can be claimed only when `live_ibm_runtime_available = True` and `anchor_priority_values` includes `live_current_run`;
- static, cached, or snapshot references are disabled in this final-paper configuration and must not be used as substitute evidence;
- the end-to-end evaluation tests error propagation between regression and classification;
- workload-level cross-validation reduces information leakage.

### 28.2 Recommended methodology sentence

> The experimental dataset was synthetically generated from parameterized profiles of algorithms, backends, and operational policies. To reduce the purely synthetic character of the evaluation, QPU-associated variable ranges were compared with the best available external reference, whose evidence scope is explicitly reported as live IBM/Qiskit Runtime metadata, cached metadata, Qiskit snapshot/fake backend, or static plausibility reference. This step was used only as a plausibility validation, not as a training source.

### 28.3 Recommended limitation sentence

> Results should be interpreted as evidence of methodological feasibility under controlled, externally anchored scenarios. If the evidence scope is static, cached, or snapshot-based, the results should not be interpreted as live IBM/Qiskit Runtime validation or deployment generalization to production QPUs.



## 29. Short model card for the pipeline

### Model type

Supervised pipeline with regression, classification, and operational ranking.

### Inputs

Workload, algorithm, backend, operational state, cost, queueing, noise, capacity, and decision-profile attributes.

### Outputs

- predicted total runtime;
- predicted expected fidelity;
- binary execution recommendation;
- backend ranking by utility;
- final selection and fallback.

### Training data

Synthetic data generated by a parameterized function. External validation is used only for plausibility, not for training.

### Recommended use

Prototyping, synthetic benchmarking, comparison of decision strategies, sensitivity analysis, and supplementary material for the paper.

### Not recommended for

Real operational decision-making without calibration with real logs, current backend metadata, observed queues, and QPU execution.



## 30. Preliminary interpretation of results

Strengthens the methodological robustness of the experiment by keeping the main evaluation synthetic while adding source-aware external validation. The dataset is still generated in a controlled way, but QPU variables are compared with the best available external reference: live IBM/Qiskit metadata when available, cached metadata, Qiskit snapshots/fake backends, or a static plausibility reference explicitly marked as non-current.

The external validation does not change model training. Its role is to check whether synthetic ranges for qubit counts, gate errors, readout errors, coherence times, and connectivity remain compatible with the recorded evidence source. Therefore, the study includes external plausibility anchoring, but the strength of the claim depends on the generated evidence-scope table.

The central interpretation remains cautious: the notebook validates an enriched synthetic decision-support policy for hybrid quantum-classical workload orchestration. The results do not demonstrate operational generalization to real backends, because the main labels are still derived from a synthetic probabilistic policy and because live QPU validation is claimed only when explicitly reported by the source-aware evidence table.



## 31. Methodological note for a full paper

In this version, the contribution should be presented as a **source-aware externally anchored synthetic benchmark**. IBM/Qiskit collection is positioned before synthetic generation to calibrate the QPU catalog by compatible modality when live or cached IBM/Qiskit metadata are available. The modeling still learns a synthetic operational decision policy, and hardware parameters used in the simulation are anchored only according to the evidence level reported by the notebook.

Recommended formulation for the paper when live metadata are available:

> We propose a source-aware, partially live-metadata-anchored synthetic benchmark and an AI-assisted pipeline for backend selection and workload orchestration in hybrid quantum-classical systems. IBM/Qiskit backend metadata are collected before dataset generation and used to calibrate compatible superconducting QPU parameters, while synthetic workloads preserve controlled experimental coverage.

Recommended formulation for the paper when the run uses static/cached/snapshot references:

> We propose a source-aware synthetic benchmark and an AI-assisted pipeline for backend selection and workload orchestration in hybrid quantum-classical systems. The synthetic QPU parameters are compared with cached, snapshot, or static external references to support order-of-magnitude plausibility, but the execution does not constitute live IBM/Qiskit Runtime validation.

Limitation that must remain explicit:

> The benchmark does not demonstrate direct deployment generalization to production QPUs. It evaluates the methodological feasibility of AI-assisted orchestration under controlled, externally anchored workload-backend scenarios. When the evidence scope is static, cached, or snapshot-based, the external check must be reported as plausibility validation rather than live Runtime validation.



In [ ]:
import shutil
from google.colab import files

output_dir = "outputs_ai_quantum_v9_6"
zip_name = "outputs_ai_quantum_v9_6.zip"

shutil.make_archive("outputs_ai_quantum_v9_6", "zip", output_dir)

files.download(zip_name)